
# Amaze KWS — self-contained Kaggle notebook

Everything runs from this one file. **No attached datasets, no repo checkout, no external
setup** — the Kaggle credentials are in one cell below, and this notebook only ever
*writes* to Kaggle (two datasets it owns), it never reads from one.

| stage | what happens | cost on a re-run |
|---|---|---|
| 1. dataset | fetch ESC-50 / UrbanSound8K / LibriSpeech, synthesise the TTS pools, build the 24-bucket corpus, publish it | zero if the build key matches |
| 2. train | `BCConformerV2` on the GPU, TPR-at-FPR-budget objective | zero if a published checkpoint matches the train key |
| 3. run | benchmark, threshold × policy sweep, freeze the operating point | ~20 min |
| 4. publish | corpus + frozen checkpoint as Kaggle Datasets | zero |

## Why the keys exist

The corpus is expensive to synthesise, and a filename cache silently serves stale data
the moment an augmentation function changes. So `build_key` is a SHA-256 over the build
config **and** the source text of every build stage in this notebook. Change one line of
`acoustic_augment`, and the corpus rebuilds itself instead of lying to you. The same idea
applies to the model: `train_key` hashes the build key plus every hyperparameter, so a
published checkpoint is only reused when it came from exactly this configuration.

## Settings

Runs on Kaggle (GPU T4 x2) **or on a local machine** — the accelerator is detected, and
with no NVIDIA GPU it falls back to CPU and prints a measured time estimate. Internet
access is required either way: the raw audio (ESC-50, UrbanSound8K, LibriSpeech) comes
from public URLs, and the corpus build shells out to `edge-tts`. Everything tunable is in
the `CFG` dict in the next cell.


In [1]:

# ==============================================================================
# CONFIG -- everything tunable lives here
# ==============================================================================
import os, sys, json, time, shutil, subprocess, hashlib, zipfile, glob, csv, tarfile
import urllib.request, tempfile
import asyncio
import threading
from collections import defaultdict

def run_coroutine_sync(coro):
    """Run a coroutine to completion from synchronous notebook code.

    asyncio.run() refuses to nest inside a running event loop, and a Jupyter kernel
    always has one -- so calling it directly to synthesise the TTS pools raises
    "RuntimeError: asyncio.run() cannot be called from a running event loop". The
    coroutine gets its own thread, and therefore its own event loop.
    """
    box = {}

    def target():
        try:
            box["value"] = asyncio.run(coro)
        except BaseException as e:          # re-raised on the calling thread below
            box["error"] = e

    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]

# Work directory. /kaggle/working only exists on Kaggle; everywhere else -- including
# this repo on a laptop -- fall back to a sibling directory so a local run writes
# somewhere real instead of trying to create a path at the filesystem root.
if "KWS_WORK_DIR" in os.environ:
    WORK = os.environ["KWS_WORK_DIR"]
elif os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working"
else:
    WORK = os.path.join(os.path.expanduser("~"), "amaze_kws_run")
os.makedirs(WORK, exist_ok=True)

CFG = dict(
    # corpus
    pos=50_000, neg=85_000, profile="v4", seed=20260927, per_voice=4,
    workers=4, fp16=True,
    # model
    arch="bcconformer_v3", epochs=60, batch_size=512, lr=2.5e-3,
    fpr_budget=0.005, hard_neg_k=192, gamma=1.2, pos_bias=1.5, spec_aug_prob=0.30,
    val_metric_cap=20_000,
    # deployment
    fa_budget=1.0, thresholds=[0.78, 0.82, 0.84, 0.86, 0.90, 0.94],
    policy="2/2", refractory=15, speech_files=150, recall_trials=300,
    # Kaggle datasets this notebook owns. Resolved after the API cell, because asking for
    # the username needs credentials that do not exist yet at config time.
    corpus_slug=None, model_slug=None,
    publish_corpus=True, publish_model=True,
    force_rebuild=False, force_retrain=False,
)

print(json.dumps(CFG, indent=2, default=str))
print("WORK =", WORK)
print("free space:", shutil.disk_usage(WORK).free / 1e9, "GB")


{
  "pos": 50000,
  "neg": 85000,
  "profile": "v4",
  "seed": 20260927,
  "per_voice": 4,
  "workers": 4,
  "fp16": true,
  "arch": "bcconformer_v2",
  "epochs": 60,
  "batch_size": 512,
  "lr": 0.0025,
  "fpr_budget": 0.005,
  "hard_neg_k": 192,
  "gamma": 1.2,
  "pos_bias": 1.5,
  "spec_aug_prob": 0.3,
  "val_metric_cap": 20000,
  "fa_budget": 1.0,
  "thresholds": [
    0.78,
    0.82,
    0.84,
    0.86,
    0.9,
    0.94
  ],
  "policy": "2/2",
  "refractory": 15,
  "speech_files": 150,
  "recall_trials": 300,
  "corpus_slug": null,
  "model_slug": null,
  "publish_corpus": true,
  "publish_model": true,
  "force_rebuild": false,
  "force_retrain": false
}
WORK = /kaggle/working
free space: 20.01983488 GB


In [2]:

# ==============================================================================
# Dependencies.
#
# JAX_PLATFORMS must be set before jax is ever imported, and the CUDA wheels must be
# installed before that import happens -- a CPU `import jax` first wins and the run
# silently falls back to the slowest path available. The flip side is that asking for
# CUDA on a machine with no NVIDIA GPU leaves jax with no devices at all, so the
# accelerator is detected before anything imports jax.
# ==============================================================================
def detect_nvidia_gpu():
    if shutil.which("nvidia-smi"):
        try:
            out = subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                                 text=True, timeout=30)
            return out.returncode == 0 and "GPU" in out.stdout
        except Exception:
            return False
    return os.path.isdir("/proc/driver/nvidia/gpus")

GPU = detect_nvidia_gpu()
if GPU:
    os.environ["JAX_PLATFORMS"] = "cuda"
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
else:
    os.environ.pop("JAX_PLATFORMS", None)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "kaggle", "soundfile", "scipy", "scikit-learn",
                       "edge-tts", "pyarrow", "huggingface_hub", "flax", "optax"])
import jax
if GPU and not any(d.platform == "gpu" for d in jax.devices()):
    print("jax is CPU-only, installing the CUDA 12 stack ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "jax[cuda12]"])
    for m in [m for m in list(sys.modules) if m.split(".")[0] == "jax"]:
        del sys.modules[m]
    import jax

_devices = jax.devices()
print("jax    :", jax.__version__)
print("devices:", _devices)
if not any(d.platform == "gpu" for d in _devices):
    print("\n*** No NVIDIA GPU visible: this will train on CPU. ***")
    print("Measured throughput for this model on a laptop-class CPU is ~200 samples/s,")
    print("so a 135k-sample epoch is ~9 min and the default 60 epochs is 9-11 hours.")
    print("To shorten a local run, lower CFG['epochs']; to shrink the corpus itself,")
    print("lower CFG['pos'] and CFG['neg'] (that changes the build key, hence a rebuild).")
    print("Or run this notebook as an actual Kaggle notebook on a T4 x2.")
else:
    print("gpu    :", _devices[0].device_kind)


jax    : 0.7.2
devices: [CudaDevice(id=0), CudaDevice(id=1)]
gpu    : Tesla T4



## Kaggle plumbing

Credentials are read from the environment by the next cell and written to
`~/.kaggle/kaggle.json`, so the client authenticates identically here and on Kaggle.
Nothing prints the key, and this file holds no secret.

On Kaggle both `KAGGLE_USERNAME` and `KAGGLE_KEY` are injected automatically. Locally,
export them before starting the kernel. With no key in the environment the notebook
still runs every stage and only the Kaggle upload/download stages are skipped.

The rest of this cell only ever *writes* to Kaggle: the two datasets the notebook owns.
It never reads from an attached input, because there are none.


In [3]:

# ==============================================================================
# Kaggle credentials, read from the environment so no secret is stored in this file.
# Kaggle injects KAGGLE_USERNAME and KAGGLE_KEY into every kernel; locally, export
# KAGGLE_API_KEY yourself. Either way the key is only ever written to the local
# ~/.kaggle/kaggle.json, and nothing below prints it.
#
# With no key present the notebook still runs end to end and only the Kaggle
# upload/download stages are skipped, so a laptop run needs no credentials at all.
# ==============================================================================
KAGGLE_USERNAME = (os.environ.get("KAGGLE_USERNAME")
                  or os.environ.get("KAGGLE_USER") or "dsainvg")
KAGGLE_API_KEY = (os.environ.get("KAGGLE_API_KEY")
                  or os.environ.get("KAGGLE_KEY") or "")
HAVE_KAGGLE_KEY = bool(KAGGLE_API_KEY)

if HAVE_KAGGLE_KEY:
    _kaggle_cfg = os.path.join(os.path.expanduser("~"), ".kaggle")
    os.makedirs(_kaggle_cfg, exist_ok=True)
    with open(os.path.join(_kaggle_cfg, "kaggle.json"), "w", encoding="utf-8") as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_API_KEY}, f)
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_API_KEY
    os.chmod(os.path.join(_kaggle_cfg, "kaggle.json"), 0o600) if os.name != "nt" else None
    print(f"credentials configured for {KAGGLE_USERNAME} (key written to ~/.kaggle/kaggle.json)")
else:
    print(f"no Kaggle key in the environment for {KAGGLE_USERNAME}; Kaggle stages will be skipped")

# ==============================================================================
# The rest of the Kaggle plumbing. This notebook only ever *writes* to Kaggle -- the two
# datasets it owns -- and never reads from an attached input, because there are none.
# ==============================================================================
from kaggle.api.kaggle_api_extended import KaggleApi

def kaggle_api():
    if not HAVE_KAGGLE_KEY:
        raise RuntimeError("no Kaggle key in the environment: set KAGGLE_API_KEY to "
                           "enable Kaggle upload/download")
    api = KaggleApi()
    api.authenticate()
    return api

def kaggle_username():
    return KAGGLE_USERNAME

def kaggle_remote_exists(slug):
    try:
        kaggle_api().dataset_status(slug)
        return True
    except Exception:
        return False

def kaggle_pull(slug, dest):
    os.makedirs(dest, exist_ok=True)
    try:
        kaggle_api().dataset_download_files(slug, path=dest, force=True)
    except Exception as e:
        print(f"[kaggle] pull failed: {str(e)[:200]}")
        return False
    # Kaggle nests the payload one directory deep
    if not any(f.endswith(".flax") or f.endswith(".npz") for f in os.listdir(dest)):
        for entry in os.listdir(dest):
            sub = os.path.join(dest, entry)
            if os.path.isdir(sub):
                for f in os.listdir(sub):
                    shutil.move(os.path.join(sub, f), os.path.join(dest, f))
                shutil.rmtree(sub, ignore_errors=True)
                break
    return True

def kaggle_publish(bundle_dir, slug, title, subtitle="", description="", tags=None):
    resources = []
    for root, _dirs, files in os.walk(bundle_dir):
        for f in files:
            rel = os.path.relpath(os.path.join(root, f), bundle_dir).replace(os.sep, "/")
            if rel in ("manifest.json", "dataset-metadata.json") or rel.startswith("."):
                continue
            resources.append(rel)
    resources.sort()
    meta = {"title": title, "id": slug, "licenses": [{"name": "CC0-1.0"}],
            "keywords": tags or ["keyword-spotting", "wake-word", "speech", "audio"],
            "subtitle": subtitle or title, "description": description or title,
            "resources": [{"description": r, "path": r} for r in resources],
            "isPrivate": True}
    with open(os.path.join(bundle_dir, "dataset-metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    ignore = [r"^dataset-metadata\.json$", r"^\."]

    def _apply(fn, folder, **kw):
        """Calls fn with only the kwargs its installed version actually accepts.

        The kaggle client grew ignore_patterns and dir_mode over time; a 2.0.x install
        raises TypeError on the first one, which killed publishing outright.
        """
        import inspect as _inspect
        try:
            params = _inspect.signature(fn).parameters
        except (TypeError, ValueError):
            params = {}
        if params:
            if "ignore_patterns" not in params:
                print("[kaggle] client predates ignore_patterns: dataset-metadata.json "
                      "will be uploaded as a resource (harmless, it is not data)")
            kw = {k: v for k, v in kw.items() if k in params}
        return fn(folder, **kw)

    try:
        exists = kaggle_remote_exists(slug)
        if exists:
            action = "new version"
            api = kaggle_api()
            _apply(api.dataset_create_version, bundle_dir,
                   version_notes=f"build_key={manifest_of(bundle_dir).get('build_key', '?')}",
                   quiet=True, dir_mode="recurse", ignore_patterns=ignore)
        else:
            action = "created"
            api = kaggle_api()
            _apply(api.dataset_create_new, bundle_dir, public=False, quiet=True,
                   dir_mode="recurse", ignore_patterns=ignore)
        print(f"[kaggle] {action}: https://www.kaggle.com/datasets/{slug}")
        return True
    except Exception as e:
        print(f"[kaggle] publish failed: {type(e).__name__}: {str(e)[:300]}")
        return False


credentials configured for dsainvg (key written to ~/.kaggle/kaggle.json)


In [4]:

# The dataset slugs need credentials, so they are resolved here rather than in CFG.
CFG["corpus_slug"] = os.environ.get("KWS_CORPUS_SLUG", f"{kaggle_username()}/amaze-kws-corpus")
CFG["model_slug"] = os.environ.get("KWS_MODEL_SLUG", f"{kaggle_username()}/amaze-kws-models")
print("corpus dataset:", CFG["corpus_slug"], "exists =", kaggle_remote_exists(CFG["corpus_slug"]))
print("model  dataset:", CFG["model_slug"], "exists =", kaggle_remote_exists(CFG["model_slug"]))


corpus dataset: dsainvg/amaze-kws-corpus exists = False
model  dataset: dsainvg/amaze-kws-models exists = False



## Content addressing

`build_key` folds the build config together with the SHA-256 of the *source text* of
every build stage in this notebook, plus the digest of the noise index. `manifest.json`
records a SHA-256 per file so a truncated download fails loudly instead of training on
corrupted features.


In [5]:

# ==============================================================================
# Content addressing. The build key hashes the build config AND the source text of
# every build stage in this notebook, so editing one augmentation function
# invalidates the cache instead of silently serving stale data. The manifest records a
# SHA-256 per file so a truncated download fails loudly rather than training on
# corrupted features.
# ==============================================================================
# Symbols whose source defines the bytes of a corpus. Hashing the symbols rather than a
# module name is what makes this work in a notebook: the modules do not exist as objects
# here, their contents are just names in one namespace.
STAGE_SYMBOLS = [
    # front end
    "AudioFeatureExtractor", "apply_spec_augment",
    # acoustics / transducer model
    "apply_reverberation", "apply_distance_attenuation", "apply_speed_perturbation",
    "apply_inmp441_response", "add_mic_self_noise", "apply_i2s_16bit_truncation",
    "draw_talker_conditions", "apply_full_acoustic_chain", "mix_dual_independent_noises",
    # level policy
    "apply_level_policy", "apply_transducer_chain", "fit_to_window", "peak_guard",
    # distortion ladder
    "apply_vtlp", "apply_adc_bitcrush", "apply_nonlinear_cadence_warp",
    "apply_telephonic_bandpass", "apply_dropout_bursts", "apply_awgn",
    # sample construction
    "NoiseBank", "SpeechBank", "build_positive", "build_negative", "build_bucket_sample",
    "mix_layers", "distortion_chain", "make_babble_from_speech", "stratified_split",
    "build_tts_pools", "build_librispeech", "build_esc50", "build_urbansound",
    "build_babble", "build_procedural", "write_index", "load_pool_cache",
]

def _sha256(path, limit=None):
    h = hashlib.sha256()
    n = 0
    with open(path, "rb") as f:
        while True:
            b = f.read(1 << 20)
            if not b:
                break
            h.update(b)
            n += len(b)
            if limit is not None and n >= limit:
                break
    return h.hexdigest()

def symbol_source(obj):
    """Source text of a build symbol.

    inspect.getsource works on functions in a notebook because their code objects point
    at a <ipython-input-...> filename that IPython registers in linecache. It does NOT
    work on classes: findsource() then looks for `class X` inside __main__.__file__,
    which is ipykernel's own file, and raises OSError. So a class is hashed as its name
    plus every method body plus its class-level attributes.
    """
    import inspect
    if inspect.isclass(obj):
        parts = [f"class {obj.__name__}"]
        for k, v in sorted(vars(obj).items()):
            if inspect.isfunction(v) or inspect.ismethod(v):
                parts.append(inspect.getsource(v))
            else:
                parts.append(f"{k}={v!r}")
        return "\n".join(parts)
    return inspect.getsource(obj)


def stage_source_digest():
    """SHA-256 over the source text of every symbol that affects the corpus bytes.

    A key that cannot see the code is worse than no key at all: it would make every edit
    look like a no-op and silently serve a stale corpus forever. So an unreadable source
    is a hard error, not a degraded digest.
    """
    import inspect
    h = hashlib.sha256()
    unreadable = []
    for name in STAGE_SYMBOLS:
        obj = globals().get(name)
        if obj is None:
            unreadable.append(f"{name} (not defined)")
            continue
        try:
            src = symbol_source(obj)
        except (OSError, TypeError) as e:
            unreadable.append(f"{name} ({type(e).__name__})")
            continue
        h.update(name.encode())
        h.update(src.encode())
    if unreadable:
        raise RuntimeError(
            "build key cannot be content-addressed: no source for "
            f"{len(unreadable)} symbol(s): {unreadable[:8]}. The corpus would be cached "
            "against a key that no code edit can change.")
    return h.hexdigest()

# NOTE: the key is deliberately not computed in this cell. The symbols above are defined
# by the stage cells that come *after* it, so hashing here would always see a notebook
# with no build stages in it and produce a key that no code edit could ever change.
# The first call is in the build cell, by which point every stage is defined.

def compute_build_key(cfg):
    ni = os.path.join(WORK, "noise_bank_v2", "index.csv")
    payload = {
        "schema": 3,
        "spec": {k: cfg[k] for k in ("pos", "neg", "profile", "seed", "per_voice", "fp16")},
        "stages": stage_source_digest(),
        "noise_index_sha256": _sha256(ni) if os.path.exists(ni) else None,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]

def manifest_of(bundle_dir):
    p = os.path.join(bundle_dir, "manifest.json")
    if not os.path.exists(p):
        return {}
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def write_manifest(bundle_dir, build_key, extra=None):
    files = {}
    for name in sorted(os.listdir(bundle_dir)):
        p = os.path.join(bundle_dir, name)
        if os.path.isfile(p) and name not in ("manifest.json", "dataset-metadata.json"):
            files[name] = {"bytes": os.path.getsize(p), "sha256": _sha256(p)}
    man = {"schema_version": 3, "build_key": build_key,
           "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "files": files, "total_bytes": sum(v["bytes"] for v in files.values())}
    if extra:
        man["extra"] = extra
    with open(os.path.join(bundle_dir, "manifest.json"), "w", encoding="utf-8") as f:
        json.dump(man, f, indent=2, sort_keys=True)
    return man

def verify_bundle(bundle_dir):
    man = manifest_of(bundle_dir)
    if not man:
        return False
    for name, meta in man.get("files", {}).items():
        p = os.path.join(bundle_dir, name)
        if not os.path.exists(p) or os.path.getsize(p) != meta["bytes"] \
                or _sha256(p) != meta["sha256"]:
            print(f"  [verify] BAD {name}")
            return False
    print(f"  [verify] ok: {len(man.get('files', {}))} files, "
          f"{man.get('total_bytes', 0)/1e6:.1f} MB, build_key={man['build_key']}")
    return True


## Feature front end

40-band log-mel, 100-7500 Hz, (49, 40) per second. The INMP441's 60 Hz - 15 kHz response fully brackets this band, so the capsule is transparent in-band.

*Inlined from `features.py`: 151 lines, 9 CLI lines and 0 cross-module imports removed — everything shares one namespace here.*


In [6]:
"""
Audio Feature Engineering & Signal Processing for TinyML KWS
------------------------------------------------------------
Implements:
1. 40-band Triangular Log-Mel Filterbank Spectrogram (matching Google MicroFrontend & ESP-DSP)
2. PCEN (Per-Channel Energy Normalization) for ambient noise robustness
3. SpecAugment (Time & Frequency masking for regularized training)
"""

import math
import numpy as np


class AudioFeatureExtractor:
    """
    Computes 40-channel Log-Mel Spectrogram or PCEN features:
    - Sample rate: 16,000 Hz
    - Window size: 30 ms (480 samples, padded to 512 for FFT)
    - Stride / Hop: 20 ms (320 samples)
    - Mel bands: 40 (100 Hz to 7500 Hz)
    - Target Output for 1.0s audio: (49, 40) [Time Frames, Mel Bands]
    """
    def __init__(
        self,
        sample_rate: int = 16000,
        n_fft: int = 512,
        win_length: int = 480,
        hop_length: int = 320,
        n_mels: int = 40,
        f_min: float = 100.0,
        f_max: float = 7500.0,
        use_pcen: bool = False
    ):
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.win_length = win_length
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.f_min = f_min
        self.f_max = f_max
        self.use_pcen = use_pcen

        # Precompute Hanning window
        self.window = np.hanning(win_length).astype(np.float32)

        # Precompute Mel Filterbank matrix (n_fft // 2 + 1, n_mels)
        self.mel_fb = self._create_mel_filterbank(sample_rate, n_fft, n_mels, f_min, f_max)

    @staticmethod
    def _hz_to_mel(hz: float) -> float:
        return 2595.0 * np.log10(1.0 + hz / 700.0)

    @staticmethod
    def _mel_to_hz(mel: float) -> float:
        return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

    def _create_mel_filterbank(self, sr: int, n_fft: int, n_mels: int, f_min: float, f_max: float) -> np.ndarray:
        mel_min = self._hz_to_mel(f_min)
        mel_max = self._hz_to_mel(f_max)
        mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
        hz_points = self._mel_to_hz(mel_points)
        bins = np.floor((n_fft + 1) * hz_points / sr).astype(int)

        weights = np.zeros((n_fft // 2 + 1, n_mels), dtype=np.float32)
        for i in range(n_mels):
            left = bins[i]
            center = bins[i + 1]
            right = bins[i + 2]
            if center > left:
                weights[left:center, i] = np.linspace(0, 1, center - left, endpoint=False)
            if right > center:
                weights[center:right, i] = np.linspace(1, 0, right - center, endpoint=False)
        return weights

    def compute_spectrogram(self, waveform: np.ndarray) -> np.ndarray:
        """
        Input: 1D numpy array of audio samples (e.g. 16,000 samples for 1.0s at 16kHz)
        Output: 2D numpy array of shape (49, 40)
        """
        # Ensure exact 16000 samples (1.0s)
        target_len = self.sample_rate
        if len(waveform) < target_len:
            waveform = np.pad(waveform, (0, target_len - len(waveform)))
        elif len(waveform) > target_len:
            waveform = waveform[:target_len]

        num_frames = (len(waveform) - self.win_length) // self.hop_length + 1
        frames = np.zeros((num_frames, self.win_length), dtype=np.float32)
        for t in range(num_frames):
            start = t * self.hop_length
            frames[t] = waveform[start:start + self.win_length] * self.window

        # Pad frames to n_fft
        padded_frames = np.pad(frames, ((0, 0), (0, self.n_fft - self.win_length)))

        # Real FFT & Power Spectrum |FFT|^2
        fft_out = np.fft.rfft(padded_frames, n=self.n_fft, axis=1)
        power_spec = np.abs(fft_out) ** 2  # (num_frames, n_fft // 2 + 1)

        # Apply 40-band Mel Filterbank
        mel_spec = np.dot(power_spec, self.mel_fb)  # (num_frames, 40)

        if self.use_pcen:
            # Per-Channel Energy Normalization
            features = self._apply_pcen(mel_spec)
        else:
            # Standard Log Compression: log(E + eps)
            features = np.log(mel_spec + 1e-5)

        # Ensure exact time steps (49 frames)
        if len(features) < 49:
            features = np.pad(features, ((0, 49 - len(features)), (0, 0)))
        elif len(features) > 49:
            features = features[:49, :]

        return features.astype(np.float32)

    def _apply_pcen(self, mel_spec: np.ndarray, s: float = 0.025, alpha: float = 0.98, delta: float = 2.0, r: float = 0.5) -> np.ndarray:
        """
        PCEN: Per-Channel Energy Normalization (Wang et al. 2017)
        Performs adaptive temporal smoothing and dynamic range compression.
        """
        m = np.zeros_like(mel_spec)
        m[0] = mel_spec[0]
        for t in range(1, len(mel_spec)):
            m[t] = (1.0 - s) * m[t - 1] + s * mel_spec[t]

        pcen = ((mel_spec / ((1e-6 + m) ** alpha)) + delta) ** r - (delta ** r)
        return pcen.astype(np.float32)


def apply_spec_augment(spec: np.ndarray, max_f: int = 4, max_t: int = 6) -> np.ndarray:
    """
    Applies SpecAugment directly to a (Time, Mel) 2D spectrogram.
    Randomly zeroes out horizontal (frequency) and vertical (time) bars.
    """
    augmented = spec.copy()
    time_len, mel_len = augmented.shape

    # Frequency masking
    f_width = np.random.randint(1, max_f + 1)
    f0 = np.random.randint(0, mel_len - f_width)
    augmented[:, f0:f0 + f_width] = 0.0

    # Time masking
    t_width = np.random.randint(1, max_t + 1)
    t0 = np.random.randint(0, time_len - t_width)
    augmented[t0:t0 + t_width, :] = 0.0

    return augmented


## Acoustic augmentation

Room and transmission distortion, talker geometry, the INMP441 spec model, speed perturbation. The signal-path order mirrors the physical path.

*Inlined from `acoustic_augment.py`: 358 lines, 0 CLI lines and 0 cross-module imports removed — everything shares one namespace here.*


In [7]:
"""
Physics-Based Acoustic Augmentation Engine for Robust Keyword Spotting
----------------------------------------------------------------------
Simulates physical environmental and transducer phenomena:
1. Room Impulse Response (RIR) Reverberation:
   - Schroeder exponentially decaying reverberation with frequency-dependent absorption
   - Simulates diverse room sizes and distances (0.5m to 5.0m, T60 from 0.15s to 0.75s)
2. Atmospheric Air Absorption:
   - High-frequency damping due to acoustic propagation through air
3. Microphone Transducer Emulation:
   - Low-cut high-pass filter (<120 Hz) modeling MEMS microphone roll-off
   - Formant resonance peak (2.2 kHz - 3.2 kHz) modeling acoustic cavity resonance
   - Soft tanh harmonic saturation modeling analog pre-amp distortion
4. Speed & Pitch Perturbation:
   - Speed changes (0.85x to 1.15x) using linear interpolation resampling
   - Pitch shifting without altering clip duration
5. SpecAugment (Time & Frequency Masking):
   - Frequency channel masking (zeroing bands)
   - Time frame masking (zeroing frames)
6. INMP441 I2S MEMS microphone (see INMP441_SPEC):
   - Absolute electronic self-noise floor derived from sensitivity and SNR
   - 60 Hz - 15 kHz response, which sits entirely OUTSIDE the 100-7500 Hz feature
     band, so it is essentially transparent in-band apart from the LDC low shelf
   - 24-bit I2S quantization, which is far below the self-noise floor
"""

import numpy as np
from scipy import signal

SAMPLE_RATE = 16000

# ==============================================================================
# Target transducer: TDK InvenSense INMP441
# ==============================================================================
# Datasheet figures and what each one actually implies for this model.
INMP441_SPEC = {
    "sensitivity_dbfs": -26.0,   # at 94 dB SPL (1 Pa)
    "snr_dba": 61.0,             # A-weighted
    "f_low_hz": 60.0,
    "f_high_hz": 15000.0,
    "ldc_shelf_db": 3.0,         # Low-Noise-Compensation gain, ~+6 dB at 100 Hz
    "unit_spread_db": 5.0,       # unit-to-unit sensitivity spread
    "full_scale_spl_db": 94.0,
    "i2s_bits": 24,
}

# Electronic noise floor at the capsule, referenced to full scale: sensitivity - SNR.
# This is the number before any board preamp gain is applied.
INMP441_NOISE_FLOOR_DBFS = INMP441_SPEC["sensitivity_dbfs"] - INMP441_SPEC["snr_dba"]

# Feature front-end band (features.py): f_min=100 Hz, f_max=7500 Hz.
FEATURE_F_MIN_HZ = 100.0
FEATURE_F_MAX_HZ = 7500.0

# Conversational speech at various distances in free field (inverse-square from 1 m,
# ~60 dB SPL at 1 m for a talker at normal volume), plus a vocal-effort offset.
SPEAKING_SPL_DB = {0.3: 70.0, 0.5: 66.0, 1.0: 60.0, 2.0: 54.0, 3.0: 50.5}
SPEAKING_DISTANCES_M = tuple(SPEAKING_SPL_DB)
SPEAKING_SPL_REF_DB = 60.0          # the 1 m reference
VOCAL_EFFORT_DB = (-15.0, 12.0)    # whisper .. shouting


def spl_to_dbfs(spl_db: float) -> float:
    """Raw sensitivity mapping: 94 dB SPL (1 Pa) -> -26 dBFS, with no board gain.

    Note the consequence: a talker at 60 dB SPL lands near -60 dBFS RMS. In practice a
    board applies preamp/software gain to lift that into a usable range, so the absolute
    digital level depends on the design while the *ratio* between speech and the capsule
    noise floor does not. That is why the level is treated as a free variable below and
    the SNR is derived from distance instead.
    """
    return spl_db - INMP441_SPEC["full_scale_spl_db"] + INMP441_SPEC["sensitivity_dbfs"]


def effective_snr_db(spl_db: float) -> float:
    """Mic SNR as the deployed system sees it.

    The datasheet's 61 dBA is quoted at maximum sensitivity (i.e. for a sound pressure
    at the top of the useful range). Once the speech level drops -- because the user is
    3 m away, or whispering -- the capsule noise stays where it is and the achievable
    SNR falls with it. This is the effect that actually matters and it is invisible if
    the noise floor is modelled as a fixed offset.
    """
    return INMP441_SPEC["snr_dba"] + (spl_db - SPEAKING_SPL_REF_DB)


def draw_talker_conditions(rng) -> tuple:
    """Draws a talker distance, vocal effort and the resulting deployed conditions.

    Returns (level_dbfs, snr_db, spl_db): where the board's gain puts the speech in the
    digital domain, and the SNR the capsule delivers at that distance and loudness.
    """
    d = float(rng.choice(SPEAKING_DISTANCES_M))
    effort = float(rng.uniform(*VOCAL_EFFORT_DB))
    spl = SPEAKING_SPL_DB[d] + effort
    return float(rng.uniform(-50.0, -14.0)), effective_snr_db(spl), spl


def apply_inmp441_response(audio: np.ndarray) -> np.ndarray:
    """INMP441 acoustic response as seen inside the 100-7500 Hz feature band.

    The capsule is specified flat from 60 Hz to 15 kHz. Our feature band sits wholly
    inside that, so there is NO meaningful roll-off to apply: the only in-band effects
    are the high-pass corner just below f_min and the LDC low-frequency shelf, which
    lifts roughly 100-200 Hz by a few dB. Applying a generic MEMS darkening curve here
    would misrepresent the hardware.
    """
    a = np.asarray(audio, dtype=np.float32)
    # 2nd-order high-pass at the 60 Hz capsule corner (well below the 100 Hz f_min)
    b, c = signal.butter(2, INMP441_SPEC["f_low_hz"] / (SAMPLE_RATE / 2.0), btype="high")
    a = signal.lfilter(b, c, a).astype(np.float32)
    # LDC low-frequency lift. Needs a 2nd-order lowpass here: a 1-pole roll-off is only
    # -0.7 dB by 4 kHz, which turns the "shelf" into a flat gain boost across the whole
    # feature band instead of a shelf.
    b2, c2 = signal.butter(2, 250.0 / (SAMPLE_RATE / 2.0), btype="low")
    low = signal.lfilter(b2, c2, a).astype(np.float32)
    g = 10 ** (INMP441_SPEC["ldc_shelf_db"] / 20.0)
    return (a + (g - 1.0) * low).astype(np.float32)


def add_mic_self_noise(audio: np.ndarray, rng, snr_db: float = 61.0,
                       extra_db: float = 0.0) -> np.ndarray:
    """Adds the capsule's own electronic noise at a given SNR below the signal.

    Modelled as white noise with a mild 1/f tilt (real capsule self-noise is not white)
    plus per-unit sensitivity spread. The noise is placed relative to the signal level,
    not at a fixed absolute level, because the board's preamp lifts speech and self-noise
    together -- what degrades the microphone is distance and loudness, which is what
    `snr_db` already encodes.
    """
    a = np.asarray(audio, dtype=np.float32)
    n = len(a)
    sig_rms = float(np.sqrt(np.mean(a ** 2)))
    if sig_rms < 1e-9:
        sig_rms = 1e-9
    sig_dbfs = 20.0 * float(np.log10(sig_rms))
    floor_dbfs = (sig_dbfs - snr_db
                  + float(rng.normal(0.0, INMP441_SPEC["unit_spread_db"] / 3.0))
                  + extra_db)
    rms = float(10 ** (floor_dbfs / 20.0))
    noise = rng.standard_normal(n).astype(np.float32)
    # 1/f tilt: one-pole lowpass blend, keeps the floor from being pure white
    b, c = signal.butter(1, min(0.99, 2000.0 / (SAMPLE_RATE / 2.0)), btype="low")
    noise = 0.7 * signal.lfilter(b, c, noise) + 0.3 * noise
    cur = float(np.sqrt(np.mean(noise ** 2))) + 1e-12
    noise *= rms / cur
    return (a + noise).astype(np.float32)


def apply_i2s_quantization(audio: np.ndarray, bits: int = 24) -> np.ndarray:
    """I2S word quantization. At 24 bits the floor is about -146 dBFS, i.e. roughly
    59 dB below the capsule self-noise, so it is a no-op in practice. Kept because a
    deployment may read the 24-bit word through a 16-bit I2S peripheral, which throws
    away the low byte."""
    q = 2 ** bits / 2.0 - 0.5
    return (np.round((np.clip(audio, -1.0, 1.0) + 1.0) * q) / q - 1.0).astype(np.float32)


def apply_i2s_16bit_truncation(audio: np.ndarray) -> np.ndarray:
    """24-bit I2S word read by a 16-bit peripheral: the low byte is dropped, so the
    effective depth is 16 bits (floor about -98 dBFS, still under the capsule)."""
    return apply_i2s_quantization(audio, bits=16)



def generate_synthetic_rir(t60_sec: float = 0.35, sr: int = SAMPLE_RATE) -> np.ndarray:
    """
    Generates a synthetic Room Impulse Response (RIR) using exponentially
    decaying filtered noise with direct-path impulse and early reflections.
    """
    length = int(t60_sec * sr)
    if length < 100:
        length = 100

    # Time vector
    t = np.arange(length) / sr

    # Decay rate: -60 dB over t60 seconds -> decay envelope = exp(-6.91 * t / t60)
    decay = np.exp(-6.91 * t / t60_sec)

    # Gaussian noise for diffuse tail
    tail = np.random.randn(length) * decay

    # Low-pass filter the tail to simulate frequency-dependent wall absorption
    # (high frequencies decay faster in real rooms)
    b, a = signal.butter(1, 4000.0 / (sr / 2.0), btype='low')
    tail_filtered = signal.lfilter(b, a, tail)

    # Direct sound and early discrete reflections (image-source approximation)
    rir = np.zeros(length, dtype=np.float32)
    rir[0] = 1.0  # Direct path impulse

    # Add 4-6 early reflections with random delays and attenuation
    num_early = np.random.randint(4, 8)
    for _ in range(num_early):
        delay_ms = np.random.uniform(5.0, 45.0)
        delay_idx = int(delay_ms * 1e-3 * sr)
        if delay_idx < length:
            gain = np.random.uniform(0.15, 0.45) * (-1.0 if np.random.random() > 0.5 else 1.0)
            rir[delay_idx] += gain

    # Combine early reflections and diffuse late reverberation
    rir = rir + (tail_filtered * 0.35)

    # Normalize energy
    norm = np.sqrt(np.sum(rir ** 2)) + 1e-8
    return (rir / norm).astype(np.float32)


def apply_reverberation(audio: np.ndarray, t60_sec: float = None) -> np.ndarray:
    """Convolves audio with a synthetic Room Impulse Response."""
    if t60_sec is None:
        t60_sec = np.random.uniform(0.15, 0.65)
    rir = generate_synthetic_rir(t60_sec)
    rev = signal.fftconvolve(audio, rir, mode='full')[:len(audio)]
    # Match RMS of original audio
    orig_rms = np.sqrt(np.mean(audio ** 2)) + 1e-8
    rev_rms = np.sqrt(np.mean(rev ** 2)) + 1e-8
    return (rev * (orig_rms / rev_rms)).astype(np.float32)


def apply_mems_mic_response(audio: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    """Emulates hardware MEMS microphone frequency response and mild saturation."""
    # 1. High-pass filter (cut below 120 Hz to mimic MEMS DC roll-off)
    b_hp, a_hp = signal.butter(2, 120.0 / (sr / 2.0), btype='high')
    filtered = signal.lfilter(b_hp, a_hp, audio)

    # 2. Resonant band boost around 2.5 kHz (mic cavity acoustic resonance)
    center_freq = np.random.uniform(2200.0, 3200.0)
    q = 1.5
    b_peak, a_peak = signal.iirpeak(center_freq / (sr / 2.0), q)
    boosted = signal.lfilter(b_peak, a_peak, filtered)
    mixed = filtered * 0.75 + boosted * 0.25

    # 3. Non-linear analog saturation (soft clipping)
    saturated = np.tanh(mixed * 1.1) * 0.95
    return saturated.astype(np.float32)


def apply_distance_attenuation(audio: np.ndarray, distance_m: float = None, sr: int = SAMPLE_RATE) -> np.ndarray:
    """Simulates speech spoken at distance (0.5m to 5.0m) with high-frequency air loss."""
    if distance_m is None:
        distance_m = np.random.uniform(0.5, 4.5)

    # Air absorption dampens frequencies above 3.5 kHz progressively with distance
    cutoff = max(2000.0, 7500.0 - (distance_m * 900.0))
    b, a = signal.butter(1, cutoff / (sr / 2.0), btype='low')
    damped = signal.lfilter(b, a, audio)

    # Inverse square distance gain attenuation (normalized with dynamic range limit)
    gain = 1.0 / (1.0 + (distance_m - 0.5) * 0.3)
    return (damped * gain).astype(np.float32)


def apply_speed_perturbation(audio: np.ndarray, speed_factor: float = None) -> np.ndarray:
    """Changes speed of audio (0.85x to 1.15x) while maintaining 1.0s target length."""
    if speed_factor is None:
        speed_factor = np.random.uniform(0.85, 1.15)
    if abs(speed_factor - 1.0) < 0.02:
        return audio

    n = len(audio)
    resampled_len = int(round(n / speed_factor))
    x_old = np.linspace(0.0, 1.0, n, endpoint=False)
    x_new = np.linspace(0.0, 1.0, resampled_len, endpoint=False)
    resampled = np.interp(x_new, x_old, audio).astype(np.float32)

    # Fit back into original length with center padding / trimming
    if len(resampled) < n:
        pad_l = (n - len(resampled)) // 2
        pad_r = n - len(resampled) - pad_l
        resampled = np.pad(resampled, (pad_l, pad_r), mode='constant')
    else:
        start = (len(resampled) - n) // 2
        resampled = resampled[start:start + n]

    return resampled


def apply_spec_augment(spectrogram: np.ndarray, max_freq_mask: int = 6, max_time_mask: int = 8) -> np.ndarray:
    """
    Applies SpecAugment directly on a (49, 40) Log-Mel spectrogram:
    - Frequency channel masking
    - Time frame masking
    """
    spec = spectrogram.copy()
    num_time, num_mel = spec.shape

    # 1. Frequency masking
    f_width = np.random.randint(1, max_freq_mask + 1)
    f_start = np.random.randint(0, num_mel - f_width) if (num_mel - f_width > 0) else 0
    spec[:, f_start:f_start + f_width] = np.min(spec)

    # 2. Time masking
    t_width = np.random.randint(1, max_time_mask + 1)
    t_start = np.random.randint(0, num_time - t_width) if (num_time - t_width > 0) else 0
    spec[t_start:t_start + t_width, :] = np.min(spec)

    return spec


def apply_full_acoustic_chain(audio: np.ndarray, noise_slice: np.ndarray = None, snr_db: float = None) -> np.ndarray:
    """
    Applies the full physical acoustic propagation chain:
    Original -> Speed Jitter -> Room Reverberation -> Distance Loss -> Mic Emulation -> Noise Mix
    """
    # 1. Speed perturbation (80% probability)
    if np.random.random() < 0.80:
        audio = apply_speed_perturbation(audio)

    # 2. Room reverberation (70% probability)
    if np.random.random() < 0.70:
        audio = apply_reverberation(audio)

    # 3. Distance decay & air absorption (70% probability)
    if np.random.random() < 0.70:
        audio = apply_distance_attenuation(audio)

    # 4. Hardware MEMS mic filtering & saturation (90% probability)
    if np.random.random() < 0.90:
        audio = apply_mems_mic_response(audio)

    # 5. Background noise mixing if provided
    if noise_slice is not None and len(noise_slice) == len(audio):
        if snr_db is None:
            snr_db = np.random.uniform(-4.0, 22.0)
        sig_pow = np.mean(audio ** 2) + 1e-8
        noise_pow = np.mean(noise_slice ** 2) + 1e-8
        target_noise_pow = sig_pow / (10 ** (snr_db / 10.0))
        scale = np.sqrt(target_noise_pow / noise_pow)
        audio = audio + (noise_slice * scale)

    # Peak normalization
    max_val = np.max(np.abs(audio))
    if max_val > 0.95:
        audio = audio / max_val * 0.95

    return audio.astype(np.float32)


def mix_dual_independent_noises(speech: np.ndarray, noise1: np.ndarray, noise2: np.ndarray, snr1_db: float, snr2_db: float) -> np.ndarray:
    """Cocktail party mixture: speech + stationary ambient noise1 + transient noise2."""
    sig_pow = np.mean(speech ** 2) + 1e-8

    p1 = np.mean(noise1 ** 2) + 1e-8
    target_p1 = sig_pow / (10 ** (snr1_db / 10.0))
    scale1 = np.sqrt(target_p1 / p1)

    p2 = np.mean(noise2 ** 2) + 1e-8
    target_p2 = sig_pow / (10 ** (snr2_db / 10.0))
    scale2 = np.sqrt(target_p2 / p2)

    mixed = speech + (noise1 * scale1) + (noise2 * scale2)
    max_val = np.max(np.abs(mixed))
    if max_val > 0.95:
        mixed = mixed / max_val * 0.95
    return mixed.astype(np.float32)


## Model architectures

Includes BCConformerV2: learned absolute temporal embedding, attention pooling over mel, in-model delta/delta-delta, multi-scale depthwise convs, attentive statistics pooling. 86,450 params, same (49, 40, 1) input contract as the 50K model.

*Inlined from `models.py`: 733 lines, 0 CLI lines and 0 cross-module imports removed — everything shares one namespace here.*


In [8]:
"""
State-of-the-Art TinyML Keyword Spotting Model Architectures in Flax / JAX
--------------------------------------------------------------------------
Implements 4 premier architectures for microcontroller edge deployment (< 256 KB RAM):

1. BC-ResNet (Broadcasting-Residual Network - Kim et al., Interspeech 2021)
   - BC-ResNet-1 (5.2k params) & BC-ResNet-3 (18.5k params)
   - Splits 2D convs into 1D temporal depthwise & 1D frequency depthwise
   - Injects global sub-band frequency context via broadcasting
   - SOTA accuracy on Speech Commands & noisy wake-word benchmarks

2. TC-ResNet (Temporal Convolutional ResNet - Choi et al., 2019)
   - 1D Temporal Dilated Convolutions along the time axis
   - Extremely cache-efficient and fast on Xtensa / ARM Cortex SIMD DSPs
   - Preserves temporal phoneme transition dynamics

3. DS-ResNet-SE (Squeeze-and-Excitation Residual Network)
   - Depthwise Separable Convolutions augmented with Squeeze-and-Excitation (SE)
   - Dynamically recalibrates Mel-frequency channel weights
   - Suppresses background noise bands while boosting keyword formant frequencies

4. DS-CNN-S (Depthwise Separable CNN - Zhang et al., ARM Hello Edge)
   - Classic Industry-standard TinyML benchmark

All models utilize GroupNorm for robust convergence across small batch sizes and noise.
"""

from typing import Sequence, Tuple
import jax
import jax.numpy as jnp
from flax import linen as nn


# ==============================================================================
# 1. BC-ResNet (Broadcasting Residual Network - Kim et al., Interspeech 2021)
# ==============================================================================

class BCResBlock(nn.Module):
    """
    Broadcasting-Residual Block:
    - 1D Frequency Depthwise Conv (1, 3)
    - 1D Temporal Depthwise Conv (3, 1)
    - Frequency Broadcasting: pools frequency channel context and broadcasts to time
    - 1x1 Pointwise Conv
    - Skip connection with average pooling if stride > 1
    """
    channels: int
    stride: int = 1

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # Residual branch
        if self.stride != 1 or x.shape[-1] != self.channels:
            res = nn.avg_pool(x, window_shape=(self.stride, 1), strides=(self.stride, 1), padding='SAME')
            if res.shape[-1] != self.channels:
                res = nn.Conv(features=self.channels, kernel_size=(1, 1), use_bias=False)(res)
                res = nn.GroupNorm(num_groups=min(4, self.channels))(res)
        else:
            res = x

        # 1. Frequency Depthwise Conv (1, 3)
        h = nn.Conv(
            features=x.shape[-1],
            kernel_size=(1, 3),
            strides=(1, 1),
            padding='SAME',
            feature_group_count=x.shape[-1],
            use_bias=False
        )(x)
        h = nn.GroupNorm(num_groups=min(4, x.shape[-1]))(h)
        h = nn.relu(h)

        # 2. Temporal Depthwise Conv (3, 1)
        h = nn.Conv(
            features=x.shape[-1],
            kernel_size=(3, 1),
            strides=(self.stride, 1),
            padding='SAME',
            feature_group_count=x.shape[-1],
            use_bias=False
        )(h)
        h = nn.GroupNorm(num_groups=min(4, x.shape[-1]))(h)
        h = nn.relu(h)

        # 3. Frequency Broadcast Connection (injects global frequency context)
        f_broadcast = jnp.mean(h, axis=2, keepdims=True)
        h = h + f_broadcast

        # 4. Pointwise Conv (1, 1) to target channels
        h = nn.Conv(features=self.channels, kernel_size=(1, 1), strides=(1, 1), use_bias=False)(h)
        h = nn.GroupNorm(num_groups=min(4, self.channels))(h)

        return nn.relu(h + res)


class BCResNet(nn.Module):
    """
    BC-ResNet Keyword Spotting Architecture.
    scale=1: BC-ResNet-1 (~5.2k params)
    scale=3: BC-ResNet-3 (~18.5k params)
    """
    num_classes: int = 2
    scale: int = 1

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        base_c = 16 if self.scale == 1 else 24

        # Stem: Conv2D with stride (2, 2)
        h = nn.Conv(
            features=base_c,
            kernel_size=(3, 3),
            strides=(2, 2),
            padding='SAME',
            use_bias=False
        )(x)
        h = nn.GroupNorm(num_groups=min(4, base_c))(h)
        h = nn.relu(h)

        # Stage 1
        h = BCResBlock(channels=base_c, stride=1)(h, train=train)
        h = BCResBlock(channels=base_c, stride=1)(h, train=train)

        # Stage 2
        stage2_c = base_c * 2
        h = BCResBlock(channels=stage2_c, stride=2)(h, train=train)
        h = BCResBlock(channels=stage2_c, stride=1)(h, train=train)

        # Stage 3
        if self.scale >= 3:
            stage3_c = base_c * 3
            h = BCResBlock(channels=stage3_c, stride=2)(h, train=train)
            h = BCResBlock(channels=stage3_c, stride=1)(h, train=train)
        else:
            h = BCResBlock(channels=stage2_c, stride=1)(h, train=train)

        # Global Average Pooling over (Time, Frequency) axes (1, 2)
        h = jnp.mean(h, axis=(1, 2))

        # Dropout
        h = nn.Dropout(rate=0.15, deterministic=not train)(h)

        # Dense projection to class logits
        logits = nn.Dense(features=self.num_classes)(h)
        return logits


# ==============================================================================
# 2. TC-ResNet (Temporal Convolutional ResNet - Choi et al., 2019)
# ==============================================================================

class TCResBlock(nn.Module):
    """1D Temporal Residual Block with dilation."""
    channels: int
    stride: int = 1
    dilation: int = 1

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # Residual shortcut
        if self.stride != 1 or x.shape[-1] != self.channels:
            res = nn.Conv(features=self.channels, kernel_size=(1,), strides=(self.stride,), use_bias=False)(x)
            res = nn.GroupNorm(num_groups=min(4, self.channels))(res)
        else:
            res = x

        # 1D Temporal Conv 1
        h = nn.Conv(
            features=self.channels,
            kernel_size=(3,),
            strides=(self.stride,),
            kernel_dilation=(self.dilation,),
            padding='SAME',
            use_bias=False
        )(x)
        h = nn.GroupNorm(num_groups=min(4, self.channels))(h)
        h = nn.relu(h)

        # 1D Temporal Conv 2
        h = nn.Conv(
            features=self.channels,
            kernel_size=(3,),
            strides=(1,),
            kernel_dilation=(1,),
            padding='SAME',
            use_bias=False
        )(h)
        h = nn.GroupNorm(num_groups=min(4, self.channels))(h)

        return nn.relu(h + res)


class TCResNet(nn.Module):
    """
    Temporal Convolutional ResNet (TC-ResNet-8):
    Treats the 40 Mel filterbanks as input feature channels, convolving solely along time.
    Parameters: ~15,200 (Flash: ~15 KB INT8)
    """
    num_classes: int = 2
    base_channels: int = 24

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # x input shape: (Batch, Time=49, Mel=40, Channels=1)
        # Reshape to (Batch, Time=49, Channels=40)
        b, t, m, c = x.shape
        x_1d = jnp.reshape(x, (b, t, m * c))

        # Stem 1D Conv
        h = nn.Conv(features=self.base_channels, kernel_size=(3,), strides=(1,), padding='SAME', use_bias=False)(x_1d)
        h = nn.GroupNorm(num_groups=min(4, self.base_channels))(h)
        h = nn.relu(h)

        # Stage 1: 2 blocks (stride 1, 2)
        h = TCResBlock(channels=self.base_channels, stride=1, dilation=1)(h, train=train)
        h = TCResBlock(channels=self.base_channels * 2, stride=2, dilation=1)(h, train=train)

        # Stage 2: 2 blocks (stride 1, 2) with dilation
        h = TCResBlock(channels=self.base_channels * 2, stride=1, dilation=2)(h, train=train)
        h = TCResBlock(channels=self.base_channels * 3, stride=2, dilation=2)(h, train=train)

        # Global Average Pooling across time dimension (axis 1)
        h = jnp.mean(h, axis=1)

        # Dropout & Classifier
        h = nn.Dropout(rate=0.15, deterministic=not train)(h)
        logits = nn.Dense(features=self.num_classes)(h)
        return logits


# ==============================================================================
# 3. DS-ResNet-SE (Depthwise Separable ResNet with Squeeze-and-Excitation)
# ==============================================================================

class SEBlock(nn.Module):
    """Squeeze-and-Excitation Channel Attention."""
    reduction: int = 4

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        channels = x.shape[-1]
        # Squeeze: Global average pooling over (H, W)
        z = jnp.mean(x, axis=(1, 2), keepdims=True)
        # Excitation: Two-layer MLP with Sigmoid gating
        s = nn.Dense(features=max(4, channels // self.reduction), use_bias=False)(z)
        s = nn.relu(s)
        s = nn.Dense(features=channels, use_bias=False)(s)
        s = nn.sigmoid(s)
        return x * s


class DSResBlockSE(nn.Module):
    """Depthwise Separable Block with Squeeze-and-Excitation."""
    channels: int
    stride: int = 1

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # Residual branch
        if self.stride != 1 or x.shape[-1] != self.channels:
            res = nn.Conv(features=self.channels, kernel_size=(1, 1), strides=(self.stride, self.stride), use_bias=False)(x)
            res = nn.GroupNorm(num_groups=min(4, self.channels))(res)
        else:
            res = x

        # Depthwise 3x3 Conv
        h = nn.Conv(
            features=x.shape[-1],
            kernel_size=(3, 3),
            strides=(self.stride, self.stride),
            padding='SAME',
            feature_group_count=x.shape[-1],
            use_bias=False
        )(x)
        h = nn.GroupNorm(num_groups=min(4, x.shape[-1]))(h)
        h = nn.relu(h)

        # Pointwise 1x1 Conv
        h = nn.Conv(features=self.channels, kernel_size=(1, 1), strides=(1, 1), use_bias=False)(h)
        h = nn.GroupNorm(num_groups=min(4, self.channels))(h)

        # Squeeze-and-Excitation Attention
        h = SEBlock(reduction=4)(h)

        return nn.relu(h + res)


class DSResNetSE(nn.Module):
    """
    Depthwise Separable ResNet with Squeeze-and-Excitation Channel Attention.
    Parameters: ~16,800 (Flash: ~16.8 KB INT8)
    """
    num_classes: int = 2
    base_channels: int = 20

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # Stem
        h = nn.Conv(features=self.base_channels, kernel_size=(3, 3), strides=(2, 2), padding='SAME', use_bias=False)(x)
        h = nn.GroupNorm(num_groups=min(4, self.base_channels))(h)
        h = nn.relu(h)

        # Stages
        h = DSResBlockSE(channels=self.base_channels, stride=1)(h, train=train)
        h = DSResBlockSE(channels=self.base_channels * 2, stride=2)(h, train=train)
        h = DSResBlockSE(channels=self.base_channels * 2, stride=1)(h, train=train)
        h = DSResBlockSE(channels=self.base_channels * 3, stride=2)(h, train=train)

        # Global Average Pooling
        h = jnp.mean(h, axis=(1, 2))
        h = nn.Dropout(rate=0.15, deterministic=not train)(h)
        logits = nn.Dense(features=self.num_classes)(h)
        return logits


# ==============================================================================
# 4. DS-CNN-S (Depthwise Separable CNN - ARM Hello Edge)
# ==============================================================================

class DepthwiseSeparableBlock(nn.Module):
    """Depthwise 2D Conv (3, 3) followed by Pointwise 1x1 Conv."""
    channels: int
    stride: int = 1

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        h = nn.Conv(
            features=x.shape[-1],
            kernel_size=(3, 3),
            strides=(self.stride, self.stride),
            padding='SAME',
            feature_group_count=x.shape[-1],
            use_bias=False
        )(x)
        h = nn.GroupNorm(num_groups=min(4, x.shape[-1]))(h)
        h = nn.relu(h)

        h = nn.Conv(
            features=self.channels,
            kernel_size=(1, 1),
            strides=(1, 1),
            padding='SAME',
            use_bias=False
        )(h)
        h = nn.GroupNorm(num_groups=min(4, self.channels))(h)
        h = nn.relu(h)
        return h


class DSCNN(nn.Module):
    """DS-CNN-S Architecture."""
    num_classes: int = 2
    num_layers: int = 4
    num_filters: int = 32

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        h = nn.Conv(features=self.num_filters, kernel_size=(10, 4), strides=(2, 2), padding='SAME', use_bias=False)(x)
        h = nn.GroupNorm(num_groups=min(4, self.num_filters))(h)
        h = nn.relu(h)

        for _ in range(self.num_layers):
            h = DepthwiseSeparableBlock(channels=self.num_filters, stride=1)(h, train=train)

        h = jnp.mean(h, axis=(1, 2))
        h = nn.Dropout(rate=0.20, deterministic=not train)(h)
        logits = nn.Dense(features=self.num_classes)(h)
        return logits


# ==============================================================================
# 5. MH-KWS / Conformer-Lite (Multi-Head Attention + Temporal Depthwise Conv)
# ==============================================================================

class ConformerLiteBlock(nn.Module):
    """
    Lightweight Conformer block with Multi-Head Self-Attention across time
    and depthwise temporal convolutions to capture phonetic sequence dynamics.
    """
    dim: int = 32
    num_heads: int = 4

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # 1. Multi-Head Self-Attention over time frames
        norm1 = nn.LayerNorm()(x)
        attn = nn.SelfAttention(num_heads=self.num_heads, qkv_features=self.dim)(norm1)
        x = x + nn.Dropout(rate=0.10, deterministic=not train)(attn)

        # 2. Depthwise Conv Module (Temporal dynamics of phonemes)
        norm2 = nn.LayerNorm()(x)
        conv = nn.Conv(features=self.dim, kernel_size=(5,), padding='SAME', feature_group_count=self.dim, use_bias=False)(norm2)
        conv = nn.relu(conv)
        conv = nn.Conv(features=self.dim, kernel_size=(1,), use_bias=False)(conv)
        x = x + nn.Dropout(rate=0.10, deterministic=not train)(conv)

        # 3. Feed-Forward Network
        norm3 = nn.LayerNorm()(x)
        ff = nn.Dense(features=self.dim * 2)(norm3)
        ff = nn.relu(ff)
        ff = nn.Dense(features=self.dim)(ff)
        x = x + nn.Dropout(rate=0.10, deterministic=not train)(ff)
        return x


class ConformerLiteKWS(nn.Module):
    """
    TinyML Multi-Head Attention Conformer for Keyword Spotting.
    Parameters: ~29,150 (Flash: ~28.5 KB INT8)
    Explicitly prevents acoustic false triggers (sirens, barks, speech babble)
    by strictly learning the cross-temporal attention transitions of /ə/ -> /m/ -> /eɪ/ -> /z/.
    """
    num_classes: int = 2
    dim: int = 32

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # x is (B, 49, 40, 1)
        # Collapse frequency into embedding
        x_proj = nn.Conv(features=self.dim, kernel_size=(3, 3), strides=(1, 2), padding='SAME', use_bias=False)(x)
        x_proj = nn.relu(x_proj)
        x_proj = nn.Conv(features=self.dim, kernel_size=(3, 3), strides=(1, 2), padding='SAME', use_bias=False)(x_proj)
        x_proj = nn.relu(x_proj)
        seq = jnp.mean(x_proj, axis=2)  # (B, 49, dim)

        # Conformer blocks
        seq = ConformerLiteBlock(dim=self.dim)(seq, train=train)
        seq = ConformerLiteBlock(dim=self.dim)(seq, train=train)

        # Global temporal pooling
        h = jnp.mean(seq, axis=1)
        logits = nn.Dense(features=self.num_classes)(h)
        return logits


# ==============================================================================
# 6. BC-Conformer-50K (Broadcasting Convolutions + Multi-Head Self-Attention)
# ==============================================================================

class TemporalConformerBlock50K(nn.Module):
    """Temporal Conformer block with 4-head Multi-Head Self-Attention and depthwise conv."""
    dim: int = 44
    num_heads: int = 4

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        norm1 = nn.LayerNorm()(x)
        attn = nn.SelfAttention(num_heads=self.num_heads, qkv_features=self.dim)(norm1)
        x = x + nn.Dropout(rate=0.15, deterministic=not train)(attn)

        norm2 = nn.LayerNorm()(x)
        conv = nn.Conv(features=self.dim, kernel_size=(5,), padding='SAME', feature_group_count=self.dim, use_bias=False)(norm2)
        conv = nn.relu(conv)
        conv = nn.Conv(features=self.dim, kernel_size=(1,), use_bias=False)(conv)
        x = x + nn.Dropout(rate=0.15, deterministic=not train)(conv)

        norm3 = nn.LayerNorm()(x)
        ff = nn.Dense(features=self.dim * 2)(norm3)
        ff = nn.relu(ff)
        ff = nn.Dense(features=self.dim)(ff)
        x = x + nn.Dropout(rate=0.15, deterministic=not train)(ff)
        return x


class BCConformer50K(nn.Module):
    """
    State-of-the-Art 50 KB KWS Architecture:
    Broadcasting Sub-Band Convolutions + Multi-Head Temporal Self-Attention.
    Parameters: ~49,880 (Flash: ~48.7 KB INT8)
    Arena RAM: ~42 KB (well below 256 KB ESP32-S3 limit).
    Combines spatial frequency broadcasting with strict multi-phoneme temporal attention.

    `conformer_dim` and `blocks` are exposed so the same topology can be scaled up
    when a larger training corpus is available (see BCConformer100K).
    """
    num_classes: int = 2
    channels: int = 32
    conformer_dim: int = 44
    blocks: int = 2

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # x is (B, 49, 40, 1)
        # Stage 1: Frequency Stem
        h = nn.Conv(features=self.channels, kernel_size=(3, 3), strides=(1, 2), padding='SAME', use_bias=False)(x)
        h = nn.GroupNorm(num_groups=4)(h)
        h = nn.relu(h)

        # Stage 2: Sub-Band Frequency Depthwise Conv + Broadcasting
        f_conv = nn.Conv(features=self.channels, kernel_size=(1, 3), padding='SAME', feature_group_count=self.channels, use_bias=False)(h)
        f_conv = nn.GroupNorm(num_groups=4)(f_conv)
        f_conv = nn.relu(f_conv)
        f_bcast = jnp.mean(f_conv, axis=2, keepdims=True)
        h = h + f_conv + f_bcast

        # Stage 3: Frequency projection to Conformer temporal sequence
        h = nn.Conv(features=self.conformer_dim, kernel_size=(3, 3), strides=(1, 2), padding='SAME', use_bias=False)(h)
        h = nn.GroupNorm(num_groups=4)(h)
        h = nn.relu(h)
        seq = jnp.mean(h, axis=2)  # (B, 49, conformer_dim)

        # Stage 4: Conformer Blocks
        for _ in range(self.blocks):
            seq = TemporalConformerBlock50K(dim=self.conformer_dim, num_heads=4)(seq, train=train)

        # Stage 5: Dual Pooling & Classification
        pooled = jnp.mean(seq, axis=1)
        max_pooled = jnp.max(seq, axis=1)
        feat = jnp.concatenate([pooled, max_pooled], axis=-1)
        feat = nn.Dropout(rate=0.20, deterministic=not train)(feat)
        logits = nn.Dense(features=self.num_classes)(feat)
        return logits


class BCConformer100K(BCConformer50K):
    """Scaled BC-Conformer (~130k params). Still ~130 KB INT8 flash, which fits the
    ESP32-S3 comfortably, and has the headroom a 40k-sample corpus needs."""
    channels: int = 40
    conformer_dim: int = 64
    blocks: int = 3


# ==============================================================================
# 7. BC-Conformer-V2 (position-aware, multi-scale, gated)
# ==============================================================================
# Four concrete weaknesses in BCConformer50K were fixed rather than guessed at:
#
#   1. NO POSITIONAL INFORMATION. TemporalConformerBlock50K uses bare
#      nn.SelfAttention, which is permutation-equivariant over time -- the model
#      structurally cannot tell where in the 1 s window a frame sits. That matters
#      enormously here, because the streaming engine slides the window every 100 ms
#      and measured recall falls from 40% to 14% as the keyword moves from the start
#      of the window to the end. V2 adds a learned absolute temporal embedding.
#
#   2. MEAN POOLING OVER MEL. Stage 3 collapsed the frequency axis with
#      jnp.mean(h, axis=2), discarding all sub-band structure after two strided
#      convs. V2 replaces it with a learned attention pool over mel bins, so the
#      network chooses which bands matter for the keyword's formants.
#
#   3. NO INPUT DERIVATIVES. Speech discrimination leans heavily on spectral
#      dynamics; static log-mel alone is a weaker feature. V2 computes the first and
#      second time-derivatives (delta / delta-delta) inside the model with fixed
#      kernels, so the dataset format is unchanged and every existing corpus still
#      loads, but the first learned convolution sees 3 channels instead of 1.
#
#   4. WEAK TEMPORAL BLOCK. V2 widens to dim 48, uses two depthwise kernel sizes
#      (3 and 7) so it sees both phoneme transitions and longer context, replaces the
#      ReLU feed-forward with Swish (the Conformer standard) and adds a squeeze-
#      excitation gate after the convolution. Three blocks instead of two.
#
# Pooling also moves from concat(mean, max) to attentive statistics pooling, which
# learns how much weight to give each frame instead of weighting all frames equally.

def _delta_stack(x: jnp.ndarray) -> jnp.ndarray:
    """(B, T, M, 1) log-mel -> (B, T, M, 3) [log-mel, delta, delta-delta].

    Computed with fixed (non-learned) kernels inside the model so the on-disk corpus
    format stays (T, M, 1) and existing datasets remain loadable.
    """
    t = x[:, :, :, 0]
    d1_pad = jnp.pad(t, ((0, 0), (1, 1), (0, 0)), mode="edge")
    d1 = 0.5 * (d1_pad[:, 2:, :] - d1_pad[:, :-2, :])
    d2_pad = jnp.pad(d1, ((0, 0), (1, 1), (0, 0)), mode="edge")
    d2 = 0.5 * (d2_pad[:, 2:, :] - d2_pad[:, :-2, :])
    return jnp.stack([t, d1, d2], axis=-1)


class ImprovedConformerBlock(nn.Module):
    """Conformer block: attention -> multi-scale depthwise conv -> Swish FFN, with an
    SE gate on the convolution output and a residual around each sub-layer."""
    dim: int = 48
    num_heads: int = 4
    se_reduction: int = 8
    dropout: float = 0.15

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # --- Multi-head self-attention ---
        # Positional information enters through the learned absolute temporal embedding
        # added once before the stack (see BCConformerV2), which is what makes this
        # attention position-aware; flax 0.12's SelfAttention has no position_bias arg.
        norm1 = nn.LayerNorm()(x)
        attn = nn.SelfAttention(
            num_heads=self.num_heads,
            qkv_features=self.dim,
        )(norm1)
        x = x + nn.Dropout(rate=self.dropout, deterministic=not train)(attn)

        # --- Multi-scale depthwise convolution + pointwise mixing + SE gate ---
        norm2 = nn.LayerNorm()(x)
        conv3 = nn.Conv(features=self.dim, kernel_size=(3,), padding="SAME",
                        feature_group_count=self.dim, use_bias=False)(norm2)
        conv7 = nn.Conv(features=self.dim, kernel_size=(7,), padding="SAME",
                        feature_group_count=self.dim, use_bias=False)(norm2)
        y = jax.nn.swish(conv3) + jax.nn.swish(conv7)
        y = nn.Conv(features=self.dim, kernel_size=(1,), use_bias=False)(y)
        # Squeeze-and-excitation over time: recalibrate which frames matter.
        # z is (B, 1, D) and broadcasts directly against y (B, T, D).
        z = jnp.mean(y, axis=1, keepdims=True)
        z = nn.Dense(features=max(4, self.dim // self.se_reduction), use_bias=False)(z)
        z = jax.nn.swish(z)
        z = nn.Dense(features=self.dim, use_bias=False)(z)
        y = y * jax.nn.sigmoid(z)
        x = x + nn.Dropout(rate=self.dropout, deterministic=not train)(y)

        # --- Swish feed-forward (Conformer standard; ReLU FFN is measurably weaker) ---
        norm3 = nn.LayerNorm()(x)
        ff = nn.Dense(features=self.dim * 2)(norm3)
        ff = jax.nn.swish(ff)
        ff = nn.Dense(features=self.dim)(ff)
        x = x + nn.Dropout(rate=self.dropout, deterministic=not train)(ff)
        return x


class AttentiveStatsPool(nn.Module):
    """Attentive statistics pooling: learned attention weights over time, then mean and
    std of the weighted sequence. Replaces concat(mean, max), which weights every frame
    equally and cannot express "the keyword is somewhere in here"."""
    dim: int = 48
    attn_dim: int = 32

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        # x: (B, T, D)
        score = nn.Dense(features=self.attn_dim)(x)
        score = jax.nn.tanh(score)
        score = nn.Dense(features=1, use_bias=False)(score)   # (B, T, 1)
        w = nn.softmax(score, axis=1)
        mean = jnp.sum(x * w, axis=1)
        var = jnp.sum((x ** 2) * w, axis=1) - mean ** 2
        std = jnp.sqrt(jnp.maximum(var, 1e-6))
        return jnp.concatenate([mean, std], axis=-1)          # (B, 2D)


class BCConformerV2(nn.Module):
    """BC-Conformer V2 -- ~86k params, ~84 KB INT8, still far inside the ESP32-S3
    8 MB flash / 512 KB SRAM envelope. Same (49, 40, 1) log-mel input contract as
    BCConformer50K, so every existing dataset and checkpoint loader still works."""
    num_classes: int = 2
    channels: int = 32
    conformer_dim: int = 48
    blocks: int = 3
    num_heads: int = 4
    dropout: float = 0.20

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # x is (B, 49, 40, 1) log-mel
        # Stage 0: static log-mel -> [log-mel, delta, delta-delta]
        h = _delta_stack(x)                                 # (B, 49, 40, 3)

        # Stage 1: frequency stem
        h = nn.Conv(features=self.channels, kernel_size=(3, 3), strides=(1, 2),
                    padding="SAME", use_bias=False)(h)
        h = nn.GroupNorm(num_groups=4)(h)
        h = jax.nn.swish(h)                                 # (B, 49, 20, 32)

        # Stage 2: sub-band depthwise conv + broadcasting (kept from v1)
        f_conv = nn.Conv(features=self.channels, kernel_size=(1, 3), padding="SAME",
                         feature_group_count=self.channels, use_bias=False)(h)
        f_conv = nn.GroupNorm(num_groups=4)(f_conv)
        f_conv = jax.nn.swish(f_conv)
        f_bcast = jnp.mean(f_conv, axis=2, keepdims=True)
        h = h + f_conv + f_bcast

        # Stage 3: project frequency into the temporal sequence
        h = nn.Conv(features=self.conformer_dim, kernel_size=(3, 3), strides=(1, 2),
                    padding="SAME", use_bias=False)(h)
        h = nn.GroupNorm(num_groups=4)(h)
        h = jax.nn.swish(h)                                 # (B, 49, 10, 48)

        # Learned attention pooling over mel instead of jnp.mean -> keeps sub-band
        # structure that v1 threw away.
        mel_score = nn.Conv(features=1, kernel_size=(1,), use_bias=False)(h)   # (B,49,10,1)
        mel_w = nn.softmax(mel_score, axis=2)
        seq = jnp.sum(h * mel_w, axis=2)                    # (B, 49, 48)

        # Learned absolute temporal embedding: attention above is permutation
        # equivariant, and the keyword's position inside the 1 s window is the single
        # biggest driver of the offset stress result.
        seq = seq + self.param("pos_emb", nn.initializers.normal(0.02),
                               (seq.shape[1], self.conformer_dim))

        # Stage 4: improved conformer stack
        for _ in range(self.blocks):
            seq = ImprovedConformerBlock(dim=self.conformer_dim,
                                         num_heads=self.num_heads)(seq, train=train)

        # Stage 5: attentive statistics pooling + classification
        feat = AttentiveStatsPool(dim=self.conformer_dim)(seq)
        feat = nn.Dropout(rate=self.dropout, deterministic=not train)(feat)
        return nn.Dense(features=self.num_classes)(feat)


class RelPosSelfAttention(nn.Module):
    """Multi-head self-attention with a learned RELATIVE position bias (T5 style).

    Motivation, measured rather than assumed. BCConformerV2 adds a learned *absolute*
    temporal embedding before the stack, on the theory that it would make the network
    position-aware and therefore position-robust. The opposite happened: a single global
    pool over a fixed 1 s window has to guess where the keyword is, and an absolute
    position parameter is free to encode a prior about that guess. Validation recall fell
    monotonically 77.7% -> 15.1% across the window, a 62.6-point spread, against 26.5
    points for the 50K model that has no such embedding.

    A relative bias is the standard fix. Attention logits get b[i - j + (T-1)] added, which
    depends only on the distance between two frames, so shifting the whole sequence leaves
    every logit unchanged. The bias is a translation-equivariant prior over *distances*
    (prefer nearby context) and cannot express "the keyword belongs at frame 14".

    Cost: (2T-1) x num_heads learned values = 97 x 4 = 388 parameters.
    """

    num_heads: int = 4
    features: int = 48
    dropout: float = 0.0

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        b, t, _ = x.shape
        d = x.shape[-1]
        h = self.num_heads
        if d % h:
            raise ValueError(f"features {d} not divisible by num_heads {h}")
        dh = d // h

        qkv = nn.Dense(features=3 * d)(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)
        # (B, T, H, dh) -> (B, H, T, dh)
        q = jnp.transpose(q.reshape(b, t, h, dh), (0, 2, 1, 3))
        k = jnp.transpose(k.reshape(b, t, h, dh), (0, 2, 1, 3))
        v = jnp.transpose(v.reshape(b, t, h, dh), (0, 2, 1, 3))

        logits = jnp.einsum("bhid,bhjd->bhij", q, k) / jnp.sqrt(jnp.asarray(dh, jnp.float32))

        # Relative position bias, indexed by the signed distance i - j.
        rel = jnp.arange(t)[:, None] - jnp.arange(t)[None, :]      # (T, T)
        rel = rel + (t - 1)                                          # shift into [0, 2T-2]
        bias = self.param("rel_bias", nn.initializers.zeros, (2 * t - 1, h))
        logits = logits + bias[rel].transpose(2, 0, 1)[None]        # (B, H, T, T)

        w = jax.nn.softmax(logits, axis=-1)
        w = nn.Dropout(rate=self.dropout, deterministic=not train)(w)
        out = jnp.einsum("bhij,bhjd->bhid", w, v)
        out = jnp.transpose(out, (0, 2, 1, 3)).reshape(b, t, d)
        out = nn.Dense(features=d)(out)
        return nn.Dropout(rate=self.dropout, deterministic=not train)(out)


class RelConformerBlock(nn.Module):
    """Identical to ImprovedConformerBlock except that attention carries a relative
    position bias instead of relying on an absolute embedding added before the stack."""

    dim: int = 48
    num_heads: int = 4
    se_reduction: int = 8
    dropout: float = 0.15

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        norm1 = nn.LayerNorm()(x)
        attn = RelPosSelfAttention(num_heads=self.num_heads, features=self.dim,
                                   dropout=self.dropout)(norm1, train=train)
        x = x + nn.Dropout(rate=self.dropout, deterministic=not train)(attn)

        norm2 = nn.LayerNorm()(x)
        conv3 = nn.Conv(features=self.dim, kernel_size=(3,), padding="SAME",
                        feature_group_count=self.dim, use_bias=False)(norm2)
        conv7 = nn.Conv(features=self.dim, kernel_size=(7,), padding="SAME",
                        feature_group_count=self.dim, use_bias=False)(norm2)
        y = jax.nn.swish(conv3) + jax.nn.swish(conv7)
        y = nn.Conv(features=self.dim, kernel_size=(1,), use_bias=False)(y)
        z = jnp.mean(y, axis=1, keepdims=True)
        z = nn.Dense(features=max(4, self.dim // self.se_reduction), use_bias=False)(z)
        z = jax.nn.swish(z)
        z = nn.Dense(features=self.dim, use_bias=False)(z)
        y = y * jax.nn.sigmoid(z)
        x = x + nn.Dropout(rate=self.dropout, deterministic=not train)(y)

        norm3 = nn.LayerNorm()(x)
        ff = nn.Dense(features=self.dim * 2)(norm3)
        ff = jax.nn.swish(ff)
        ff = nn.Dense(features=self.dim)(ff)
        x = x + nn.Dropout(rate=self.dropout, deterministic=not train)(ff)
        return x


class SoftORStatsPool(nn.Module):
    """Mean + log-sum-exp 'soft OR' + max over time.

    Replaces attentive statistics pooling. A softmax attention pool is free to collapse
    onto one preferred frame, which is the mechanism behind a positional prior: the
    network learns "trust frame 14" instead of "trust whichever frame looks like the
    keyword". A soft-OR asks the deployed question instead -- is there ANY frame carrying
    the keyword? -- and being a symmetric function of the frame scores, it cannot encode
    *where* that evidence was.

    mean keeps the global context (a keyword is also "a short loud event in a quiet room"),
    max keeps the strongest single-frame evidence, and the log-sum-exp term interpolates
    between them: one dominant frame and many moderate frames both raise it.
    """

    features: int = 48
    hidden: int = 24

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        mean = jnp.mean(x, axis=1)                                   # (B, C)
        mx = jnp.max(x, axis=1)                                      # (B, C)
        proj = nn.Dense(features=self.hidden)(x)                     # (B, T, H)
        w = self.param("or_w", nn.initializers.zeros, (self.hidden,))
        b = self.param("or_b", nn.initializers.zeros, ())
        s = jnp.einsum("bth,h->bt", proj, w) + b                    # (B, T)
        soft_or = jax.scipy.special.logsumexp(s, axis=1) - jnp.log(s.shape[1])
        return jnp.concatenate([mean, soft_or[:, None], mx], axis=-1)   # (B, 2C+1)


class BCConformerV3(nn.Module):
    """BC-Conformer V3 -- relative position bias, soft-OR pooling, no absolute temporal
    embedding. Same (49, 40, 1) input contract as every other architecture here, so all
    existing corpora and every existing checkpoint loader keep working.

    Deliberately a NEW name rather than an edit to BCConformerV2: best_v3_50k.flax and the
    other shipped checkpoints are restored by parameter name, so changing a shared block in
    place would break them.
    """

    num_classes: int = 2
    channels: int = 32
    conformer_dim: int = 48
    blocks: int = 3
    num_heads: int = 4
    dropout: float = 0.20

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        h = _delta_stack(x)                                          # (B, 49, 40, 3)

        h = nn.Conv(features=self.channels, kernel_size=(3, 3), strides=(1, 2),
                    padding="SAME", use_bias=False)(h)
        h = nn.GroupNorm(num_groups=4)(h)
        h = jax.nn.swish(h)                                          # (B, 49, 20, 32)

        f_conv = nn.Conv(features=self.channels, kernel_size=(1, 3), padding="SAME",
                         feature_group_count=self.channels, use_bias=False)(h)
        f_conv = nn.GroupNorm(num_groups=4)(f_conv)
        f_conv = jax.nn.swish(f_conv)
        f_bcast = jnp.mean(f_conv, axis=2, keepdims=True)
        h = h + f_conv + f_bcast

        h = nn.Conv(features=self.conformer_dim, kernel_size=(3, 3), strides=(1, 2),
                    padding="SAME", use_bias=False)(h)
        h = nn.GroupNorm(num_groups=4)(h)
        h = jax.nn.swish(h)                                          # (B, 49, 10, 48)

        mel_score = nn.Conv(features=1, kernel_size=(1,), use_bias=False)(h)
        mel_w = nn.softmax(mel_score, axis=2)
        seq = jnp.sum(h * mel_w, axis=2)                             # (B, 49, 48)

        # NOTE: no absolute positional embedding here. That omission is the point of V3.
        for _ in range(self.blocks):
            seq = RelConformerBlock(dim=self.conformer_dim,
                                    num_heads=self.num_heads)(seq, train=train)

        feat = SoftORStatsPool(features=self.conformer_dim)(seq)
        feat = nn.Dropout(rate=self.dropout, deterministic=not train)(feat)
        return nn.Dense(features=self.num_classes)(feat)


# ==============================================================================
# Factory Dispatcher & Utilities
# ==============================================================================
def get_model(arch: str = "bcresnet3", num_classes: int = 2) -> nn.Module:
    """Returns initialized Flax Module based on architecture name."""
    arch_lower = arch.lower().strip()
    if arch_lower in ["bcresnet1", "bcresnet"]:
        return BCResNet(num_classes=num_classes, scale=1)
    elif arch_lower in ["bcresnet3", "bcresnet_sota"]:
        return BCResNet(num_classes=num_classes, scale=3)
    elif arch_lower in ["tcresnet", "tc_resnet"]:
        return TCResNet(num_classes=num_classes, base_channels=24)
    elif arch_lower in ["dsresnet_se", "se_resnet"]:
        return DSResNetSE(num_classes=num_classes, base_channels=20)
    elif arch_lower in ["dscnn", "dscnn_s"]:
        return DSCNN(num_classes=num_classes)
    elif arch_lower in ["conformer", "conformer_lite", "mh_kws"]:
        return ConformerLiteKWS(num_classes=num_classes, dim=32)
    elif arch_lower in ["bcconformer_50k", "conformer_50k", "bc_conformer_50k", "50k"]:
        return BCConformer50K(num_classes=num_classes)
    elif arch_lower in ["bcconformer_100k", "conformer_100k", "bc_conformer_100k", "100k"]:
        return BCConformer100K(num_classes=num_classes)
    elif arch_lower in ["bcconformer_v2", "conformer_v2", "bc_conformer_v2", "v2"]:
        return BCConformerV2(num_classes=num_classes)
    elif arch_lower in ["bcconformer_v3", "conformer_v3", "bc_conformer_v3", "v3"]:
        return BCConformerV3(num_classes=num_classes)
    else:
        raise ValueError(f"Unknown architecture '{arch}'. Available: bcresnet1, bcresnet3, tcresnet, dsresnet_se, dscnn, conformer_lite, bcconformer_50k, bcconformer_100k, bcconformer_v2, bcconformer_v3")


def count_params(model: nn.Module, input_shape: Tuple[int, ...] = (1, 49, 40, 1)) -> int:
    """Computes exact number of trainable parameters for any model."""
    rng = jax.random.PRNGKey(0)
    dummy_input = jnp.ones(input_shape, dtype=jnp.float32)
    variables = model.init(rng, dummy_input, train=False)
    params = variables['params']
    return sum(x.size for x in jax.tree_util.tree_leaves(params))




## Speech source resolver

Resolves LibriSpeech dev-clean from OpenSLR and Speech Commands from its public archive, with safe extraction.

*Inlined from `build_speech_corpus.py`: 153 lines, 3 CLI lines and 0 cross-module imports removed — everything shares one namespace here.*


In [9]:
"""
Fetch the real-human-speech source corpus from public URLs
-----------------------------------------------------------
Everything else in this project already regenerates itself from public URLs on demand
(ESC-50 from GitHub, UrbanSound8K from HuggingFace). This closes the loop for speech
so a Kaggle notebook can build the whole corpus with no multi-gigabyte upload.

Sources
  LibriSpeech dev-clean  337 MB   ~5,270 utterances of clean read human speech.
                                Used for the `real` negative content type, for the
                                multi-talker babble mixer, and for the handful of real
                                human utterances containing "amaze".
  Speech Commands v0.02 2.3 GB   ~105k single-word recordings. Optional, off by
                                default: the TTS filler pool already covers the
                                "unimportant word" content type, and 2.3 GB is a lot
                                of notebook time for marginal extra variety.

Everything is written to speech_corpus/ in the layout build_corpus_v2.SpeechBank
expects (LibriSpeech/**.flac and speech_commands/**/*.wav).

    python build_speech_corpus.py                 # LibriSpeech dev-clean only
    python build_speech_corpus.py --speech_commands
"""

import argparse
import os
import shutil
import tarfile
import tempfile
import urllib.request

SPEECH_CORPUS = "speech_corpus"

LIBRISPEECH_SOURCES = {
    "dev-clean": "https://www.openslr.org/resources/12/dev-clean.tar.gz",
}
SPEECH_COMMANDS_URL = "https://www.tensorflow.org/data/speech_commands_v0.02.tar.gz"

UA = {"User-Agent": "Mozilla/5.0 (amaze-kws-corpus-builder)"}


def _download(url: str, dest: str, quiet: bool = False) -> str:
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        if not quiet:
            print(f"  cached: {os.path.basename(dest)}")
        return dest
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length") or 0)
        done = 0
        last = 0
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if not quiet and total and done - last > (32 << 20):
                last = done
                print(f"  {done/1e6:7.0f} / {total/1e6:.0f} MB", flush=True)
    return dest


def _safe_extract(archive: str, dest: str) -> None:
    """Extracts without letting archive members escape the destination directory."""
    dest_abs = os.path.abspath(dest)
    with tarfile.open(archive, "r:*") as tf:
        for member in tf.getmembers():
            target = os.path.abspath(os.path.join(dest, member.name))
            if not target.startswith(dest_abs + os.sep) and target != dest_abs:
                print(f"  skipping path traversal: {member.name}")
                continue
            tf.extract(member, dest, set_attrs=False)


def build_librispeech(out_dir: str = SPEECH_CORPUS, split: str = "dev-clean",
                     quiet: bool = False) -> int:
    url = LIBRISPEECH_SOURCES[split]
    target = os.path.join(out_dir, "LibriSpeech")
    if os.path.isdir(target):
        n = sum(len(f) for _, _, f in os.walk(target) if any(x.endswith(".flac") for x in f))
        if n:
            if not quiet:
                print(f"[librispeech] already present: {n:,} flac files under {target}")
            return n
    with tempfile.TemporaryDirectory() as td:
        tarball = _download(url, os.path.join(td, f"{split}.tar.gz"), quiet=quiet)
        if not quiet:
            print(f"[librispeech] extracting {split} -> {out_dir}")
        _safe_extract(tarball, out_dir)
    n = sum(1 for r, _, fs in os.walk(target) for f in fs if f.endswith(".flac"))
    if not quiet:
        print(f"[librispeech] {n:,} flac files")
    return n


def build_speech_commands(out_dir: str = SPEECH_CORPUS, quiet: bool = False) -> int:
    target = os.path.join(out_dir, "speech_commands")
    if os.path.isdir(target):
        n = sum(1 for r, _, fs in os.walk(target) for f in fs if f.endswith(".wav"))
        if n:
            if not quiet:
                print(f"[speech_commands] already present: {n:,} wav files")
            return n
    with tempfile.TemporaryDirectory() as td:
        tarball = _download(SPEECH_COMMANDS_URL, os.path.join(td, "sc.tar.gz"), quiet=quiet)
        if not quiet:
            print("[speech_commands] extracting (this is 2.3 GB, be patient)")
        _safe_extract(tarball, td)
        # v0.02 unpacks to a single directory of labelled subfolders plus validation/
        # and test/ splits; keep only the labelled training folders we can train on.
        root = None
        for entry in os.listdir(td):
            p = os.path.join(td, entry)
            if os.path.isdir(p) and any(
                os.path.isdir(os.path.join(p, sub)) for sub in os.listdir(p)
                if os.path.isdir(os.path.join(p, sub))
            ):
                root = p
                break
        if root is None:
            print("[speech_commands] unexpected archive layout; skipped")
            return 0
        os.makedirs(target, exist_ok=True)
        for sub in os.listdir(root):
            s = os.path.join(root, sub)
            if os.path.isdir(s) and sub not in ("testing", "validation", "_background_noise_"):
                shutil.copytree(s, os.path.join(target, sub), dirs_exist_ok=True)
    n = sum(1 for r, _, fs in os.walk(target) for f in fs if f.endswith(".wav"))
    if not quiet:
        print(f"[speech_commands] {n:,} wav files")
    return n


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", type=str, default=SPEECH_CORPUS)
    ap.add_argument("--split", type=str, default="dev-clean", choices=sorted(LIBRISPEECH_SOURCES))
    ap.add_argument("--speech_commands", action="store_true",
                    help="also fetch Speech Commands v0.02 (2.3 GB, optional)")
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    print(f"[speech corpus] target {args.out}")
    n_lib = build_librispeech(args.out, args.split)
    n_cmd = build_speech_commands(args.out) if args.speech_commands else 0
    total = n_lib + n_cmd
    print(f"[speech corpus] ready: {n_lib:,} LibriSpeech flac"
          + (f" + {n_cmd:,} Speech Commands wav" if n_cmd else ""))
    if not n_lib:
        raise SystemExit("no speech available; corpus cannot be built")


## Noise bank builder

ESC-50 + UrbanSound8K + procedural multi-talker babble + procedural device sounds, into a class-labelled index.csv.

*Inlined from `build_noise_bank_v2.py`: 675 lines, 3 CLI lines and 0 cross-module imports removed — everything shares one namespace here.*


In [10]:
"""
Noise Bank v2 Builder
---------------------
Expands the background-noise inventory so the model sees far more acoustic variety,
with explicit coverage of the classes it currently fails on (multi-talker babble,
crowded human chatter) plus 50x more real recordings per ESC-50 class.

Sources
  1. ESC-50 (full, 2000 clips / 50 classes) from the public GitHub mirror.
  2. UrbanSound8K (8732 clips / 18 classes incl. children_playing, crowd_hailing,
     street_music, police_car) from the HuggingFace parquet mirror.
  3. Procedural multi-talker babble built from the real human speech already in
     speech_corpus (mixes 2-14 independent talkers with per-talker gain, band
     limiting, overlap and light RIR) -- this is the class the v1 model has
     essentially never seen.
  4. Procedural device / human-event noises (HVAC hum, blender, printer, microwave
     beeps, cash register, TV static, rain on glass, ...) for appliance and office
     realism that ESC-50 does not cover.

Everything is resampled to 16 kHz mono float32 and written to noise_bank_v2/ with
an index CSV so the corpus builder can do class-stratified sampling.

    python build_noise_bank_v2.py --esc50 --urbansound --procedural
"""

import argparse
import csv
import io
import os
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import soundfile as sf
from scipy import signal

SAMPLE_RATE = 16000
OUT_DIR = "noise_bank_v2"
RAW_DIR = os.path.join(OUT_DIR, "raw")
INDEX_CSV = os.path.join(OUT_DIR, "index.csv")

USER_AGENT = {"User-Agent": "Mozilla/5.0"}


# -----------------------------------------------------------------------------
# helpers
# -----------------------------------------------------------------------------

def to_16k_mono(data: np.ndarray, sr: int) -> np.ndarray:
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != SAMPLE_RATE:
        data = signal.resample(data, int(round(len(data) * SAMPLE_RATE / sr)))
    if len(data) < SAMPLE_RATE // 2:
        data = np.pad(data, (0, SAMPLE_RATE // 2 - len(data)))
    peak = np.max(np.abs(data))
    if peak > 0:
        data = data / peak * 0.89
    return data.astype(np.float32)


def save_clip(clip: np.ndarray, path: str, sr: int = SAMPLE_RATE) -> bool:
    try:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        sf.write(path, clip.astype(np.float32), sr)
        return True
    except Exception:
        return False


def _fetch(url: str, timeout: int = 30) -> bytes:
    req = urllib.request.Request(url, headers=USER_AGENT)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


# -----------------------------------------------------------------------------
# 1. ESC-50 full set
# -----------------------------------------------------------------------------

ESC50_META_URL = "https://raw.githubusercontent.com/karoldvl/ESC-50/master/meta/esc50.csv"
ESC50_TREE_URL = "https://api.github.com/repos/karoldvl/ESC-50/git/trees/master?recursive=1"
ESC50_AUDIO_BASE = "https://raw.githubusercontent.com/karoldvl/ESC-50/master/"


def esc50_class_map() -> dict:
    import json
    text = _fetch(ESC50_META_URL).decode("utf-8")
    rows = list(csv.DictReader(io.StringIO(text)))
    return {r["filename"]: r["category"] for r in rows}


def build_esc50(max_files: int = 2000, workers: int = 16):
    print("\n[1/4] ESC-50 full set")
    try:
        import json
        tree = json.loads(_fetch(ESC50_TREE_URL, timeout=40).decode("utf-8"))["tree"]
        wavs = [t["path"] for t in tree
                if t["path"].startswith("audio/") and t["path"].endswith(".wav")]
        wavs.sort()
    except Exception as e:
        print(f"  could not list ESC-50: {e}")
        return []
    if max_files and len(wavs) > max_files:
        step = len(wavs) / max_files
        wavs = [wavs[int(i * step)] for i in range(max_files)]

    try:
        cmap = esc50_class_map()
    except Exception as e:
        print(f"  could not read ESC-50 metadata: {e}")
        cmap = {}

    written = []

    def grab(rel):
        fname = os.path.basename(rel)
        cls = cmap.get(fname, "esc50_unknown")
        dst = os.path.join(RAW_DIR, "esc50", f"{cls}__{fname}")
        if os.path.exists(dst):
            return (dst, cls)
        data = _fetch(ESC50_AUDIO_BASE + rel, timeout=40)
        arr, sr = sf.read(io.BytesIO(data))
        if save_clip(to_16k_mono(arr, sr), dst):
            return (dst, cls)
        return None

    done = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(grab, w): w for w in wavs}
        for fut in as_completed(futs):
            try:
                r = fut.result()
                if r:
                    written.append(r)
            except Exception:
                pass
            done += 1
            if done % 200 == 0:
                print(f"  ESC-50 {done}/{len(wavs)}", flush=True)
    print(f"  ESC-50 clips on disk: {len(written)}")
    return written


# -----------------------------------------------------------------------------
# 2. UrbanSound8K
# -----------------------------------------------------------------------------

US8K_REPO = "danavery/urbansound8K"


def build_urbansound(shard_limit: int = 4, per_class_cap: int = 220):
    """Streams a few UrbanSound8K parquet shards and unpacks the WAV payloads.

    The HF mirror stores clips as raw WAV bytes inside a struct column, so each
    shard is large; we only pull a handful of shards and cap clips per class.
    """
    print("\n[2/4] UrbanSound8K")
    try:
        import json
        api = json.loads(_fetch(f"https://huggingface.co/api/datasets/{US8K_REPO}", timeout=30).decode("utf-8"))
        parquets = sorted(s["rfilename"] for s in api.get("siblings", [])
                          if s["rfilename"].endswith(".parquet"))
        if not parquets:
            print("  no parquet shards listed")
            return []
    except Exception as e:
        print(f"  could not list UrbanSound8K: {e}")
        return []

    try:
        import pyarrow.parquet as pq
    except Exception as e:
        print(f"  pyarrow unavailable ({e}); skipping UrbanSound8K")
        return []

    written = []
    counts = {}
    for pq_name in parquets[:max(0, shard_limit)]:
        try:
            url = f"https://huggingface.co/datasets/{US8K_REPO}/resolve/main/{pq_name}"
            raw = _fetch(url, timeout=300)
            table = pq.read_table(io.BytesIO(raw))
        except Exception as e:
            print(f"  shard {pq_name} failed: {str(e)[:70]}")
            continue

        cols = table.column_names
        if "audio" not in cols or "class" not in cols:
            print(f"  shard {pq_name}: unexpected columns {cols}")
            continue

        audio_data = table.column("audio").to_pylist()
        labels = table.column("class").to_pylist()

        for rec, lbl in zip(audio_data, labels):
            lbl = str(lbl)
            if counts.get(lbl, 0) >= per_class_cap:
                continue
            try:
                if not isinstance(rec, dict) or not rec.get("bytes"):
                    continue
                arr, sr = sf.read(io.BytesIO(rec["bytes"]))
                counts[lbl] = counts.get(lbl, 0) + 1
                dst = os.path.join(RAW_DIR, "urbansound", f"us8k_{counts[lbl]:05d}_{lbl}.wav")
                if save_clip(to_16k_mono(arr, sr), dst):
                    written.append((dst, f"us8k_{lbl}"))
            except Exception:
                continue
        print(f"  shard {pq_name}: cumulative {len(written)} clips", flush=True)

    print(f"  UrbanSound8K clips on disk: {len(written)}")
    return written


# -----------------------------------------------------------------------------
# 3. Procedural multi-talker babble from real human speech
# -----------------------------------------------------------------------------

def _load_speech_pool(speech_dir: str = "speech_corpus", max_files: int = 3000) -> list:
    """Loads short real-human speech excerpts to use as babble talkers."""
    import glob
    pool = []
    flacs = []
    for root, _, files in os.walk(os.path.join(speech_dir, "LibriSpeech")):
        for f in files:
            if f.endswith(".flac"):
                flacs.append(os.path.join(root, f))
    np.random.shuffle(flacs)
    for p in flacs[:max_files]:
        try:
            d, sr = sf.read(p)
            d = to_16k_mono(d, sr)
            if len(d) >= SAMPLE_RATE:
                pool.append(d)
        except Exception:
            pass
    wavs = glob.glob(os.path.join(speech_dir, "speech_commands", "**", "*.wav"), recursive=True)
    np.random.shuffle(wavs)
    for p in wavs[:2000]:
        try:
            d, sr = sf.read(p)
            d = to_16k_mono(d, sr)
            if len(d) >= SAMPLE_RATE // 2:
                pool.append(d)
        except Exception:
            pass
    print(f"  babble talker pool: {len(pool)} excerpts")
    return pool


def _fit_len(a: np.ndarray, n: int, rng) -> np.ndarray:
    if len(a) >= n:
        st = rng.integers(0, len(a) - n + 1)
        return a[st:st + n]
    reps = int(np.ceil(n / max(1, len(a))))
    return np.tile(a, reps)[:n]


def make_babble(pool: list, rng, n_talkers: int, dur: float = 12.0) -> np.ndarray:
    n = int(dur * SAMPLE_RATE)
    mix = np.zeros(n, dtype=np.float32)
    for _ in range(n_talkers):
        voice = pool[rng.integers(0, len(pool))]
        seg = _fit_len(voice, n, rng).astype(np.float32)
        # per-talker gain spread so some voices dominate, others sit in the background
        g = float(10 ** (rng.uniform(-14.0, 3.0) / 20.0))
        # occasional lowpass (distant / across-the-room talkers)
        if rng.random() < 0.45:
            fc = float(rng.uniform(900.0, 3800.0))
            b, a = signal.butter(1, fc / (SAMPLE_RATE / 2.0), btype="low")
            seg = signal.lfilter(b, a, seg)
        # occasional highpass (close mic, plosives)
        if rng.random() < 0.3:
            b, a = signal.butter(1, float(rng.uniform(90.0, 260.0)) / (SAMPLE_RATE / 2.0), btype="high")
            seg = signal.lfilter(b, a, seg)
        # slow level drift (turn-taking)
        drift = 1.0 + 0.35 * np.sin(np.linspace(0, rng.uniform(1.0, 5.0), n) + rng.uniform(0, 6.28))
        mix += (seg * g * drift).astype(np.float32)

    mix = mix / (np.max(np.abs(mix)) + 1e-8)

    # small room reverb so the crowd sits in a space
    if rng.random() < 0.7:
        t60 = float(rng.uniform(0.25, 0.9))
        L = int(t60 * SAMPLE_RATE)
        ir = np.exp(-6.91 * np.arange(L) / SAMPLE_RATE / t60).astype(np.float32)
        ir *= rng.standard_normal(L).astype(np.float32) * 0.4
        ir[0] = 1.0
        ir = signal.lfilter(*signal.butter(1, 3800.0 / (SAMPLE_RATE / 2.0), btype="low"), ir)
        ir /= (np.sqrt(np.sum(ir ** 2)) + 1e-8)
        mix = signal.fftconvolve(mix, ir)[:n].astype(np.float32)

    # channel colouring
    if rng.random() < 0.5:
        b, a = signal.butter(2, float(rng.uniform(3000.0, 7500.0)) / (SAMPLE_RATE / 2.0), btype="low")
        mix = signal.lfilter(b, a, mix)
    if rng.random() < 0.4:
        b, a = signal.butter(1, float(rng.uniform(90.0, 200.0)) / (SAMPLE_RATE / 2.0), btype="high")
        mix = signal.lfilter(b, a, mix)
    if rng.random() < 0.35:  # light bitcrush (cheap mic / codec in a crowd)
        bits = int(rng.choice([6, 8]))
        q = 2 ** bits / 2.0 - 0.5
        mix = np.round((np.clip(mix, -1, 1) + 1.0) * q) / q - 1.0
    peak = np.max(np.abs(mix))
    return (mix / peak * float(rng.uniform(0.35, 0.9))).astype(np.float32) if peak > 0 else mix


def build_babble(per_config: int = 6):
    print("\n[3/4] Procedural multi-talker babble")
    pool = _load_speech_pool()
    if not pool:
        print("  no speech pool; skipping babble")
        return []
    rng = np.random.default_rng(1234)
    written = []
    configs = [(2, "two_talker"), (3, "small_group"), (4, "small_group"), (5, "crowd"),
               (6, "crowd"), (8, "crowd"), (10, "dense_crowd"), (12, "dense_crowd"), (14, "dense_crowd")]
    for n_t, name in configs:
        for i in range(per_config):
            clip = make_babble(pool, rng, n_t, dur=float(rng.uniform(8.0, 20.0)))
            dst = os.path.join(RAW_DIR, f"babble_{name}_{n_t}t_{i:03d}.wav")
            if save_clip(clip, dst):
                written.append((dst, f"babble_{name}"))
    print(f"  babble clips on disk: {len(written)}")
    return written


# -----------------------------------------------------------------------------
# 4. Procedural device / human-event noises
# -----------------------------------------------------------------------------

def _env(n: int, rng) -> np.ndarray:
    return rng.standard_normal(n).astype(np.float32)


def _tone(freq, n, sr=SAMPLE_RATE, phase=0.0) -> np.ndarray:
    t = np.arange(n) / sr
    return np.sin(2 * np.pi * freq * t + phase).astype(np.float32)


def _motor(n, rng, base_hz=48.0):
    """Combustion / compressor motor: harmonic stack + roughness + broadband."""
    sig = np.zeros(n, dtype=np.float32)
    h = 1
    while base_hz * h < 3000:
        amp = 1.0 / (h ** 1.15)
        sig += amp * _tone(base_hz * h * float(rng.uniform(0.985, 1.015)), n)
        h += 1
    sig /= np.max(np.abs(sig)) + 1e-8
    rough = _env(n, rng)
    b, a = signal.butter(2, float(rng.uniform(700.0, 2600.0)) / (SAMPLE_RATE / 2.0), btype="low")
    rough = signal.lfilter(b, a, rough)
    return (0.75 * sig + 0.55 * rough).astype(np.float32)


def gen_noise(name: str, rng, dur: float = 12.0) -> np.ndarray:
    n = int(dur * SAMPLE_RATE)
    r = rng

    if name == "ac_hum":
        base = float(r.choice([50.0, 60.0]))
        sig = 0.6 * _tone(base, n) + 0.4 * _tone(2 * base, n) + 0.2 * _tone(3 * base, n)
        hiss = signal.lfilter(*signal.butter(2, 3000.0 / 8000.0, btype="low"), _env(n, r))
        am = 1.0 + 0.06 * _tone(float(r.uniform(0.2, 1.5)), n)
        return (sig * am + 0.12 * hiss).astype(np.float32)

    if name == "fridge_compressor":
        sig = _motor(n, r, base_hz=float(r.uniform(48.0, 122.0)))
        b, a = signal.butter(2, 1400.0 / 8000.0, btype="low")
        return signal.lfilter(b, a, sig)

    if name == "transformer_hum":
        base = float(r.choice([50.0, 60.0]))
        sig = _tone(base, n) + 0.5 * _tone(2 * base, n) + 0.3 * _tone(4 * base, n) + 0.15 * _tone(6 * base, n)
        return (sig / 1.95).astype(np.float32)

    if name == "blender":
        sig = _motor(n, r, base_hz=float(r.uniform(90.0, 190.0)))
        jet = signal.lfilter(*signal.butter(2, float(r.uniform(1800.0, 5000.0)) / 8000.0, btype="low"), _env(n, r))
        am = 0.75 + 0.25 * np.abs(_tone(float(r.uniform(3.0, 9.0)), n))
        return (0.6 * sig + 0.7 * jet * am).astype(np.float32)

    if name == "vacuum_cleaner":
        sig = _motor(n, r, base_hz=float(r.uniform(70.0, 150.0)))
        hiss = signal.lfilter(*signal.butter(2, float(r.uniform(1200.0, 3600.0)) / 8000.0, btype="low"), _env(n, r))
        am = 0.8 + 0.2 * _tone(float(r.uniform(4.0, 11.0)), n)
        return (0.5 * sig + 0.9 * hiss * am).astype(np.float32)

    if name == "hair_dryer":
        sig = _motor(n, r, base_hz=float(r.uniform(110.0, 200.0)))
        hiss = signal.lfilter(*signal.butter(2, float(r.uniform(2000.0, 5500.0)) / 8000.0, btype="low"), _env(n, r))
        return (0.45 * sig + 1.0 * hiss).astype(np.float32)

    if name == "power_tool":
        sig = _motor(n, r, base_hz=float(r.uniform(140.0, 300.0)))
        chip = np.zeros(n, dtype=np.float32)
        period = max(8, int(SAMPLE_RATE / float(r.uniform(30.0, 90.0))))
        L = 60
        for k in range(0, n - L, period):
            chip[k:k + L] += r.standard_normal(L).astype(np.float32) * 2.5
        return (0.6 * sig + 0.35 * chip).astype(np.float32)

    if name == "printer":
        out = np.zeros(n, dtype=np.float32)
        t = 0.0
        while t < dur - 1.0:
            seg_len = min(int(r.uniform(0.8, 3.0) * SAMPLE_RATE), n)
            if seg_len < 100:
                break
            step = _motor(seg_len, r, base_hz=float(r.uniform(80.0, 180.0)))
            fader = np.linspace(1.0, 0.4, seg_len).astype(np.float32)
            st = int(t * SAMPLE_RATE)
            if st >= n:
                break
            end = min(st + seg_len, n)
            out[st:end] += step[:end - st] * fader[:end - st]
            for _ in range(int(r.integers(3, 12))):
                p = st + int(r.integers(0, max(1, end - st)))
                if p < n:
                    out[p] += float(r.uniform(0.2, 0.8))
            t += seg_len / SAMPLE_RATE + float(r.uniform(0.05, 0.4))
        return out

    if name == "keyboard_extra":
        out = np.zeros(n, dtype=np.float32)
        for _ in range(int(dur * float(r.uniform(6.0, 16.0)))):
            p = int(r.integers(0, n - 400))
            L = int(r.uniform(120, 320))
            burst = r.standard_normal(L).astype(np.float32)
            burst *= np.exp(-np.linspace(0, 1, L) * 6)
            b, a = signal.butter(2, float(r.uniform(1500.0, 5000.0)) / 8000.0, btype="low")
            out[p:p + L] += signal.lfilter(b, a, burst) * float(r.uniform(0.3, 1.0))
        return out

    if name == "plastic_crumple":
        out = np.zeros(n, dtype=np.float32)
        for _ in range(int(dur * float(r.uniform(2.0, 6.0)))):
            p = int(r.integers(0, max(1, n - 2000)))
            L = int(r.uniform(800, 2600))
            burst = r.standard_normal(L).astype(np.float32)
            burst *= np.exp(-np.linspace(0, 1, L) * 2.0)
            b, a = signal.butter(2, float(r.uniform(2500.0, 7000.0)) / 8000.0, btype="low")
            out[p:p + L] += signal.lfilter(b, a, burst) * float(r.uniform(0.4, 1.0))
        return out

    if name == "microwave_beep":
        out = np.zeros(n, dtype=np.float32)
        f = float(r.uniform(880.0, 2600.0))
        t = 0.0
        while t < dur:
            for _ in range(int(r.integers(1, 5))):
                L = int(0.12 * SAMPLE_RATE)
                gap = int(r.uniform(0.08, 0.5) * SAMPLE_RATE)
                st = int(t * SAMPLE_RATE)
                if st + L < n:
                    tone = _tone(f, L) * np.hanning(L).astype(np.float32)
                    out[st:st + L] += tone * float(r.uniform(0.5, 1.0))
            t += float(r.uniform(1.5, 5.0))
        return out

    if name == "phone_ring":
        out = np.zeros(n, dtype=np.float32)
        t = float(r.uniform(0.0, 1.0))
        f0 = float(r.uniform(400.0, 480.0))
        while t < dur:
            st = int(t * SAMPLE_RATE)
            for rep in range(2):
                s2 = st + rep * int(0.02 * SAMPLE_RATE)
                L = int(0.9 * SAMPLE_RATE)
                if s2 + L >= n:
                    break
                tone = (_tone(f0 * 2, L) + _tone(f0 * 3.2, L) + _tone(f0 * 4.1, L)) / 3.0
                out[s2:s2 + L] += tone.astype(np.float32) * float(r.uniform(0.6, 1.0))
            t += float(r.uniform(2.5, 6.0))
        return out

    if name == "notification_chime":
        out = np.zeros(n, dtype=np.float32)
        roots = [523.25, 587.33, 659.25, 783.99, 880.0, 987.77, 1046.5]
        t = 0.0
        while t < dur:
            st = int(t * SAMPLE_RATE)
            for k in range(int(r.integers(2, 5))):
                f = float(r.choice(roots))
                L = int(float(r.uniform(0.25, 0.9)) * SAMPLE_RATE)
                s2 = st + k * int(0.14 * SAMPLE_RATE)
                if s2 + L < n:
                    out[s2:s2 + L] += _tone(f, L) * np.exp(-np.linspace(0, 1, L) * 3).astype(np.float32) * 0.5
            t += float(r.uniform(2.0, 6.0))
        return out

    if name == "cash_register":
        out = np.zeros(n, dtype=np.float32)
        t = float(r.uniform(0.0, 2.0))
        while t < dur:
            st = int(t * SAMPLE_RATE)
            drawer = np.zeros(int(0.35 * SAMPLE_RATE), dtype=np.float32)
            drawer[:] = r.standard_normal(len(drawer)).astype(np.float32) * np.exp(-np.linspace(0, 1, len(drawer)) * 5)
            if st + len(drawer) < n:
                out[st:st + len(drawer)] += signal.lfilter(*signal.butter(2, 1200.0 / 8000.0, btype="low"), drawer) * 0.8
            for _ in range(int(r.integers(4, 14))):
                p = st + int(r.integers(0, int(0.4 * SAMPLE_RATE)))
                L = 200
                if p + L < n:
                    out[p:p + L] += r.standard_normal(L).astype(np.float32) * 0.4
            t += float(r.uniform(3.0, 9.0))
        return out

    if name == "tv_static":
        base = _env(n, r)
        out = base
        if r.random() < 0.5:  # AM radio: narrowband + carrier whistles
            fc = float(r.uniform(400.0, 1500.0))
            b, a = signal.butter(2, 1200.0 / 8000.0, btype="low")
            out = signal.lfilter(b, a, base) * 3.0
            for f in (fc, fc * 1.5, fc * 2.0):
                out = out + 0.12 * _tone(f, n) * (1.0 + 0.5 * _tone(float(r.uniform(0.3, 2.0)), n))
        voice = _tone(float(r.uniform(80.0, 250.0)), n)
        out = out * (0.5 + 0.5 * np.abs(voice))
        return out.astype(np.float32)

    if name == "rain_on_glass":
        base = _env(n, r)
        taps = np.zeros(n, dtype=np.float32)
        for _ in range(int(dur * float(r.uniform(20.0, 70.0)))):
            p = int(r.integers(0, max(1, n - 300)))
            L = int(r.uniform(60, 240))
            b = r.standard_normal(L).astype(np.float32) * np.exp(-np.linspace(0, 1, L) * 8)
            taps[p:p + L] += b
        hiss = signal.lfilter(*signal.butter(2, 4500.0 / 8000.0, btype="low"), base)
        return (0.75 * hiss + 0.6 * signal.lfilter(*signal.butter(2, 3000.0 / 8000.0, btype="low"), taps)).astype(np.float32)

    if name == "washing_machine":
        sig = _motor(n, r, base_hz=float(r.uniform(40.0, 90.0)))
        slosh = signal.lfilter(*signal.butter(2, 500.0 / 8000.0, btype="low"), _env(n, r))
        am = 0.7 + 0.3 * np.abs(_tone(float(r.uniform(0.6, 2.4)), n))
        return (0.6 * sig + 0.7 * slosh * am).astype(np.float32)

    if name == "sewing_machine":
        sig = _motor(n, r, base_hz=float(r.uniform(150.0, 320.0)))
        clack = np.zeros(n, dtype=np.float32)
        period = int(SAMPLE_RATE / float(r.uniform(5.0, 14.0)))
        for k in range(0, n, period):
            L = 60
            clack[k:k + L] += r.standard_normal(L).astype(np.float32) * 0.8
        return (0.6 * sig + 0.5 * clack).astype(np.float32)

    if name == "elevator_ding":
        out = np.zeros(n, dtype=np.float32)
        t = float(r.uniform(0.0, 1.0))
        while t < dur:
            st = int(t * SAMPLE_RATE)
            for f in (1567.98, 2093.0, 2637.0):
                L = int(float(r.uniform(0.5, 1.2)) * SAMPLE_RATE)
                if st + L < n:
                    out[st:st + L] += _tone(f, L) * np.exp(-np.linspace(0, 1, L) * 3.5).astype(np.float32) * 0.4
            t += float(r.uniform(4.0, 12.0))
        return out

    if name == "mic_self_noise":
        hiss = signal.lfilter(*signal.butter(2, float(r.uniform(2000.0, 6000.0)) / 8000.0, btype="low"), _env(n, r))
        agc = 1.0 / (1.0 + np.abs(hiss))
        return (0.25 * hiss * agc).astype(np.float32)

    # fallback
    return _env(n, r).astype(np.float32)


PROCEDURAL_NAMES = [
    "ac_hum", "fridge_compressor", "transformer_hum", "blender", "vacuum_cleaner",
    "hair_dryer", "power_tool", "printer", "keyboard_extra", "plastic_crumple",
    "microwave_beep", "phone_ring", "notification_chime", "cash_register",
    "tv_static", "rain_on_glass", "washing_machine", "sewing_machine",
    "elevator_ding", "mic_self_noise",
]


def build_procedural(per_name: int = 4):
    print("\n[4/4] Procedural device / event noises")
    written = []
    for name in PROCEDURAL_NAMES:
        for i in range(per_name):
            rng = np.random.default_rng(abs(hash((name, i))) % (2 ** 31))
            try:
                clip = gen_noise(name, rng, dur=float(rng.uniform(8.0, 18.0)))
                peak = np.max(np.abs(clip))
                if not np.isfinite(peak) or peak <= 0:
                    continue
                clip = clip / peak * float(rng.uniform(0.3, 0.85))
                dst = os.path.join(RAW_DIR, f"proc_{name}_{i:02d}.wav")
                if save_clip(clip.astype(np.float32), dst):
                    written.append((dst, f"proc_{name}"))
            except Exception as e:
                print(f"  {name}[{i}] failed: {str(e)[:60]}")
    print(f"  procedural clips on disk: {len(written)}")
    return written


# -----------------------------------------------------------------------------
# index
# -----------------------------------------------------------------------------

def _class_of(path: str) -> str:
    base = os.path.basename(path)
    if "__" in base:
        return base.split("__", 1)[0]
    stem = os.path.splitext(base)[0]
    parts = stem.split("_")
    if parts and parts[0] == "esc50":
        return "_".join(parts[1:-1]) or "esc50"
    if parts and parts[0] == "us8k":
        return "us8k_" + parts[-1]
    if parts and parts[0] == "proc":
        return "proc_" + "_".join(parts[1:-1])
    if parts and parts[0] == "babble":
        return "_".join(parts[:-2])
    return stem


def write_index(entries=None):
    """Indexes every clip already on disk (idempotent, safe to re-run)."""
    os.makedirs(OUT_DIR, exist_ok=True)
    rows = []
    for dirpath, _, files in os.walk(RAW_DIR):
        for f in sorted(files):
            if not f.endswith(".wav"):
                continue
            path = os.path.join(dirpath, f)
            try:
                info = sf.info(path)
                rows.append({"path": path.replace("\\", "/"), "class": _class_of(path),
                             "duration_s": round(info.duration, 3), "sr": info.samplerate})
            except Exception:
                continue
    with open(INDEX_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["path", "class", "duration_s", "sr"])
        w.writeheader()
        w.writerows(rows)
    classes = sorted({r["class"] for r in rows})
    print(f"\nIndex written: {INDEX_CSV}  ({len(rows)} clips, {len(classes)} classes)")
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--esc50", action="store_true")
    ap.add_argument("--urbansound", action="store_true")
    ap.add_argument("--procedural", action="store_true")
    ap.add_argument("--babble", action="store_true")
    ap.add_argument("--all", action="store_true")
    ap.add_argument("--esc50_max", type=int, default=2000)
    ap.add_argument("--babble_per_config", type=int, default=6)
    ap.add_argument("--procedural_per_name", type=int, default=4)
    ap.add_argument("--urbansound_shards", type=int, default=4)
    args = ap.parse_args()

    do_all = args.all or not any([args.esc50, args.urbansound, args.procedural, args.babble])
    entries = []
    if do_all or args.esc50:
        entries += build_esc50(max_files=args.esc50_max)
    if do_all or args.babble:
        entries += build_babble(per_config=args.babble_per_config)
    if do_all or args.procedural:
        entries += build_procedural(per_name=args.procedural_per_name)
    if args.urbansound or (do_all and args.urbansound_shards > 0):
        entries += build_urbansound(shard_limit=args.urbansound_shards)

    rows = write_index(entries)
    if rows:
        total_s = sum(r["duration_s"] for r in rows)
        print(f"Total noise material: {total_s/3600:.2f} hours across {len(rows)} clips")
    else:
        print("\n[Notice] Nothing new was written. Existing files stay listed only if rebuilt.")


## Corpus builder

24 mixture buckets over 4 recipes and 6 content types, identical for positives and negatives so layer count can never leak the label.

*Inlined from `build_corpus_v2.py`: 1345 lines, 3 CLI lines and 2 cross-module imports removed — everything shares one namespace here.*


In [11]:
"""
Industrial Corpus v2 for Wake Word 'Amaze'
-------------------------------------------
Builds a much larger, better-structured corpus than build_mega_corpus.py, driven by
three failure modes measured on the v1 model:

  1. TEMPORAL POSITION OVERFIT
     v1 centre-padded every positive, so the model learned "keyword sits at frame ~14".
     Rolling a val positive by 18 frames dropped TPR from 83% to 70%.
     -> v2 places the keyword at a random offset in the 1 s window and also emits
        explicit multi-offset copies of each voice base.

  2. NO MULTI-TALKER BABBLE NEGATIVES
     v1 scored 0.933 on pub_babble and 0.852 on restaurant_cafeteria -- both above the
     0.85 threshold -- because the noise bank had essentially no crowded-speech class.
     -> v2 mixes 2-8 real speech slices as the negative class, and mixes 2-3 noise
        layers into every positive.

  3. 88 NOISE CLASSES / 5 s EACH
     -> v2 uses noise_bank_v2 (full ESC-50 x40, 90 multi-talker babble tracks, ~160
        procedural device/event clips) with class-stratified sampling and random
        start offsets, so a 1 s window can sit anywhere inside a 10-20 s soundscape.

Positives  : 1.0 s windows, keyword at random offset, 2-3 noise layers at a
             SNR ladder that reaches -6 dB, full distortion chain, plus context
             carriers ("hey amaze", "amaze the lights") and intra-word jitter.
Negatives  : (a) clean/noisy real speech, (b) keyword-adjacent confusables,
             (c) pure + layered soundscapes, (d) MULTI-TALKER BABBLE,
             (e) speech-in-noise at low SNR (the hardest real deployment case),
             (f) keyword-adjacent fragments ("a maze", "amaz", "amuses").

Split discipline: the 80/20 split is stratified by (label, negative source kind) so every
negative category keeps its share in both splits. With ~28k negatives the val FPR
resolves to ~0.02% instead of the single-sample 0.12% v1 was reporting.

    python build_corpus_v2.py --pos 14000 --neg 28000 --out dataset_v2.npz
"""

import argparse
import asyncio
import csv
import glob
import io
import os
import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import soundfile as sf
from scipy import signal


SAMPLE_RATE = 16000
TOTAL_SAMPLES = 16000

KEYWORD = "amaze"

# Carriers that still contain the exact wake word (keeps the label valid)
POSITIVE_CARRIERS = [
    "amaze",
    "amaze.",
    "hey amaze",
    "ok amaze",
    "amaze play music",
    "amaze stop",
    "computer amaze",
    "amaze the lights",
    "amazing",
]
# 'amazing' is intentionally absent as a bare positive-only signal: the confusable
# miner below keeps inflected forms out of the positive set.

EDGE_VOICES = [
    "en-US-GuyNeural", "en-US-JennyNeural", "en-US-AriaNeural", "en-US-ChristopherNeural",
    "en-US-EricNeural", "en-US-MichelleNeural", "en-US-RogerNeural", "en-US-SteffanNeural",
    "en-US-EmmaNeural", "en-US-AndrewNeural", "en-US-BrianNeural", "en-US-AvaNeural",
    "en-GB-SoniaNeural", "en-GB-RyanNeural", "en-GB-LibbyNeural", "en-GB-ThomasNeural",
    "en-AU-NatashaNeural", "en-AU-WilliamNeural", "en-AU-KenNeural",
    "en-IN-NeerjaNeural", "en-IN-PrabhatNeural", "en-IN-NeerjaExpressiveNeural",
    "en-CA-ClaraNeural", "en-CA-LiamNeural",
    "en-IE-EmilyNeural", "en-IE-ConnorNeural",
    "en-NZ-MitchellNeural", "en-NZ-MollyNeural",
    "en-ZA-LeahNeural", "en-ZA-LukeNeural",
    "en-NG-AbeoNeural", "en-NG-EzinneNeural",
    "en-PH-RosaNeural", "en-PH-JamesNeural",
    "en-SG-LunaNeural", "en-SG-WayneNeural",
    "en-KE-AsiliaNeural", "en-KE-ChilembaNeural",
    "en-NG-ChidiNeural", "en-PH-DeniseNeural",
]

GTTS_TLDS = ["com", "co.uk", "ca", "com.au", "co.in", "co.za", "ie", "com.ng", "co.nz"]

# Keyword-adjacent negatives: these are the words that actually make KWS systems fire.
CONFUSABLE_WORDS = [
    "amazed", "amazement", "amazes", "amazing", "amusing", "amused", "amuses",
    "a maze", "a maiz", "amaz", "amaize", "amusse", "always", "amass", "amassed",
    "blaze", "blazes", "raise", "raises", "praise", "phase", "phrase", "chase",
    "erase", "gaze", "graze", "craze", "glaze", "maize", "amazingly",
    "haste", "paste", "taste", "waste", "vast", "mask", "masks", "ask",
    "am I late", "amazing grace", "may as well", "me as well", "aims",
    "he was amazed", "I'm amazed", "are you amazed",
]

NEGATIVE_COMMANDS = [
    "turn on the lights", "what is the weather", "set a timer for ten minutes",
    "play some jazz", "stop the music", "cancel that", "open the front door",
    "volume up", "increase the sound", "navigate home", "yes", "no", "hello",
    "good morning", "how are you today", "check my schedule", "send a message",
    "close the window", "turn off the heater", "system reboot", "start the timer",
    "what time is it", "call mom", "play the news", "shut down", "wake up",
    "lights off", "lights on", "pause", "resume", "next track", "louder", "quieter",
]


# ==============================================================================
# Acoustic helpers
# ==============================================================================

def apply_vtlp(audio: np.ndarray, alpha: float) -> np.ndarray:
    new_len = max(64, int(len(audio) / alpha))
    warped = signal.resample(audio, new_len)
    t_orig = np.linspace(0, 1, len(audio))
    t_warp = np.linspace(0, 1, len(warped))
    return np.interp(t_orig, t_warp, warped).astype(np.float32)


def apply_adc_bitcrush(audio: np.ndarray, bits: int = 8) -> np.ndarray:
    q = 2 ** bits / 2.0 - 0.5
    scaled = np.clip(audio, -1.0, 1.0)
    return (np.round((scaled + 1.0) * q) / q - 1.0).astype(np.float32)


def apply_nonlinear_cadence_warp(audio: np.ndarray, beta: float = None) -> np.ndarray:
    n = len(audio)
    if beta is None:
        beta = np.random.uniform(-0.25, 0.25)
    t = np.linspace(0, 1, n)
    tw = np.clip(t + beta * t * (1.0 - t), 0, 1)
    return np.interp(t, tw, audio).astype(np.float32)


def apply_telephonic_bandpass(audio: np.ndarray) -> np.ndarray:
    b, a = signal.butter(2, [300.0 / 8000.0, 3400.0 / 8000.0], btype="band")
    return signal.lfilter(b, a, audio).astype(np.float32)


def apply_burst_dropout(audio: np.ndarray) -> np.ndarray:
    burst = int(np.random.uniform(0.02, 0.045) * SAMPLE_RATE)
    if len(audio) > burst:
        st = np.random.randint(0, len(audio) - burst)
        audio = audio.copy()
        audio[st:st + burst] = 0.0
    return audio


def apply_dropout_bursts(audio: np.ndarray, max_bursts: int = 3) -> np.ndarray:
    out = audio
    for _ in range(int(np.random.randint(1, max_bursts + 1))):
        out = apply_burst_dropout(out)
    return out


def apply_awgn(speech: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    sp = np.mean(speech ** 2) + 1e-9
    npow = np.mean(noise ** 2) + 1e-9
    return (speech + noise * np.sqrt(sp / (10 ** (snr_db / 10.0) * npow))).astype(np.float32)


def peak_guard(audio: np.ndarray, ceiling: float = 0.95) -> np.ndarray:
    m = np.max(np.abs(audio))
    if m > ceiling:
        audio = audio / m * ceiling
    return audio


def fit_to_window(sig: np.ndarray, rng, offset: str = "random") -> np.ndarray:
    """Places `sig` inside a 1 s window at a random offset (or centred).

    Random offset is the fix for the v1 temporal-position overfit: the streaming
    engine slides the window every 100 ms, so the keyword lands at an arbitrary
    position in the frame.
    """
    sig = np.asarray(sig, dtype=np.float32).ravel()
    n = len(sig)
    if n >= TOTAL_SAMPLES:
        if offset == "random":
            st = rng.integers(0, n - TOTAL_SAMPLES + 1)
        else:
            st = (n - TOTAL_SAMPLES) // 2
        return sig[st:st + TOTAL_SAMPLES].copy()
    if offset == "random":
        left = int(rng.integers(0, TOTAL_SAMPLES - n + 1))
    else:
        left = (TOTAL_SAMPLES - n) // 2
    out = np.zeros(TOTAL_SAMPLES, dtype=np.float32)
    out[left:left + n] = sig
    return out


def trim_silence(audio: np.ndarray, thresh_db: float = -45.0, pad_ms: int = 25) -> np.ndarray:
    """Trims leading/trailing near-silence so short words are not mostly padding."""
    if len(audio) == 0:
        return audio
    win = int(0.005 * SAMPLE_RATE)
    n_win = max(1, len(audio) // win)
    frames = audio[:n_win * win].reshape(n_win, win)
    energy = 20 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)
    voiced = np.where(energy > thresh_db)[0]
    if len(voiced) == 0:
        return audio
    pad = int(pad_ms * SAMPLE_RATE / 1000)
    st = max(0, voiced[0] * win - pad)
    en = min(len(audio), (voiced[-1] + 1) * win + pad)
    return audio[st:en]


# ==============================================================================
# Noise / speech banks
# ==============================================================================

class NoiseBank:
    """Class-stratified sampler over noise_bank_v2/index.csv (or legacy noise_bank).

    The decoded-clip cache is bounded: a full 84-class bank is several GB of float32
    audio and each build worker holds its own copy, so only the most recently used
    clips stay resident.
    """

    MAX_CACHE = 256

    def __init__(self, index_csv: str = "noise_bank_v2/index.csv", legacy_dir: str = "noise_bank"):
        self.by_class = {}
        paths = []
        if os.path.exists(index_csv):
            with open(index_csv, newline="", encoding="utf-8") as f:
                for row in csv.DictReader(f):
                    paths.append((row["path"].replace("/", os.sep), row["class"]))
        if not paths:
            for p in sorted(glob.glob(os.path.join(legacy_dir, "*.wav"))):
                paths.append((p, os.path.splitext(os.path.basename(p))[0]))
        self.paths = paths
        for p, c in paths:
            self.by_class.setdefault(c, []).append(p)
        self.classes = sorted(self.by_class)
        self._cache = {}
        print(f"[NoiseBank] {len(paths)} clips across {len(self.classes)} classes")

    def load(self, path: str) -> np.ndarray:
        if path in self._cache:
            return self._cache[path]
        try:
            d, sr = sf.read(path)
            if d.ndim > 1:
                d = d[:, 0]
            if sr != SAMPLE_RATE:
                d = signal.resample(d, int(round(len(d) * SAMPLE_RATE / sr)))
            out = d.astype(np.float32)
        except Exception:
            out = np.zeros(SAMPLE_RATE, dtype=np.float32)
        if len(self._cache) >= self.MAX_CACHE:
            self._cache.pop(next(iter(self._cache)))
        self._cache[path] = out
        return out

    def sample_slice(self, rng, exclude=()) -> tuple:
        """Returns (slice, class). Class is drawn uniformly, position inside the clip
        is drawn uniformly, so a 1 s window can land anywhere in a 10-20 s soundscape."""
        pool = [c for c in self.classes if c not in exclude] or self.classes
        cls = str(rng.choice(pool))
        path = str(rng.choice(self.by_class[cls]))
        clip = self.load(path)
        if len(clip) >= TOTAL_SAMPLES:
            st = int(rng.integers(0, len(clip) - TOTAL_SAMPLES + 1))
            return clip[st:st + TOTAL_SAMPLES].copy(), cls
        reps = int(np.ceil(TOTAL_SAMPLES / max(1, len(clip))))
        padded = np.tile(clip, reps)[:TOTAL_SAMPLES]
        return padded, cls

    def babble_classes(self):
        return [c for c in self.classes if "babble" in c]


class SpeechBank:
    """Real human speech: LibriSpeech + Speech Commands, for negatives and babble."""

    MAX_CACHE = 192

    def __init__(self, corpus_dir: str = "speech_corpus", max_flacs: int = 6000, max_cmds: int = 6000):
        self.flacs = []
        for root, _, files in os.walk(os.path.join(corpus_dir, "LibriSpeech")):
            for f in files:
                if f.endswith(".flac"):
                    self.flacs.append(os.path.join(root, f))
        np.random.shuffle(self.flacs)
        self.flacs = self.flacs[:max_flacs]
        self.cmds = glob.glob(os.path.join(corpus_dir, "speech_commands", "**", "*.wav"), recursive=True)
        np.random.shuffle(self.cmds)
        self.cmds = self.cmds[:max_cmds]
        self._cache = {}
        print(f"[SpeechBank] {len(self.flacs)} LibriSpeech flacs, {len(self.cmds)} Speech Commands wavs")

    def load(self, path: str) -> np.ndarray:
        if path in self._cache:
            return self._cache[path]
        try:
            d, sr = sf.read(path)
            if d.ndim > 1:
                d = d[:, 0]
            if sr != SAMPLE_RATE:
                d = signal.resample(d, int(round(len(d) * SAMPLE_RATE / sr)))
            out = d.astype(np.float32)
        except Exception:
            out = np.zeros(SAMPLE_RATE, dtype=np.float32)
        if len(self._cache) >= self.MAX_CACHE:
            self._cache.pop(next(iter(self._cache)))
        self._cache[path] = out
        return out

    def sample_slice(self, rng, cmd_bias: float = 0.0) -> np.ndarray:
        use_cmd = self.cmds and (rng.random() < cmd_bias)
        pool = self.cmds if use_cmd else self.flacs
        if not pool:
            return np.zeros(TOTAL_SAMPLES, dtype=np.float32)
        path = str(rng.choice(pool))
        clip = self.load(path)
        if len(clip) >= TOTAL_SAMPLES:
            st = int(rng.integers(0, len(clip) - TOTAL_SAMPLES + 1))
            return clip[st:st + TOTAL_SAMPLES].copy()
        reps = int(np.ceil(TOTAL_SAMPLES / max(1, len(clip))))
        return np.tile(clip, reps)[:TOTAL_SAMPLES]


def make_babble_from_speech(speech_bank: SpeechBank, rng, n_talkers: int, dur: float = 4.0) -> np.ndarray:
    """Multi-talker crowd built from real speech -- the negative class v1 never had."""
    n = int(dur * SAMPLE_RATE)
    mix = np.zeros(n, dtype=np.float32)
    for _ in range(n_talkers):
        seg = speech_bank.sample_slice(rng, cmd_bias=0.15)
        if len(seg) < n:
            seg = np.tile(seg, int(np.ceil(n / len(seg))))
        seg = seg[:n]
        g = float(10 ** (rng.uniform(-13.0, 3.0) / 20.0))
        if rng.random() < 0.5:
            fc = float(rng.uniform(900.0, 4000.0))
            b, a = signal.butter(1, fc / 8000.0, btype="low")
            seg = signal.lfilter(b, a, seg)
        if rng.random() < 0.35:
            b, a = signal.butter(1, float(rng.uniform(90.0, 260.0)) / 8000.0, btype="high")
            seg = signal.lfilter(b, a, seg)
        mix += (seg * g).astype(np.float32)
    mix /= (np.max(np.abs(mix)) + 1e-8)
    if rng.random() < 0.6:
        mix = apply_reverberation(mix, t60_sec=float(rng.uniform(0.2, 0.8)))
    return peak_guard(mix).astype(np.float32)


# ==============================================================================
# Synthesis engines for keyword / confusable waveforms
# ==============================================================================

async def synth_edge(text: str, voice: str, rate: str, pitch: str) -> np.ndarray:
    import edge_tts
    comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
    out = io.BytesIO()

    async def drain():
        async for chunk in comm.stream():
            if chunk["type"] == "audio":
                out.write(chunk["data"])

    # edge-tts websockets occasionally stall; bound every request so a single
    # hung voice cannot stall the whole pool build.
    await asyncio.wait_for(drain(), timeout=25.0)
    out.seek(0)
    data, sr = sf.read(out)
    if data.ndim > 1:
        data = data[:, 0]
    if sr != SAMPLE_RATE:
        data = signal.resample(data, int(round(len(data) * SAMPLE_RATE / sr)))
    return data.astype(np.float32)


def synth_gtts(text: str, tld: str, slow: bool) -> np.ndarray:
    from gtts import gTTS
    tts = gTTS(text=text, lang="en", tld=tld, slow=slow)
    fp = io.BytesIO()
    tts.write_to_fp(fp)
    fp.seek(0)
    data, sr = sf.read(fp)
    if data.ndim > 1:
        data = data[:, 0]
    if sr != SAMPLE_RATE:
        data = signal.resample(data, int(round(len(data) * SAMPLE_RATE / sr)))
    return data.astype(np.float32)


def synth_sapi(text: str, rate: int, voice_idx: int) -> np.ndarray:
    import pyttsx3
    eng = pyttsx3.init()
    voices = eng.getProperty("voices")
    if voice_idx < len(voices):
        eng.setProperty("voice", voices[voice_idx].id)
    eng.setProperty("rate", rate)
    tmp = f"scratch_v2_{rate}_{voice_idx}.wav"
    eng.save_to_file(text, tmp)
    eng.runAndWait()
    data, sr = sf.read(tmp)
    if os.path.exists(tmp):
        os.remove(tmp)
    if data.ndim > 1:
        data = data[:, 0]
    if sr != SAMPLE_RATE:
        data = signal.resample(data, int(round(len(data) * SAMPLE_RATE / sr)))
    return data.astype(np.float32)


# ==============================================================================
# Positive pipeline
# ==============================================================================

def mix_layers(speech: np.ndarray, layers, snr_ladder) -> np.ndarray:
    out = speech
    for layer, snr in zip(layers, snr_ladder):
        if layer is None or len(layer) == 0:
            continue
        if len(layer) < len(out):
            reps = int(np.ceil(len(out) / len(layer)))
            layer = np.tile(layer, reps)
        layer = layer[:len(out)]
        if np.std(layer) < 1e-7:
            continue
        out = apply_awgn(out, layer, snr)
    return out


def distortion_chain(audio: np.ndarray, rng, aggressive: float) -> np.ndarray:
    """aggressive in [0,1] controls how far down the robustness ladder we go."""
    if rng.random() < 0.35 + 0.35 * aggressive:
        audio = apply_nonlinear_cadence_warp(audio)
    if rng.random() < 0.70:
        audio = apply_reverberation(audio)
    if rng.random() < 0.70:
        audio = apply_distance_attenuation(audio)
    if rng.random() < 0.15 * aggressive:
        audio = apply_telephonic_bandpass(audio)
    if rng.random() < 0.15 + 0.25 * aggressive:
        audio = apply_dropout_bursts(audio, max_bursts=1 + int(2 * aggressive))
    if rng.random() < 0.85:
        audio = apply_mems_mic_response(audio)
    if rng.random() < 0.25 + 0.30 * aggressive:
        audio = apply_adc_bitcrush(audio, bits=int(rng.choice([6, 8])))
    return peak_guard(audio)


# Deployment-realistic profile. v3 (the default) keeps a large near-field / clean
# mode because the v2 corpus was so uniformly destroyed that the model never saw a
# normal-level keyword and lost its high-confidence response entirely: the trained
# model's confidence ceiling was 0.86, which capped recall at any usable threshold.
PROFILES = {
    "v2": {
        "clean_frac": 0.0,
        "snr1": (2.0, 22.0), "snr2": (-2.0, 16.0), "snr3": (-6.0, 12.0),
        "telephonic": 0.15, "bitcrush": (0.25, 0.30), "dropout": (0.15, 0.25),
        "gain": (1.0, 1.0), "sin_snr": (-4.0, 14.0), "mems": 1.0,
    },
    "v3": {
        "clean_frac": 0.30,
        "snr1": (8.0, 24.0), "snr2": (0.0, 16.0), "snr3": (-3.0, 12.0),
        "telephonic": 0.06, "bitcrush": (0.12, 0.20), "dropout": (0.08, 0.18),
        "gain": (0.45, 1.60), "sin_snr": (0.0, 16.0), "mems": 1.0,
    },
    # v4/v5: buckets + a transducer chain matched to the INMP441.
    # `mems` goes to 0 because the INMP441 is flat from 60 Hz to 15 kHz, which
    # brackets the 100-7500 Hz feature band entirely: a generic MEMS darkening curve
    # would misrepresent the hardware. `bitcrush` drops to a rare probe because a
    # 24-bit I2S link quantizes ~59 dB below the capsule noise floor.
    "v4": {
        "clean_frac": 0.0,
        "snr1": (8.0, 24.0), "snr2": (0.0, 16.0), "snr3": (-3.0, 12.0),
        "telephonic": 0.06, "bitcrush": (0.02, 0.05), "dropout": (0.08, 0.18),
        "gain": (1.0, 1.0), "sin_snr": (0.0, 16.0), "mems": 0.0,
    },
}
_PROFILE = dict(PROFILES["v4"])


def _distortion_chain(audio: np.ndarray, rng, aggressive: float) -> np.ndarray:
    """Room / transmission / mic-stage effects only. The capsule response is NOT
    applied here -- it is handled by the INMP441 transducer chain after the level is
    set, so that ordering matches the physical signal path."""
    p = _PROFILE
    if rng.random() < 0.35 + 0.35 * aggressive:
        audio = apply_nonlinear_cadence_warp(audio)
    if rng.random() < 0.70:
        audio = apply_reverberation(audio)
    if rng.random() < 0.62:
        audio = apply_distance_attenuation(audio)
    if rng.random() < p["telephonic"] * aggressive:
        audio = apply_telephonic_bandpass(audio)
    if rng.random() < p["dropout"][0] + p["dropout"][1] * aggressive:
        audio = apply_dropout_bursts(audio, max_bursts=1 + int(2 * aggressive))
    if p.get("mems", 0.0) > 0 and rng.random() < p["mems"]:
        audio = apply_mems_mic_response(audio)
    if rng.random() < p["bitcrush"][0] + p["bitcrush"][1] * aggressive:
        audio = apply_adc_bitcrush(audio, bits=int(rng.choice([6, 8])))
    return peak_guard(audio)


def build_positive(base: np.ndarray, noise_bank: NoiseBank, rng) -> tuple:
    """base is a trimmed keyword waveform (< 1 s). Returns (1 s window, snr_mid, noise_classes)."""
    p = _PROFILE
    base = np.asarray(base, dtype=np.float32).ravel()
    audio = base
    # speed / VTLP: speaker- and rate-level variation
    if rng.random() < 0.85:
        audio = apply_speed_perturbation(audio, float(rng.uniform(0.80, 1.22)))
    if rng.random() < 0.70:
        audio = apply_vtlp(audio, float(rng.uniform(0.82, 1.22)))

    audio = fit_to_window(audio, rng, offset="random")

    # Recording-gain variation: the mic sits at a fixed gain, so the keyword level
    # tracks the distance the user happens to be at.
    if p["gain"][1] > p["gain"][0]:
        audio = (audio * float(rng.uniform(*p["gain"]))).astype(np.float32)

    classes, layers, snrs = [], [], []
    if rng.random() < p["clean_frac"]:
        # Near-field / clean mode: at most one quiet-ish layer, light distortion.
        if rng.random() < 0.55:
            nz, cls = noise_bank.sample_slice(rng)
            layers.append(nz)
            classes.append(cls)
            snrs.append(float(rng.uniform(*p["snr1"])))
    else:
        for i in range(int(rng.choice([1, 2, 2, 3, 3]))):
            nz, cls = noise_bank.sample_slice(rng)
            layers.append(nz)
            classes.append(cls)
            if i == 0:
                snrs.append(float(rng.uniform(*p["snr1"])))
            elif i == 1:
                snrs.append(float(rng.uniform(*p["snr2"])))
            else:
                snrs.append(float(rng.uniform(*p["snr3"])))
    audio = mix_layers(audio, layers, snrs)

    aggressive = float(rng.uniform(0.0, 1.0)) if len(layers) > 1 else float(rng.uniform(0.0, 0.35))
    audio = _distortion_chain(audio, rng, aggressive=aggressive)
    return audio, float(np.mean(snrs)) if snrs else 40.0, "|".join(classes[:3])


def build_positive_from_stream(audio: np.ndarray, noise_bank: NoiseBank, rng) -> tuple:
    """For long recordings (carrier phrases, human speech): place the whole utterance."""
    audio = fit_to_window(audio, rng, offset="random")
    classes, layers, snrs = [], [], []
    for i in range(int(rng.choice([1, 2, 2, 3]))):
        nz, cls = noise_bank.sample_slice(rng)
        layers.append(nz)
        classes.append(cls)
        snrs.append(float(rng.uniform(2.0, 22.0)) if i == 0 else float(rng.uniform(-4.0, 16.0)))
    audio = mix_layers(audio, layers, snrs)
    audio = distortion_chain(audio, rng, aggressive=float(rng.uniform(0.0, 1.0)))
    return audio, float(np.mean(snrs)), "|".join(classes[:3])


# ==============================================================================
# Negative pipeline
# ==============================================================================

def build_negative(kind: str, noise_bank: NoiseBank, speech_bank: SpeechBank,
                   rng, tts_neg_pool: list) -> tuple:
    if kind == "speech":
        a = speech_bank.sample_slice(rng, cmd_bias=0.35)
        if rng.random() < 0.5:
            nz, cls = noise_bank.sample_slice(rng)
            a = peak_guard(apply_awgn(a, nz, float(rng.uniform(0.0, 25.0))))
            return a, cls
        return a, "clean_speech"

    if kind == "babble":
        n_t = int(rng.integers(2, 9))
        a = make_babble_from_speech(speech_bank, rng, n_t, dur=float(rng.uniform(1.0, 4.0)))
        if rng.random() < 0.45:
            nz, cls = noise_bank.sample_slice(rng)
            a = peak_guard(apply_awgn(a, nz, float(rng.uniform(0.0, 20.0))))
            return a, f"babble|{cls}"
        return a, "babble"

    if kind == "noise":
        a, cls = noise_bank.sample_slice(rng)
        if rng.random() < 0.5:
            a2, cls2 = noise_bank.sample_slice(rng)
            a = peak_guard(0.6 * a + 0.4 * a2)
            cls = f"{cls}|{cls2}"
        a = a * float(rng.uniform(0.25, 1.0))
        if rng.random() < 0.25:
            a = apply_mems_mic_response(peak_guard(a))
        return a, cls

    if kind == "speech_in_noise":
        # Hardest deployment case: human speech buried in noise. The model must
        # stay silent because the wake word is not present.
        a = speech_bank.sample_slice(rng, cmd_bias=0.25)
        lo, hi = _PROFILE["sin_snr"]
        layers, classes, snrs = [], [], []
        for i in range(int(rng.choice([1, 2, 2, 3]))):
            nz, cls = noise_bank.sample_slice(rng)
            layers.append(nz)
            classes.append(cls)
            snrs.append(float(rng.uniform(lo, hi)) if i else float(rng.uniform(lo - 2, hi + 4)))
        a = mix_layers(a, layers, snrs)
        if rng.random() < 0.3:
            b = make_babble_from_speech(speech_bank, rng, int(rng.integers(2, 6)), dur=1.0)
            a = peak_guard(apply_awgn(a, b, float(rng.uniform(0.0, 12.0))))
            classes.append("babble")
        a = _distortion_chain(a, rng, aggressive=float(rng.uniform(0.0, 1.0)))
        return a, "|".join(classes[:3])

    if kind == "confusable":
        idx = int(rng.integers(0, len(tts_neg_pool))) if tts_neg_pool else 0
        base = tts_neg_pool[idx][1] if tts_neg_pool else np.zeros(4000, dtype=np.float32)
        a = fit_to_window(base, rng, offset="random")
        if rng.random() < 0.85:
            a = apply_speed_perturbation(a, float(rng.uniform(0.85, 1.18)))
        layers, classes, snrs = [], [], []
        for i in range(int(rng.choice([1, 2, 3]))):
            nz, cls = noise_bank.sample_slice(rng)
            layers.append(nz)
            classes.append(cls)
            snrs.append(float(rng.uniform(2.0, 22.0)) if i == 0 else float(rng.uniform(-4.0, 14.0)))
        a = mix_layers(a, layers, snrs)
        a = _distortion_chain(a, rng, aggressive=float(rng.uniform(0.0, 1.0)))
        return a, "|".join(classes[:3])

    raise ValueError(kind)


NEG_MIX = {
    "speech": 0.22,
    "babble": 0.16,
    "noise": 0.24,
    "speech_in_noise": 0.24,
    "confusable": 0.14,
}


# ==============================================================================
# v4/v5: explicit mixture taxonomy
# ==============================================================================
# The v3 corpus mixed "how much noise" and "what the content is" together, so the
# model could in principle use layer count as a class cue and the reported metrics
# hid which regime was actually failing. v4+ makes the two axes explicit and
# identical for positives and negatives:
#
#   recipe  M0_direct        content only, no noise at all
#           M1_one_noise     content + exactly ONE noise layer   <- lion share
#           M2_two_noise     content + exactly TWO noise layers
#           M3_two_noise_hard content + 2 layers, low SNR, full distortion
#
#   content pos  the keyword
#           neg  filler    ordinary English words with NO phonetic overlap
#           neg  real      real human speech slices (LibriSpeech / Speech Commands)
#           neg  babble    2-8 real talkers mixed live
#           neg  noise     soundscape only, no speech content
#           neg  confus    near-miss words (kept small on purpose)
#
# Every negative content type is instantiated in all four recipes, so no recipe is
# ever positive-only or negative-only. M1 dominates by design: a single interfering
# source is the most common real condition, and "one noise layer, no keyword" is the
# most common false-alarm condition.

UNIMPORTANT_WORDS = [
    # determiners / pronouns / function words -- nothing like "amaze"
    "the", "a", "that", "this", "these", "those", "there", "then", "than", "when",
    "where", "what", "which", "while", "would", "could", "should", "here", "their",
    "there", "about", "through", "before", "between", "without", "under", "after",
    # everyday verbs
    "open", "close", "carry", "follow", "listen", "measure", "picture", "remember",
    "consider", "discover", "explain", "finish", "gather", "happen", "imagine",
    "decide", "enjoy", "expect", "forget", "notice", "prepare", "receive", "search",
    "touch", "walk", "watch", "write", "jump", "climb", "brush", "count", "study",
    # everyday nouns
    "bottle", "candle", "doorway", "engine", "fabric", "garden", "hammer", "island",
    "jacket", "kettle", "ladder", "mirror", "napkin", "orange", "pencil", "quilt",
    "ribbon", "saddle", "table", "umbrella", "violet", "wagon", "anchor", "basket",
    "candle", "cupboard", "drawer", "fountain", "gravel", "helmet", "iron", "jug",
    # everyday adjectives
    "brave", "clever", "deep", "empty", "fancy", "gentle", "heavy", "just",
    "kind", "large", "mellow", "narrow", "odd", "polite", "quiet", "rapid",
    "shiny", "tidy", "vast", "warm", "young",
    # food / household / nature words
    "bread", "carrot", "cheese", "coffee", "salad", "butter", "cinnamon", "melon",
    "onion", "pepper", "soup", "dinner", "copper", "marble", "willow", "meadow",
    "thunder", "harvest", "blanket", "candle",
]

# bucket id -> (content, recipe, share_within_class, loss_weight)
# M1 ("one noise layer") is the lion share on the positive side (0.40 of 1.00) and the
# single largest negative bucket overall is N13_noise_one_noise (0.12), i.e. a lone
# soundscape with no speech in it -- the most common false-alarm condition in a room.
BUCKET_TABLE = {
    # positives -------------------------------------------------------------
    "P0_word_direct":         ("word",  "M0", 0.10, 1.00),
    "P1_word_one_noise":      ("word",  "M1", 0.40, 1.15),
    "P2_word_two_noise":      ("word",  "M2", 0.28, 1.25),
    "P3_word_two_noise_hard": ("word",  "M3", 0.22, 1.30),
    # negatives: filler words (unimportant, not confusable) ----------------
    "N0_filler_direct":         ("filler", "M0", 0.05, 0.80),
    "N1_filler_one_noise":      ("filler", "M1", 0.05, 0.90),
    "N2_filler_two_noise":      ("filler", "M2", 0.04, 1.00),
    "N3_filler_two_noise_hard": ("filler", "M3", 0.03, 1.10),
    # negatives: real human speech, no keyword ---------------------------
    "N4_real_direct":         ("real",  "M0", 0.05, 0.90),
    "N5_real_one_noise":      ("real",  "M1", 0.06, 1.00),
    "N6_real_two_noise":      ("real",  "M2", 0.04, 1.10),
    "N7_real_two_noise_hard": ("real",  "M3", 0.03, 1.20),
    # negatives: live multi-talker babble --------------------------------
    "N8_babble_direct":         ("babble", "M0", 0.04, 1.00),
    "N9_babble_one_noise":      ("babble", "M1", 0.06, 1.10),
    "N10_babble_two_noise":     ("babble", "M2", 0.05, 1.20),
    "N11_babble_two_noise_hard": ("babble", "M3", 0.06, 1.30),
    # negatives: soundscape only (no speech) ----------------------------
    "N12_noise_direct":         ("noise", "M0", 0.10, 0.90),
    "N13_noise_one_noise":      ("noise", "M1", 0.12, 1.00),
    "N14_noise_two_noise":      ("noise", "M2", 0.05, 1.00),
    "N15_noise_two_noise_hard": ("noise", "M3", 0.03, 1.10),
    # negatives: hard near-miss words ------------------------------------
    "N16_confus_direct":         ("confus", "M0", 0.03, 1.00),
    "N17_confus_one_noise":      ("confus", "M1", 0.05, 1.10),
    "N18_confus_two_noise":      ("confus", "M2", 0.04, 1.10),
    "N19_confus_two_noise_hard": ("confus", "M3", 0.02, 1.20),
}

POS_BUCKETS = [b for b in BUCKET_TABLE if b.startswith("P")]
NEG_BUCKETS = [b for b in BUCKET_TABLE if b.startswith("N")]
BUCKET_LOSS_WEIGHT = {b: BUCKET_TABLE[b][3] for b in BUCKET_TABLE}

# recipe -> layer count, per-layer SNR ranges, and the distortion-aggressiveness range
RECIPES = {
    "M0": {"layers": 0, "snr": None, "aggr": (0.00, 0.20)},
    "M1": {"layers": 1, "snr": ((12.0, 26.0),), "aggr": (0.05, 0.35)},
    "M2": {"layers": 2, "snr": ((8.0, 22.0), (0.0, 16.0)), "aggr": (0.20, 0.60)},
    "M3": {"layers": 2, "snr": ((2.0, 14.0), (-4.0, 10.0)), "aggr": (0.55, 1.00)},
}
RECIPE_LABEL = {"M0": "direct", "M1": "1 noise", "M2": "2 noise", "M3": "2 noise hard"}

# Explicit level policy, derived from the INMP441 datasheet rather than guessed.
#
# The v4 policy (-34..-14 dBFS) came from nowhere in particular and was ~8 dB too hot.
# With -26 dBFS sensitivity and 94 dB SPL full scale the SPL -> digital mapping is fixed
# (acoustic_augment.spl_to_dbfs), so the deployed level is set by (a) how far the user is
# standing and (b) whatever preamp/software gain the board applies. We treat the board
# gain as the free variable and let draw_talker_conditions() pick it, so a 3 m talker in
# a quiet voice lands genuinely lower than one at 30 cm.
#
#   distance   SPL (normal voice)  raw dBFS   @ +20 dB board gain
#   0.3 m      70.0 dB             -50.0      -30.0
#   0.5 m      66.0 dB             -54.0      -34.0
#   1.0 m      60.0 dB             -60.0      -40.0
#   2.0 m      54.0 dB             -66.0      -46.0
#   3.0 m      50.5 dB             -69.5      -49.5
#   shouting 82.0 dB              -38.0      -18.0
#   whisper   45.0 dB              -75.0      -55.0
#
# The INMP441 has no AGC, so nothing compresses this range: it is what the front end sees.
LEVEL_DBFS = (-50.0, -14.0)
# Below this the log-mel is pinned at its epsilon floor and the sample carries no
# information, so it is a label-noise sample rather than a hard one.
LEVEL_FLOOR_DBFS = -70.0
CLIP_PROB = 0.30
CLIP_CEILING = (0.35, 1.00)

# 24-bit I2S means real quantization is negligible, so aggressive bit-crushing is not
# physically justified for this hardware. Kept only as a rare robustness probe.
BITCRUSH_PROB_V4 = (0.02, 0.05)


def apply_level_policy(audio: np.ndarray, rng, level_dbfs: float = None) -> tuple:
    """Normalises loudness to a designed range, optionally through preamp overload.

    `level_dbfs` overrides the random draw (used by the transducer chain, which already
    picked a level from the talker's distance and loudness).
    """
    a = np.asarray(audio, dtype=np.float32).ravel()
    if rng.random() < CLIP_PROB:
        # Preamp overload: the gain stage clips before the level is settled.
        a = np.clip(a, -float(rng.uniform(*CLIP_CEILING)), float(rng.uniform(*CLIP_CEILING)))
    rms = float(np.sqrt(np.mean(a ** 2)))
    if rms < 1e-8:
        return peak_guard(a), -60.0
    target_db = float(rng.uniform(*LEVEL_DBFS) if level_dbfs is None else level_dbfs)
    a = a * (10 ** (target_db / 20.0) / rms)
    peak = float(np.max(np.abs(a)))
    if peak > 1.0:
        a = np.clip(a * (0.99 / peak), -1.0, 1.0)
    achieved = 20.0 * float(np.log10(np.sqrt(np.mean(a ** 2)) + 1e-9))
    return a.astype(np.float32), achieved


def apply_transducer_chain(audio: np.ndarray, rng, snr_db: float = 61.0) -> np.ndarray:
    """Final sensor model: INMP441 response + capsule self-noise + I2S quantization.

    Order mirrors the physical signal path: the capsule response shapes the incoming
    sound, then the capsule's own electronic noise is added, then the I2S word is
    quantized. `snr_db` comes from the drawn talker distance and vocal effort, so a
    user 3 m away in a quiet voice genuinely sees a worse SNR than one at 30 cm.
    """
    audio = apply_inmp441_response(audio)
    audio = add_mic_self_noise(audio, rng, snr_db=snr_db)
    if rng.random() < 0.12:
        # Deployment may read the 24-bit word through a 16-bit I2S peripheral.
        audio = apply_i2s_16bit_truncation(audio)
    return np.clip(audio, -1.0, 1.0).astype(np.float32)


def _sample_layers(noise_bank: NoiseBank, rng, snr_specs):
    layers, classes, snrs = [], [], []
    for spec in snr_specs:
        nz, cls = noise_bank.sample_slice(rng)
        layers.append(nz)
        classes.append(cls)
        snrs.append(float(rng.uniform(*spec)))
    return layers, classes, snrs


def build_bucket_sample(bucket: str, noise_bank: NoiseBank, speech_bank: SpeechBank,
                        rng, tts_bases: list, tts_filler: list, tts_conf: list) -> tuple:
    """Builds one sample for a named bucket. Returns (audio, snr, noise_classes, level_dbfs)."""
    content, recipe = BUCKET_TABLE[bucket][0], BUCKET_TABLE[bucket][1]
    rec = RECIPES[recipe]

    if content == "word":
        base = tts_bases[int(rng.integers(0, len(tts_bases)))][1]
        base = np.asarray(base, dtype=np.float32).ravel()
        if rng.random() < 0.85:
            base = apply_speed_perturbation(base, float(rng.uniform(0.80, 1.22)))
        if rng.random() < 0.70:
            base = apply_vtlp(base, float(rng.uniform(0.82, 1.22)))
        content_sig = fit_to_window(base, rng, offset="random")
    elif content == "filler":
        pool = tts_filler or tts_conf
        base = np.asarray(pool[int(rng.integers(0, len(pool)))][1], dtype=np.float32).ravel()
        if rng.random() < 0.8:
            base = apply_speed_perturbation(base, float(rng.uniform(0.85, 1.20)))
        content_sig = fit_to_window(base, rng, offset="random")
    elif content == "confus":
        pool = tts_conf or tts_filler
        base = np.asarray(pool[int(rng.integers(0, len(pool)))][1], dtype=np.float32).ravel()
        if rng.random() < 0.8:
            base = apply_speed_perturbation(base, float(rng.uniform(0.85, 1.18)))
        content_sig = fit_to_window(base, rng, offset="random")
    elif content == "real":
        content_sig = speech_bank.sample_slice(rng, cmd_bias=0.30)
    elif content == "babble":
        content_sig = make_babble_from_speech(
            speech_bank, rng, int(rng.integers(2, 9)), dur=float(rng.uniform(1.0, 4.0)))
    elif content == "noise":
        # The "content" for a noise-only negative is itself a soundscape, so M0 is a
        # real single soundscape rather than digital silence.
        content_sig, _cls = noise_bank.sample_slice(rng)
    else:
        raise ValueError(content)

    if rec["layers"] == 0:
        audio = content_sig
        classes, snrs = ["direct"], []
    else:
        layers, classes, snrs = _sample_layers(noise_bank, rng, rec["snr"])
        audio = mix_layers(content_sig, layers, snrs)

    lo, hi = rec["aggr"]
    audio = _distortion_chain(audio, rng, aggressive=float(rng.uniform(lo, hi)))

    # Talker geometry first: it sets both where the board's gain puts the speech and
    # the SNR the INMP441 actually delivers, which is what the 61 dBA figure means in
    # a real room.
    level_dbfs, snr_db, _spl = draw_talker_conditions(rng)
    audio, _ = apply_level_policy(audio, rng, level_dbfs=level_dbfs)
    audio = apply_transducer_chain(audio, rng, snr_db=snr_db)

    # Level is measured after the transducer chain so the recorded dBFS is the number
    # the deployed front end will actually see. The 60 Hz capsule high-pass can strip a
    # low-frequency-heavy soundscape by tens of dB, which would pin the log-mel at its
    # epsilon floor and make the sample unlearnable, so anything that falls through the
    # floor is lifted back to target.
    lvl = 20.0 * float(np.log10(np.sqrt(np.mean(audio ** 2)) + 1e-9))
    if lvl < LEVEL_FLOOR_DBFS:
        gain = 10 ** ((level_dbfs - lvl) / 20.0)
        audio = np.clip(audio * gain, -1.0, 1.0).astype(np.float32)
        lvl = 20.0 * float(np.log10(np.sqrt(np.mean(audio ** 2)) + 1e-9))
    return audio, float(np.mean(snrs)) if snrs else 40.0, "|".join(classes[:3]), lvl


def _bucket_sampler(buckets, rng):
    keys = list(buckets)
    w = np.array([buckets[k][2] for k in keys], dtype=np.float64)
    w /= w.sum()
    probs = w.cumsum()

    def draw():
        r = float(rng.random())
        return keys[int(np.searchsorted(probs, r, side="right"))]

    return draw



# ==============================================================================
# Worker
# ==============================================================================

_W = {}


def _init_worker(noise_paths, speech_flacs, speech_cmds, tts_bases, tts_confs, seed,
                 profile=None, tts_filler=None):
    global _PROFILE
    if profile:
        _PROFILE.clear()
        _PROFILE.update(PROFILES[profile])
    _W["noise_bank"] = NoiseBank.__new__(NoiseBank)
    _W["noise_bank"].by_class = {}
    for p, c in noise_paths:
        _W["noise_bank"].by_class.setdefault(c, []).append(p)
    _W["noise_bank"].paths = noise_paths
    _W["noise_bank"].classes = sorted(_W["noise_bank"].by_class)
    _W["noise_bank"]._cache = {}

    _W["speech_bank"] = SpeechBank.__new__(SpeechBank)
    _W["speech_bank"].flacs = speech_flacs
    _W["speech_bank"].cmds = speech_cmds
    _W["speech_bank"]._cache = {}

    _W["tts_bases"] = tts_bases
    _W["tts_confs"] = tts_confs
    _W["tts_filler"] = tts_filler or []
    _W["seed"] = seed


def _run_chunk(args):
    start, n_pos, n_neg_total, seed, use_buckets = args
    rng = np.random.default_rng(seed)
    extractor = AudioFeatureExtractor()
    nb = _W["noise_bank"]
    sb = _W["speech_bank"]
    tb = _W["tts_bases"]
    tc = _W["tts_confs"]
    tf = _W.get("tts_filler", [])

    Xs, ys, groups, snrs, buckets, lvls = [], [], [], [], [], []

    def emit(bucket, is_pos):
        audio, snr, grp, lvl = build_bucket_sample(bucket, nb, sb, rng, tb, tf, tc)
        Xs.append(extractor.compute_spectrogram(audio))
        ys.append(1 if is_pos else 0)
        groups.append(("pos|" if is_pos else "neg|") + grp)
        snrs.append(snr)
        buckets.append(bucket)
        lvls.append(lvl)

    if use_buckets:
        draw_pos = _bucket_sampler({b: BUCKET_TABLE[b] for b in POS_BUCKETS}, rng)
        draw_neg = _bucket_sampler({b: BUCKET_TABLE[b] for b in NEG_BUCKETS}, rng)
        for _ in range(n_pos):
            emit(draw_pos(), True)
        for _ in range(n_neg_total):
            emit(draw_neg(), False)
    else:
        for i in range(n_pos):
            base = tb[(start + i) % len(tb)][1]
            audio, snr, grp = build_positive(base, nb, rng)
            Xs.append(extractor.compute_spectrogram(audio))
            ys.append(1)
            groups.append("pos|" + grp)
            snrs.append(snr)
            buckets.append("P_legacy")
            lvls.append(20.0 * float(np.log10(np.sqrt(np.mean(audio ** 2)) + 1e-9)))

        kinds = list(NEG_MIX)
        weights = np.array([NEG_MIX[k] for k in kinds], dtype=np.float64)
        weights /= weights.sum()
        for _ in range(n_neg_total):
            kind = str(rng.choice(kinds, p=weights))
            try:
                audio, grp = build_negative(kind, nb, sb, rng, tc)
            except Exception:
                continue
            Xs.append(extractor.compute_spectrogram(audio))
            ys.append(0)
            groups.append(f"neg|{kind}|{grp}")
            snrs.append(0.0)
            buckets.append("N_legacy")
            lvls.append(20.0 * float(np.log10(np.sqrt(np.mean(audio ** 2)) + 1e-9)))

    return (np.array(Xs, dtype=np.float32),
            np.array(ys, dtype=np.int32),
            np.array(groups),
            np.array(snrs, dtype=np.float32),
            np.array(buckets),
            np.array(lvls, dtype=np.float32))


# ==============================================================================
# Base waveform synthesis (run once, shared by workers)
# ==============================================================================

def _pool_cache_path(cache):
    # Pickle, not npz. The payload is a dict of ragged float32 arrays, so it needs an
    # object container anyway, and Kaggle auto-extracts .zip resources from a dataset
    # (a .npz is a zip) which silently ships one member instead of the whole 356 MB
    # file. A .pkl is passed through untouched and pickle.load reads it back fine.
    return os.path.join(cache, "tts_pools.pkl") if cache else None


def save_pool_cache(pools, cache_dir):
    path = _pool_cache_path(cache_dir)
    if not path:
        return
    try:
        import pickle
        os.makedirs(cache_dir, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump({k: [(t, np.asarray(w, dtype=np.float32)) for t, w in v]
                         for k, v in pools.items()}, f, protocol=4)
        print(f"  TTS pool cache written: {path} "
              f"({os.path.getsize(path)/1e6:.1f} MB)")
    except Exception as e:
        print(f"  could not write pool cache: {e}")


def load_pool_cache(cache_dir, want_filler=True):
    path = _pool_cache_path(cache_dir)
    if not path or not os.path.exists(path):
        return None
    try:
        import pickle
        with open(path, "rb") as f:
            raw = pickle.load(f)
        pools = {k: [(str(t), np.asarray(w, dtype=np.float32)) for t, w in v]
                 for k, v in raw.items()}
        if not pools.get("kw") or (want_filler and not pools.get("filler")):
            return None
        print(f"  TTS pool cache loaded: {path} "
              f"(kw={len(pools['kw'])}, conf={len(pools.get('conf', []))}, "
              f"filler={len(pools.get('filler', []))})")
        return pools
    except Exception as e:
        print(f"  could not read pool cache: {e}")
        return None


async def build_tts_pools(per_voice: int = 3, want_filler: bool = True):
    print("\n[Synthesis] Building keyword, filler and confusable waveform pools...")
    sem = asyncio.Semaphore(12)
    results = {"kw": [], "conf": [], "filler": []}

    async def one(text, voice, rate, pitch, bucket):
        async with sem:
            for attempt in range(2):
                try:
                    w = await synth_edge(text, voice, rate, pitch)
                except Exception:
                    w = None
                if w is not None and len(w) > 800:
                    results[bucket].append((f"{voice}|{rate}|{pitch}|{text}", w))
                    return 1
                await asyncio.sleep(0.4 * (attempt + 1))
            return 0

    tasks = []
    rates = ["-25%", "-18%", "-12%", "-6%", "+0%", "+6%", "+12%", "+18%", "+25%"]
    pitches = ["-25Hz", "-15Hz", "-8Hz", "+0Hz", "+8Hz", "+15Hz", "+25Hz"]
    for v in EDGE_VOICES:
        combos = [(r, p) for r in rates for p in pitches]
        np.random.shuffle(combos)
        for k in range(per_voice * 3):
            rate, pitch = combos[k % len(combos)]
            text = POSITIVE_CARRIERS[(k + abs(hash(v)) % 7) % len(POSITIVE_CARRIERS)]
            tasks.append(one(text, v, rate, pitch, "kw"))
    await asyncio.gather(*tasks)

    tasks = []
    for w in CONFUSABLE_WORDS:
        for v in EDGE_VOICES[::4]:
            for r, p in (("-10%", "+5Hz"), ("+0%", "+0Hz"), ("+14%", "-6Hz")):
                tasks.append(one(w, v, r, p, "conf"))
    for w in NEGATIVE_COMMANDS:
        for v in EDGE_VOICES[::6]:
            tasks.append(one(w, v, "+0%", "+0Hz", "conf"))
    await asyncio.gather(*tasks)

    if want_filler:
        # Ordinary words with no phonetic overlap with the keyword. These are the
        # "unimportant word" negatives the user asked for in every mixture recipe:
        # without them the model only ever learns to reject speech that is either a
        # near-miss of the keyword or real human audio.
        tasks = []
        seen = set()
        words = [w for w in UNIMPORTANT_WORDS if not (w in seen or seen.add(w))]
        for w in words:
            for v in EDGE_VOICES[::5]:
                r, p = (("-8%", "+0Hz"), ("+0%", "+6Hz"), ("+12%", "-6Hz"))[len(tasks) % 3]
                tasks.append(one(w, v, r, p, "filler"))
        await asyncio.gather(*tasks)

    for tld in GTTS_TLDS:
        for slow in (False, True):
            try:
                w = trim_silence(synth_gtts(KEYWORD, tld, slow))
                results["kw"].append((f"gtts|{tld}|{slow}", w))
            except Exception:
                pass
            try:
                w = trim_silence(synth_gtts(rng_word_pick(), tld, slow))
                results["conf"].append((f"gtts|{tld}|{slow}", w))
            except Exception:
                pass

    try:
        eng_voices = 4
        for r in (110, 140, 170, 200, 230):
            for vi in range(eng_voices):
                try:
                    w = trim_silence(synth_sapi(KEYWORD, r, vi))
                    results["kw"].append((f"sapi|{r}|{vi}", w))
                except Exception:
                    pass
        for w in CONFUSABLE_WORDS[::4]:
            for vi in (0, 1):
                try:
                    results["conf"].append((f"sapi|{w}|{vi}", trim_silence(synth_sapi(w, 175, vi))))
                except Exception:
                    pass
    except Exception as e:
        print(f"  SAPI unavailable: {e}")

    # human "amaze" from LibriSpeech
    human = slice_human_keyword()
    for tag, w in human:
        results["kw"].append((f"human|{tag}", w))

    print(f"  keyword bases: {len(results['kw'])}  confusable/command bases: {len(results['conf'])}")
    return results


_rng_state = np.random.default_rng(7)


def rng_word_pick() -> str:
    return str(_rng_state.choice(CONFUSABLE_WORDS))


def slice_human_keyword(corpus_dir: str = "speech_corpus/LibriSpeech", max_files: int = 400) -> list:
    """Extracts 1 s windows centred on real human utterances containing 'amaze*'."""
    out = []
    txt_files = glob.glob(os.path.join(corpus_dir, "**", "*.trans.txt"), recursive=True)
    targets = []
    for tf in txt_files:
        try:
            with open(tf, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 2:
                        continue
                    if any("AMAZ" in w for w in parts[1:]):
                        flac = os.path.join(os.path.dirname(tf), f"{parts[0]}.flac")
                        if os.path.exists(flac):
                            targets.append(flac)
        except Exception:
            pass
    np.random.shuffle(targets)
    for p in targets[:max_files]:
        try:
            d, sr = sf.read(p)
            if d.ndim > 1:
                d = d[:, 0]
            if sr != SAMPLE_RATE:
                d = signal.resample(d, int(round(len(d) * SAMPLE_RATE / sr)))
            if len(d) < TOTAL_SAMPLES:
                continue
            st = int(np.random.randint(0, len(d) - TOTAL_SAMPLES + 1))
            out.append((os.path.basename(p), d[st:st + TOTAL_SAMPLES].copy()))
        except Exception:
            pass
    print(f"  human keyword windows: {len(out)}")
    return out


# ==============================================================================
# Main
# ==============================================================================

def stratified_split(groups, y, val_frac=0.2, seed=7, buckets=None):
    """Stratified by (label, bucket) so every mixture regime keeps its share in both
    splits -- otherwise the rare hard buckets vanish from val and stop being measured."""
    rng = np.random.default_rng(seed)
    from collections import defaultdict
    buckets_map = defaultdict(list)
    for i, g in enumerate(groups):
        if buckets is not None:
            kind = str(buckets[i])
        else:
            kind = "|".join(g.split("|")[:2]) if not g.startswith("pos|") else "pos"
        buckets_map[(int(y[i]), kind)].append(i)
    val_idx = []
    for key, idxs in buckets_map.items():
        idxs = np.array(idxs)
        rng.shuffle(idxs)
        k = max(1, int(round(len(idxs) * val_frac)))
        val_idx.extend(idxs[:k].tolist())
    val_idx = set(val_idx)
    train_idx = [i for i in range(len(y)) if i not in val_idx]
    return np.array(sorted(train_idx)), np.array(sorted(val_idx))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pos", type=int, default=14000)
    ap.add_argument("--neg", type=int, default=28000)
    ap.add_argument("--out", type=str, default="dataset_v2.npz")
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--chunks", type=int, default=0, help="0 = workers * 4")
    ap.add_argument("--per_voice", type=int, default=3)
    ap.add_argument("--noise_index", type=str, default="noise_bank_v2/index.csv")
    ap.add_argument("--dump_bases", type=str, default="keyword_bases.npy",
                    help="path for the clean keyword waveform cache ('' to skip)")
    ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--profile", type=str, default="v4", choices=sorted(PROFILES))
    ap.add_argument("--pool_cache", type=str, default="cache",
                    help="directory for the TTS pool cache ('' to disable)")
    args = ap.parse_args()

    _PROFILE.clear()
    _PROFILE.update(PROFILES[args.profile])
    use_buckets = args.profile in ("v4", "v5")
    print(f"[Profile] {args.profile}: buckets={use_buckets}  "
          f"bitcrush={_PROFILE['bitcrush']}  level={LEVEL_DBFS} dBFS")

    t0 = time.time()
    n_chunks = args.chunks or max(1, args.workers * 4)

    pools = load_pool_cache(args.pool_cache, want_filler=use_buckets)
    if pools is None:
        pools = asyncio.run(build_tts_pools(per_voice=args.per_voice, want_filler=use_buckets))
        save_pool_cache(pools, args.pool_cache)
    kw_bases = [(t, trim_silence(w)) for t, w in pools["kw"]]
    kw_bases = [(t, w) for t, w in kw_bases if w is not None and len(w) > 600]
    conf_bases = [(t, trim_silence(w)) for t, w in pools["conf"]]
    conf_bases = [(t, w) for t, w in conf_bases if w is not None and len(w) > 600]
    filler_bases = [(t, trim_silence(w)) for t, w in pools.get("filler", [])]
    filler_bases = [(t, w) for t, w in filler_bases if w is not None and len(w) > 400]
    if not kw_bases:
        raise SystemExit("No keyword bases synthesised; cannot build corpus.")
    print(f"  filler bases: {len(filler_bases)}")

    # Held-out clean keyword waveforms, used by evaluate_industrial_v2.py for the
    # streaming-recall test.
    dump = args.dump_bases or "keyword_bases.npy"
    if dump:
        try:
            np.save(dump, np.array([w for _, w in kw_bases[:600]], dtype=object),
                    allow_pickle=True)
            print(f"  keyword bases cached for evaluation: {dump}")
        except Exception as e:
            print(f"  could not write {dump}: {e}")

    nb = NoiseBank(index_csv=args.noise_index)
    sb = SpeechBank()

    total = args.pos + args.neg
    per_chunk = int(np.ceil(total / n_chunks))
    jobs = []
    assigned = 0
    for c in range(n_chunks):
        take = min(per_chunk, total - assigned)
        if take <= 0:
            break
        pos_share = int(round(take * args.pos / total))
        jobs.append((assigned, pos_share, take - pos_share, args.seed + c * 7919, use_buckets))
        assigned += take

    print(f"\n[Build] {args.pos:,} positives + {args.neg:,} negatives "
          f"across {len(jobs)} chunks on {args.workers} workers")
    noise_paths = nb.paths
    speech_flacs, speech_cmds = sb.flacs, sb.cmds

    results = []
    if args.workers <= 1:
        _init_worker(noise_paths, speech_flacs, speech_cmds, kw_bases, conf_bases,
                     args.seed, args.profile, filler_bases)
        for j in jobs:
            results.append(_run_chunk(j))
    else:
        with ProcessPoolExecutor(
            max_workers=args.workers,
            initializer=_init_worker,
            initargs=(noise_paths, speech_flacs, speech_cmds, kw_bases, conf_bases,
                      args.seed, args.profile, filler_bases),
        ) as ex:
            futs = [ex.submit(_run_chunk, j) for j in jobs]
            for k, f in enumerate(futs):
                results.append(f.result())
                print(f"  chunk {k+1}/{len(jobs)} done", flush=True)

    X = np.concatenate([r[0] for r in results], axis=0)
    y = np.concatenate([r[1] for r in results], axis=0)
    groups = np.concatenate([r[2] for r in results], axis=0)
    snrs = np.concatenate([r[3] for r in results], axis=0)
    buckets = np.concatenate([r[4] for r in results], axis=0)
    lvls = np.concatenate([r[5] for r in results], axis=0)

    if len(X.shape) == 3:
        X = np.expand_dims(X, axis=-1)
    X = X.astype(np.float32)

    tr, va = stratified_split(groups, y, 0.2, seed=args.seed, buckets=buckets)
    print(f"\n[Split] train={len(tr):,}  val={len(va):,}")

    np.savez_compressed(
        args.out,
        X_train=X[tr], y_train=y[tr], g_train=groups[tr], snr_train=snrs[tr],
        b_train=buckets[tr], lvl_train=lvls[tr],
        X_val=X[va], y_val=y[va], g_val=groups[va], snr_val=snrs[va],
        b_val=buckets[va], lvl_val=lvls[va],
    )

    print(f"  positives: {int((y==1).sum()):,}   negatives: {int((y==0).sum()):,}")
    if use_buckets:
        print(f"  {'bucket':<28} | {'n':>8} | {'share':>6} | {'loss w':>6} | {'lvl dBFS':>16}")
        for b, (_c, _r, _s, w) in BUCKET_TABLE.items():
            m = buckets == b
            c = int(m.sum())
            if c == 0:
                continue
            lv = lvls[m]
            print(f"  {b:<28} | {c:8,} | {c/len(buckets)*100:5.1f}% | {w:6.2f} | "
                  f"{lv.min():7.1f}..{lv.max():5.1f}")
    else:
        for kind in NEG_MIX:
            c = int(np.sum([g.startswith(f"neg|{kind}|") for g in groups]))
            print(f"    neg/{kind:<16} {c:,}")
    print(f"  file: {args.out}  ({os.path.getsize(args.out)/1024/1024:.1f} MB)")
    print(f"  elapsed: {time.time()-t0:.1f}s")


## Trainer

Checkpoint selection maximises offset-stressed TPR subject to FPR <= budget; OHEM, focal, sqrt-law class weights, mic-gain jitter, per-bucket loss and metrics.

*Inlined from `train_industrial_v2.py`: 559 lines, 37 CLI lines and 1 cross-module imports removed — everything shares one namespace here.*


In [12]:
"""
Industrial KWS Training v2 -- optimise TPR at a fixed FPR budget
----------------------------------------------------------------
The v1 trainer selected checkpoints on ROC-AUC measured at a hardcoded 0.85
threshold, and weighted negatives up to 2.5x. Both choices push the operating
point toward "say no": 17% of v1 positives scored below 0.85 while the soundscape
bank already had non-speech slices above it (pub_babble 0.933).

v2 changes
  1. Checkpoint selection maximises TPR subject to FPR <= --fpr_budget, computed on
     the full validation ROC. The threshold that satisfies the budget is stored in
     the checkpoint so deployment does not have to guess.
  2. Class weights are computed from the dataset ratio with a sqrt law and an
     optional positive bias, instead of the old "penalise negatives" rule.
  3. On-batch hard-negative mining (OHEM): only the hardest fraction of negatives in
     each minibatch contributes gradient. This tightens the FPR tail without
     weakening the positive gradient.
  4. Focal gamma is lowered (default 1.5) because gamma=2.0 down-weights exactly the
     easy-but-quiet positives that dominate the recall loss.
  5. Offset-stress validation: val positives are also scored after a random temporal
     roll, and the reported TPR is the mean over the unshifted and shifted copies.
     This is what a 100 ms-stride streaming window actually sees.
  6. Per-bucket loss and metrics. When the dataset carries mixture-recipe tags
     (dataset_v4.npz, written by build_corpus_v2.py --profile v4) every sample gets
     a bucket id such as P2_word_two_noise_hard or N7_babble_two_noise_hard. The
     trainer reports loss, accuracy and TPR/FPR separately per bucket every epoch and
     applies a per-bucket loss weight, so a regression in one regime cannot be hidden
     inside a single average.
  7. Mic-gain augmentation. A constant added to the log-mel is a multiplicative change
     in waveform amplitude, so jittering it simulates a preamp/AGC change and forces the
     model to key on spectral shape rather than absolute loudness.
  8. Two local-run affordances: --hard_neg_k mines a fixed *count* of negatives so the
     OHEM behaviour does not change with batch size (the Kaggle path already assumed
     this flag existed), and --memmap streams X_train/X_val out of the compressed .npz
     into .npy sidecars that are then memory-mapped, so an 850 MB corpus does not have to
     compete with the XLA arena for RAM.

    python train_industrial_v2.py --data dataset_v4.npz --arch bcconformer_50k \
        --epochs 24 --fpr_budget 0.005 --save best_v4_50k.flax
"""

import argparse
import functools
import os
import time
from collections import defaultdict

import numpy as np
import jax
import jax.numpy as jnp
import optax
from flax import serialization
from flax.training import train_state
from sklearn.metrics import roc_auc_score


# Must match build_corpus_v2.BUCKET_LOSS_WEIGHT; kept local so the trainer does not
# need to import the (heavy) dataset builder.
BUCKET_LOSS_WEIGHT = {
    "P0_word_direct": 1.00,
    "P1_word_one_noise": 1.15,
    "P2_word_two_noise": 1.25,
    "P3_word_two_noise_hard": 1.30,
    "N0_filler_direct": 0.80,
    "N1_filler_one_noise": 0.90,
    "N2_filler_two_noise": 1.00,
    "N3_filler_two_noise_hard": 1.10,
    "N4_real_direct": 0.90,
    "N5_real_one_noise": 1.00,
    "N6_real_two_noise": 1.10,
    "N7_real_two_noise_hard": 1.20,
    "N8_babble_direct": 1.00,
    "N9_babble_one_noise": 1.10,
    "N10_babble_two_noise": 1.20,
    "N11_babble_two_noise_hard": 1.30,
    "N12_noise_direct": 0.90,
    "N13_noise_one_noise": 1.00,
    "N14_noise_two_noise": 1.00,
    "N15_noise_two_noise_hard": 1.10,
    "N16_confus_direct": 1.00,
    "N17_confus_one_noise": 1.10,
    "N18_confus_two_noise": 1.10,
    "N19_confus_two_noise_hard": 1.20,
}


# ==============================================================================
# Augmentation
# ==============================================================================

def spec_augment(batch_x: np.ndarray, rng: np.random.Generator, prob: float = 0.5,
                 max_f: int = 5, max_t: int = 7) -> np.ndarray:
    b, t, m, c = batch_x.shape
    out = batch_x.copy()
    for i in range(b):
        if rng.random() < prob:
            f_w = int(rng.integers(1, max_f + 1))
            f_st = int(rng.integers(0, max(1, m - f_w)))
            out[i, :, f_st:f_st + f_w, :] = np.min(out[i])
            t_w = int(rng.integers(1, max_t + 1))
            t_st = int(rng.integers(0, max(1, t - t_w)))
            out[i, t_st:t_st + t_w, :, :] = np.min(out[i])
    return out


def gain_augment(batch_x: np.ndarray, rng: np.random.Generator, prob: float = 0.6,
                 span_db: float = 12.0) -> np.ndarray:
    """Mic preamp / AGC jitter.

    Log-mel is a log of energy, so adding a constant d to the log-mel multiplies the
    waveform energy by e^d, i.e. the amplitude by e^(d/2). A +/- span_db/20 dB swing
    in amplitude therefore becomes a constant offset of span_db/20*ln(10) in feature
    space. This is the cheapest way to make the model stop keying on absolute level.
    """
    out = batch_x.copy()
    b = out.shape[0]
    k = int(np.ceil(span_db / 20.0 * np.log(10.0)))
    for i in range(b):
        if rng.random() < prob:
            out[i] = out[i] + float(rng.uniform(-k, k))
    return out


def roll_batch(specs: np.ndarray, rng: np.random.Generator, offsets: np.ndarray) -> np.ndarray:
    out = np.empty_like(specs)
    for i, off in enumerate(offsets):
        out[i] = np.roll(specs[i], int(off), axis=0)
    return out


# ==============================================================================
# Training state
# ==============================================================================

def create_train_state(rng, model, learning_rate: float, total_steps: int, init_params=None,
                      weight_decay: float = 1e-4):
    dummy = jnp.ones((1, 49, 40, 1), dtype=jnp.float32)
    if init_params is None:
        variables = model.init(rng, dummy, train=False)
        params = variables["params"]
    else:
        params = init_params

    warmup = max(30, int(0.06 * total_steps))
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=1e-5, peak_value=learning_rate, warmup_steps=warmup,
        decay_steps=max(total_steps, warmup + 1), end_value=1e-5)
    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=schedule, weight_decay=weight_decay))
    return train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)


@functools.partial(jax.jit, static_argnames=("gamma", "hard_neg_frac", "hard_neg_k"))
def train_step(state, batch_x, batch_y, rng, gamma, class_weights, hard_neg_frac,
               hard_neg_k=0, bucket_w=None):
    def loss_fn(params):
        logits = state.apply_fn({"params": params}, batch_x, train=True,
                                rngs={"dropout": rng})
        probs = jax.nn.softmax(logits, axis=-1)
        p_t = jnp.take_along_axis(probs, jnp.expand_dims(batch_y, axis=-1), axis=-1)[:, 0]
        ce = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=batch_y)
        focal = jnp.power(jnp.clip(1.0 - p_t, 1e-4, 1.0), gamma)
        sw = class_weights[batch_y]
        per_sample = ce * focal * sw
        if bucket_w is not None:
            per_sample = per_sample * bucket_w

        # On-batch hard-negative mining: keep only the hardest negatives.
        # `hard_neg_k` (a count) and `hard_neg_frac` (a fraction of the batch) are both
        # static, so the cutoff index derives from static values only. A count is the
        # batch-size-independent form: the same number of negatives is mined whether the
        # batch is 64 (local CPU) or 512 (Kaggle GPU).
        is_neg = batch_y == 0
        k = int(hard_neg_k) if hard_neg_k > 0 else int(hard_neg_frac * batch_x.shape[0])
        if hard_neg_k > 0 or hard_neg_frac < 1.0:
            k = max(1, k)
            neg_vals = jnp.where(is_neg, per_sample, jnp.inf)
            min_neg = jnp.min(jnp.where(is_neg, per_sample, jnp.inf))
            cut = jnp.minimum(jnp.sort(neg_vals)[k - 1], min_neg)
            keep = jnp.logical_or(batch_y == 1, per_sample >= cut)
            per_sample = per_sample * jnp.where(keep, 1.0, 0.0)

        return jnp.mean(per_sample), logits

    (loss, logits), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    new_state = state.apply_gradients(grads=grads)
    acc = jnp.mean(jnp.argmax(logits, axis=-1) == batch_y)
    return new_state, loss, acc


@jax.jit
def predict_probs(state, batch_x):
    logits = state.apply_fn({"params": state.params}, batch_x, train=False)
    return jax.nn.softmax(logits, axis=-1)[:, 1]


# ==============================================================================
# Operating-point helpers
# ==============================================================================

def threshold_for_fpr(neg_probs: np.ndarray, fpr_budget: float) -> float:
    """Smallest threshold whose negative false-positive rate stays within budget."""
    if len(neg_probs) == 0:
        return 0.5
    allowed = int(np.floor(fpr_budget * len(neg_probs)))
    if allowed <= 0:
        return float(np.nextafter(neg_probs.max(), 1.0))
    srt = np.sort(neg_probs)[::-1]
    return float(srt[allowed - 1])


def tpr_at_threshold(pos_probs: np.ndarray, t: float) -> float:
    return float(np.mean(pos_probs >= t)) if len(pos_probs) else 0.0


def tpr_at_fpr_budget(pos_probs: np.ndarray, neg_probs: np.ndarray, budget: float):
    t = threshold_for_fpr(neg_probs, budget)
    return tpr_at_threshold(pos_probs, t), t, float(np.mean(neg_probs >= t))


# ==============================================================================
# Per-bucket metrics
# ==============================================================================

BUCKET_REASON = {
    "P0_word_direct": "word only, no noise",
    "P1_word_one_noise": "word + 1 noise layer",
    "P2_word_two_noise": "word + 2 noise layers",
    "P3_word_two_noise_hard": "word + 2 noise layers, low SNR, full distortion",
    "N0_filler_direct": "unimportant word, no noise",
    "N1_filler_one_noise": "unimportant word + 1 noise layer",
    "N2_filler_two_noise": "unimportant word + 2 noise layers",
    "N3_filler_two_noise_hard": "unimportant word + 2 layers, low SNR, full distortion",
    "N4_real_direct": "real human speech, no noise",
    "N5_real_one_noise": "real human speech + 1 noise layer",
    "N6_real_two_noise": "real human speech + 2 noise layers",
    "N7_real_two_noise_hard": "real human speech + 2 layers, low SNR",
    "N8_babble_direct": "2-8 live talkers, no extra noise",
    "N9_babble_one_noise": "2-8 live talkers + 1 noise layer",
    "N10_babble_two_noise": "2-8 live talkers + 2 noise layers",
    "N11_babble_two_noise_hard": "2-8 live talkers + 2 layers, low SNR",
    "N12_noise_direct": "single soundscape, nothing added",
    "N13_noise_one_noise": "soundscape + 1 noise layer",
    "N14_noise_two_noise": "soundscape + 2 more noise layers",
    "N15_noise_two_noise_hard": "soundscape + 2 more layers, full distortion",
    "N16_confus_direct": "near-miss word, no noise",
    "N17_confus_one_noise": "near-miss word + 1 noise layer",
    "N18_confus_two_noise": "near-miss word + 2 noise layers",
    "N19_confus_two_noise_hard": "near-miss word + 2 layers, low SNR",
}


def bucket_stats(probs: np.ndarray, buckets: np.ndarray, y: np.ndarray,
                 thr: float) -> dict:
    """TPR for positive buckets, FPR for negative buckets, at the operating point."""
    out = {}
    for b in np.unique(buckets):
        m = buckets == b
        p = probs[m]
        lab = y[m]
        out[str(b)] = float(np.mean(p >= thr)) if (lab == 1).any() else float(np.mean(p >= thr))
    return out


def print_bucket_table(probs, buckets, y, thr):
    pos = defaultdict(lambda: [0, 0, []])
    neg = defaultdict(lambda: [0, 0, []])
    for p, b, l in zip(probs, buckets, y):
        tgt = pos if l == 1 else neg
        e = tgt[str(b)]
        e[0] += 1
        e[1] += int(p >= thr)
        e[2].append(float(p))
    # Two tables, because a single column cannot say whether a large number is good: 3%
    # recall is a failure and 3% false-alarm rate is also a failure, but a reader has to
    # know which polarity a row is before the number means anything.
    print(f"    POSITIVE buckets @ thr {thr:.3f}   (recall, higher is better)")
    print(f"    {'bucket':<28} | {'n':>6} | {'recall':>7} | {'MISSED':>7} | "
          f"{'mean conf':>9} | {'p05 conf':>8}")
    for name, e in pos.items():
        r = e[1] / max(1, e[0])
        print(f"    {name:<28} | {e[0]:6d} | {r*100:6.2f}% | {(1-r)*100:6.2f}% | "
              f"{np.mean(e[2]):9.4f} | {np.percentile(e[2],5):8.4f}")
    _p = sum(e[0] for e in pos.values())
    _h = sum(e[1] for e in pos.values())
    if _p:
        print(f"    {'POOLED':<28} | {_p:6d} | {_h/_p*100:6.2f}% | "
              f"{(1-_h/_p)*100:6.2f}% |")

    print(f"    NEGATIVE buckets @ thr {thr:.3f}   (false-alarm rate, lower is better)")
    print(f"    {'bucket':<28} | {'n':>6} | {'FA rate':>8} | {'FA count':>8} | "
          f"{'mean conf':>9} | {'p95 conf':>8}")
    for name, e in neg.items():
        print(f"    {name:<28} | {e[0]:6d} | {e[1]/max(1,e[0])*100:7.2f}% | {e[1]:8d} | "
              f"{np.mean(e[2]):9.4f} | {np.percentile(e[2],95):8.4f}")
    _n = sum(e[0] for e in neg.values())
    _f = sum(e[1] for e in neg.values())
    if _n:
        print(f"    {'POOLED':<28} | {_n:6d} | {_f/_n*100:7.2f}% | {_f:8d} |")


# ==============================================================================
# Training
# ==============================================================================

# ==============================================================================
# Low-memory corpus loading
# ==============================================================================

def stream_npy_member_to_file(npz_path: str, member: str, out_path: str,
                               chunk: int = 1 << 22):
    """Copy a single .npy member out of an .npz to a standalone .npy in bounded RAM.

    `np.load(npz)["X_train"]` inflates the whole member into anonymous memory, which for
    dataset_v5 is 847 MB -- more than this machine has free. A memmapped sidecar keeps the
    corpus on disk and lets the OS page it in per batch, so the training loop and the
    XLA arena are the only large anonymous allocations. The sidecar is written once and
    reused: a header/size check decides whether it is already complete.
    """
    from numpy.lib import format as npformat
    import zipfile

    with zipfile.ZipFile(npz_path) as z, z.open(member) as src:
        version = npformat.read_magic(src)
        reader = (npformat.read_array_header_1_0 if version == (1, 0)
                  else npformat.read_array_header_2_0)
        shape, fortran, dtype = reader(src)
        nbytes = int(np.prod(shape)) * dtype.itemsize

        if os.path.exists(out_path):
            try:
                with open(out_path, "rb") as f:
                    v2 = npformat.read_magic(f)
                    r2 = (npformat.read_array_header_1_0 if v2 == (1, 0)
                          else npformat.read_array_header_2_0)
                    s2, _, d2 = r2(f)
                    if s2 == shape and d2 == dtype and os.path.getsize(out_path) == f.tell() + nbytes:
                        return shape, dtype
            except Exception:
                pass

        tmp = out_path + ".part"
        with open(tmp, "wb") as dst:
            header = {"descr": npformat.dtype_to_descr(dtype),
                      "fortran_order": fortran, "shape": shape}
            if version == (1, 0):
                npformat.write_array_header_1_0(dst, header)
            else:
                npformat.write_array_header_2_0(dst, header)
            while True:
                buf = src.read(chunk)
                if not buf:
                    break
                dst.write(buf)
        os.replace(tmp, out_path)
    return shape, dtype


def load_corpus(data_path: str, use_memmap: bool, cache_dir: str = "corpus_cache"):
    """Returns (X_train, X_val, NpzFile). X_* are memmaps when use_memmap is set."""
    data = np.load(data_path, allow_pickle=True)
    if not use_memmap:
        return data["X_train"], data["X_val"], data

    os.makedirs(cache_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(data_path))[0]
    out = {}
    for name in ("X_train", "X_val"):
        side = os.path.join(cache_dir, f"{stem}_{name}.npy")
        t0 = time.time()
        shape, dtype = stream_npy_member_to_file(data_path, f"{name}.npy", side)
        print(f" corpus sidecar    : {side} {shape} {dtype} "
              f"({os.path.getsize(side)/1e6:.0f} MB, {time.time()-t0:.0f}s)")
        out[name] = np.load(side, mmap_mode="r")
    return out["X_train"], out["X_val"], data


# ==============================================================================
# Training
# ==============================================================================

def run_training(data_path="dataset_v2.npz", arch="bcconformer_50k", epochs=30,
                 batch_size=64, lr=1e-3, save_path="best_v2.flax",
                 fpr_budget=0.005, hard_neg_frac=0.5, hard_neg_k=0, gamma=1.5,
                 pos_bias=1.15, spec_aug_prob=0.35, offset_stress=True,
                 warm_start=None, seed=7, gain_aug_prob=0.6, gain_aug_db=12.0,
                 bucket_weight=True, val_metric_cap=12000, use_memmap=False):
    print("=" * 78)
    print(f" INDUSTRIAL TRAINING v2  |  arch={arch}  |  objective = max TPR @ FPR <= {fpr_budget*100:.2f}%")
    print("=" * 78)

    X_train, X_val, data = load_corpus(data_path, use_memmap)
    y_train, y_val = data["y_train"], data["y_val"]
    if X_train.ndim == 3:
        X_train = np.expand_dims(X_train, -1)
    if X_val.ndim == 3:
        X_val = np.expand_dims(X_val, -1)
    b_train = data["b_train"].astype(str) if "b_train" in data.files else None
    b_val = data["b_val"].astype(str) if "b_val" in data.files else None

    print(f" dataset            : {data_path}")
    print(f" train              : {len(X_train):,}  ({int((y_train==1).sum()):,} pos / {int((y_train==0).sum()):,} neg)")
    print(f" val                : {len(X_val):,}  ({int((y_val==1).sum()):,} pos / {int((y_val==0).sum()):,} neg)")
    if b_train is not None:
        print(f" mixture buckets    : {len(np.unique(b_train))}  (per-bucket loss + metrics enabled)")

    model = get_model(arch, num_classes=2)
    n_params = count_params(model)
    print(f" model              : {arch}  {n_params:,} params  ({n_params/1024:.1f} KB INT8)")

    # Class weights: sqrt law, optional positive bias. Unlike v1 this does not
    # penalise the negative class, which is what capped recall.
    n_pos = max(1, int((y_train == 1).sum()))
    n_neg = max(1, int((y_train == 0).sum()))
    ratio = n_neg / n_pos
    neg_w = float(np.clip(np.sqrt(ratio), 0.5, 2.0))
    pos_w = float(pos_bias * neg_w)
    cw = jnp.array([neg_w, pos_w], dtype=jnp.float32)
    print(f" class weights      : neg={neg_w:.3f} pos={pos_w:.3f}  (pos:neg count ratio 1:{ratio:.2f})")
    print(f" focal gamma        : {gamma}   hard-negative mining: "
          f"{'k=' + str(hard_neg_k) if hard_neg_k > 0 else 'frac=' + str(hard_neg_frac)}")
    print(f" spec-augment prob  : {spec_aug_prob}   mic-gain aug: p={gain_aug_prob}, +/-{gain_aug_db} dB")
    print(f" bucket loss weight : {'on' if (bucket_weight and b_train is not None) else 'off'}")

    rng_np = np.random.default_rng(seed)
    key = jax.random.PRNGKey(seed)
    key, init_key = jax.random.split(key)

    init_params = None
    if warm_start and os.path.exists(warm_start):
        with open(warm_start, "rb") as f:
            ck = serialization.msgpack_restore(f.read())
        try:
            variables = model.init(init_key, jnp.ones((1, 49, 40, 1), jnp.float32), train=False)
            template = variables["params"]
            init_params = serialization.from_bytes(template, ck["params"])
            print(f" warm start         : {warm_start}")
        except Exception as e:
            print(f" warm start skipped ({str(e)[:60]})")
            init_params = None

    steps_per_epoch = len(X_train) // batch_size
    total_steps = epochs * steps_per_epoch
    state = create_train_state(init_key, model, lr, total_steps, init_params=init_params)

    pos_mask = y_val == 1
    neg_mask = y_val == 0
    X_val_pos = X_val[pos_mask][:, :, :, 0]
    X_val_neg = X_val[neg_mask][:, :, :, 0]

    # Per-bucket validation subset: cheap enough to score every epoch, and it keeps
    # the rare hard regimes visible instead of averaging them away.
    bucket_val = b_val if b_val is not None else np.where(y_val == 1, "pos", "neg")
    if val_metric_cap and len(X_val) > val_metric_cap:
        sub = rng_np.choice(len(X_val), size=val_metric_cap, replace=False)
        X_val_b_spec = X_val[sub][:, :, :, 0]
        y_val_b = y_val[sub]
        bucket_val = bucket_val[sub]
    else:
        X_val_b_spec = X_val[:, :, :, 0]
        y_val_b = y_val

    roll_offsets = rng_np.integers(0, 20, size=len(X_val_pos))
    X_val_pos_rolled = roll_batch(X_val_pos, rng_np, roll_offsets)
    print(f" offset-stress set  : {len(X_val_pos)} positives rolled by U[0,20) frames")
    print(f" per-bucket report  : {len(X_val_b_spec):,} val samples across "
          f"{len(np.unique(bucket_val))} buckets")

    # Bucket -> per-sample loss weight, as a dense vector for the training step.
    bucket_w_train = None
    if bucket_weight and b_train is not None:
        default_w = 1.0
        bucket_w_train = np.array([BUCKET_LOSS_WEIGHT.get(str(b), default_w) for b in b_train],
                                  dtype=np.float32)

    best_score = -1.0
    best = {}
    # The old table printed an "FPR" column next to every recall column, which read like a
    # measurement but was not one: the threshold is *chosen* so that exactly the budget's
    # worth of negatives cross it, so the column always echoed the budget. Worse, "recall
    # 77.74% at FPR 0.500%" is precisely the sentence that gets mistaken for a deployment
    # claim, and on real audio that same threshold fired 4.25 times per hour.
    print("=" * 96)
    print(" HOW TO READ THIS TABLE")
    print("=" * 96)
    print(f"  thr          the operating threshold, chosen so that {fpr_budget*100:.2f}% of the")
    print(f"               validation NEGATIVES cross it. That is a property of 1 s,")
    print(f"               level-normalised slices -- it is NOT a deployment number.")
    print(f"  FA/1k neg    how many of every 1000 validation negatives cross thr, as a whole")
    print(f"               number. At a {fpr_budget*100:.2f}% budget that is about "
          f"{int(round(fpr_budget*1000))} negatives; a printed")
    print(f"               percentage like 0.500% would be a rounded echo of the budget.")
    print("  recall@bud   true-positive rate on validation positives at thr. Higher better.")
    print("  recall@roll  the SAME positives, randomly rolled inside the 1 s window.")
    print("               The gap between the two columns is the model's position")
    print("               dependence, and it is the number that decides deployment value.")
    print("  SCORE        0.5*(recall@bud + recall@roll). This is the checkpoint selection")
    print("               criterion; the * marks the epoch that owns the saved checkpoint.")
    print("  recall@0.12% the v1 reference budget, kept only for continuity with old tables.")
    print()
    print("  MEASURED DEPLOYMENT FALSE ALARMS ARE NOT IN THIS TABLE. A checkpoint can look")
    print("  excellent here and still fire constantly on real audio -- that has happened twice")
    print("  in this project. Re-freeze the threshold against sweep_threshold.py output,")
    print("  which scores real soundscape hours and continuous human speech.")
    print("=" * 96)
    header = (f"{'Ep':>3} | {'Loss':>7} | {'TrAcc':>6} | {'AUC':>7} | {'thr':>6} | "
              f"{'FA/1k':>6} | {'recall@bud':>10} | {'recall@roll':>11} | {'SCORE':>7} | "
              f"{'recall@0.12':>10}")
    print("-" * 96)
    print(header)
    print("-" * 96)

    t_start = time.time()
    for ep in range(1, epochs + 1):
        # Gather per batch rather than materialising X_train[perm]: a full permuted copy
        # of the corpus is another 850 MB on a 108k-sample corpus, which does not fit
        # alongside the corpus itself on a 16 GB machine. The batch contents are
        # identical either way.
        perm = rng_np.permutation(len(X_train))

        losses, accs = [], []
        for s in range(steps_per_epoch):
            idx = perm[s * batch_size:(s + 1) * batch_size]
            bx = X_train[idx]
            by = y_train[idx]
            bx = spec_augment(bx, rng_np, prob=spec_aug_prob)
            bx = gain_augment(bx, rng_np, prob=gain_aug_prob, span_db=gain_aug_db)
            bw = None
            if bucket_w_train is not None:
                bw = jnp.asarray(bucket_w_train[idx], jnp.float32)
            key, step_key = jax.random.split(key)
            state, loss, acc = train_step(state, jnp.asarray(bx, jnp.float32),
                                          jnp.asarray(by, jnp.int32), step_key,
                                          gamma=gamma, class_weights=cw,
                                          hard_neg_frac=hard_neg_frac,
                                          hard_neg_k=hard_neg_k, bucket_w=bw)
            losses.append(float(loss))
            accs.append(float(acc))

        # ---- validation scoring ----
        def score(arr):
            out = []
            a = np.asarray(arr, dtype=np.float32)
            if a.ndim == 3:
                a = np.expand_dims(a, -1)
            for i in range(0, len(a), 512):
                out.extend(np.array(predict_probs(state, jnp.asarray(a[i:i + 512]))))
            return np.array(out)

        p_pos = score(X_val_pos)
        p_neg = score(X_val_neg)
        p_pos_roll = score(X_val_pos_rolled)
        p_b = score(X_val_b_spec)
        p_all = np.concatenate([p_pos, p_neg])
        y_all = np.concatenate([np.ones(len(p_pos), np.int32), np.zeros(len(p_neg), np.int32)])
        auc = roc_auc_score(y_all, p_all) if len(set(y_all.tolist())) > 1 else 0.5

        tpr_b, thr_b, fpr_b = tpr_at_fpr_budget(p_pos, p_neg, fpr_budget)
        tpr_b_roll, thr_r, fpr_r = tpr_at_fpr_budget(p_pos_roll, p_neg, fpr_budget)
        tpr_off = 0.5 * (tpr_b + tpr_b_roll) if offset_stress else tpr_b

        # hard FPR reference: the 0.12% budget v1 claimed
        tpr_h, thr_h, fpr_h = tpr_at_fpr_budget(p_pos, p_neg, 0.0012)
        tpr_h_roll, _, _ = tpr_at_fpr_budget(p_pos_roll, p_neg, 0.0012)

        score_key = tpr_off if offset_stress else tpr_b
        marker = ""
        improved = score_key > best_score + 1e-6
        if improved and ep >= 3:
            best_score = score_key
            thr_pick = thr_b if offset_stress else thr_b
            best = {
                "params": state.params, "arch": arch, "param_count": n_params,
                "epoch": ep, "val_auc": float(auc), "fpr_budget": fpr_budget,
                "threshold": float(thr_pick), "tpr_at_budget": float(tpr_b),
                "fpr_at_budget": float(fpr_b),
                "tpr_offset_stress": float(tpr_off),
                "tpr_at_0012": float(tpr_h), "fpr_at_0012": float(fpr_h),
                "tpr_offset_0012": float(0.5 * (tpr_h + tpr_h_roll)),
                "tpr_at_085": tpr_at_threshold(p_pos, 0.85),
                "fpr_at_085": float(np.mean(p_neg >= 0.85)),
            }
            best["bucket_metrics"] = {str(k): float(v) for k, v in bucket_stats(p_b, bucket_val,
                                                                              y_val_b, thr_b).items()}
            with open(save_path, "wb") as f:
                f.write(serialization.to_bytes(best))
            marker = " *"

        if ep % 2 == 0 or ep == epochs or marker:
            fa_per_k = int(round(1000.0 * float(np.sum(p_neg >= thr_b)) / max(1, len(p_neg))))
            print(f"{ep:3d} | {np.mean(losses):7.4f} | {np.mean(accs)*100:5.1f}% | {auc*100:6.3f}% | "
                  f"{thr_b:6.3f} | {fa_per_k:6d} | {tpr_b*100:9.2f}% | {tpr_b_roll*100:10.2f}% | "
                  f"{score_key*100:6.2f}% | {tpr_h*100:9.2f}%{marker}", flush=True)
            if b_val is not None and (ep % 4 == 0 or ep == epochs or marker):
                print_bucket_table(p_b, bucket_val, y_val_b, thr_b)

    print("-" * 96)
    print(f" training time      : {time.time() - t_start:.1f}s")
    print(f" best epoch         : {best.get('epoch')}")
    print(f" SCORE (selection)  : {best.get('tpr_offset_stress', 0)*100:.2f}%  "
          f"= 0.5*(recall {best.get('tpr_at_budget', 0)*100:.2f}% + "
          f"rolled {0.5*best.get('tpr_offset_stress', 0)*100 - best.get('tpr_at_budget', 0)*100:.2f}%)")
    print(f" recall @ 0.12% FPR: {best.get('tpr_at_0012', 0)*100:.2f}%"
          f"  (offset-stressed {best.get('tpr_offset_0012', 0)*100:.2f}%)")
    print(f" stored threshold   : {best.get('threshold', 0):.4f}")
    print("                      ^ a VALIDATION threshold. It has not been checked against")
    print("                        real audio. Run sweep_threshold.py and freeze a point")
    print("                        before treating it as a deployment operating point.")
    print(f" checkpoint         : {save_path}")
    print("=" * 96, flush=True)
    return best


## Benchmark

ROC, per-bucket breakdown, level robustness, offset stress, soundscape false alarms, continuous-speech false alarms, streaming recall.

*Inlined from `evaluate_industrial_v2.py`: 516 lines, 3 CLI lines and 2 cross-module imports removed — everything shares one namespace here.*


In [13]:
"""
Industrial Evaluation v2
------------------------
End-to-end benchmark that scores the thing that actually ships: a 1 s sliding
window detector, not a single-frame classifier.

Sections
  1. Validation ROC + TPR at several FPR budgets (with the checkpoint's stored
     threshold highlighted).
  2. Offset stress: TPR as a function of where the keyword sits inside the 1 s
     window. v1 fell from 83% to 70% across this sweep; anything close to flat is
     what a 100 ms-stride streamer needs.
  3. Soundscape stress over noise_bank_v2: per-class max confidence, false alarms
     per hour of audio, and the worst offenders by name.
  4. Continuous-speech false alarm rate with the industrial debouncer
     (EMA + 2 consecutive frames + 1.5 s refractory).
  5. Streaming recall: keyword spliced into real noise+speech beds at random
     offsets and random SNR, then detected through the sliding-window engine.
     Reports hit rate and detection latency.

    python evaluate_industrial_v2.py --model best_v2_50k.flax --arch bcconformer_50k
"""

import argparse
import csv
import glob
import os
import time
from collections import defaultdict

import numpy as np
import soundfile as sf
import jax
import jax.numpy as jnp
from flax import serialization
from scipy import signal


SAMPLE_RATE = 16000
WIN = 16000
STRIDE = 1600  # 100 ms


# ==============================================================================
# Streaming detector (mirrors kws_engine.py / evaluate_industrial.py)
# ==============================================================================

class StreamingDetector:
    """1 s sliding-window detector with a configurable confirmation policy.

    `need` hits inside any `span` consecutive windows is the general form of the
    classic "2 consecutive frames" rule. A 0.6 s wake word only fully fits one or two
    1 s windows, so requiring 2 *consecutive* frames is a much harsher test than
    requiring 2 hits inside 3 windows, while still rejecting isolated single spikes.
    """

    def __init__(self, predict_fn, extractor, threshold, ema_alpha=0.6,
                 need=2, span=2, refractory_steps=15, rms_gate=0.008, agg="ema",
                 hold_decay=0.85):
        self.predict_fn = predict_fn
        self.extractor = extractor
        self.threshold = threshold
        self.ema_alpha = ema_alpha
        self.need = need
        self.span = span
        self.refractory = refractory_steps
        self.rms_gate = rms_gate
        self.agg = agg
        self.hold_decay = hold_decay
        self.reset()

    def reset(self):
        self.ema = 0.0
        self.hold = 0.0
        self.hits = []
        self.lock = 0
        self.raw_peak = 0.0

    def _score(self, raw: float) -> float:
        """Aggregate the raw frame probability into a confirmation score.

        'ema'     -- exponential moving average. Penalises bursts: two high frames
                     in a row only reach ~0.71 of the way to the raw peak at alpha=0.6.
        'maxhold' -- decaying peak hold. A two-frame burst keeps its full value for
                     `span` windows, so a 0.6 s keyword still confirms, while an
                     isolated single-frame spike still does not.
        """
        if self.agg == "maxhold":
            self.hold = max(raw, self.hold * self.hold_decay)
            return self.hold
        self.ema = self.ema_alpha * raw + (1.0 - self.ema_alpha) * self.ema
        return self.ema

    def _decay(self):
        if self.agg == "maxhold":
            self.hold *= self.hold_decay
        else:
            self.ema = (1.0 - self.ema_alpha) * self.ema

    def run(self, audio, collect_raw=True):
        """Streams `audio`, returns (detections, first_detect_step, raw_triggers)."""
        self.reset()
        dets = []
        first = None
        raw_hits = 0
        n_steps = max(0, (len(audio) - WIN) // STRIDE + 1)
        for s in range(n_steps):
            if self.lock > 0:
                self.lock -= 1
            w = audio[s * STRIDE:s * STRIDE + WIN]
            rms = float(np.sqrt(np.mean(w ** 2)))
            if rms < self.rms_gate:
                self._decay()
                self.hits.append(0)
                if len(self.hits) > self.span:
                    self.hits.pop(0)
                continue
            spec = self.extractor.compute_spectrogram(w)
            p = float(self.predict_fn(jnp.expand_dims(jnp.expand_dims(jnp.array(spec), 0), -1))[0])
            if p > self.raw_peak:
                self.raw_peak = p
            if collect_raw and p >= self.threshold:
                raw_hits += 1
            score = self._score(p)
            self.hits.append(1 if score >= self.threshold else 0)
            if len(self.hits) > self.span:
                self.hits.pop(0)
            if sum(self.hits) >= self.need and self.lock == 0:
                dets.append(s * STRIDE)
                if first is None:
                    first = s * STRIDE
                self.lock = self.refractory
                self.hits = [0] * len(self.hits)
                self.ema = 0.0
                self.hold = 0.0
        return dets, first, raw_hits


# ==============================================================================
# Loading helpers
# ==============================================================================

def load_predict(checkpoint_path, arch):
    with open(checkpoint_path, "rb") as f:
        ck = serialization.msgpack_restore(f.read())
    model = get_model(arch, num_classes=2)
    params = ck["params"]

    @jax.jit
    def predict(x):
        logits = model.apply({"params": params}, x, train=False)
        return jax.nn.softmax(logits, axis=-1)[:, 1]

    return predict, ck


def to_16k(d, sr):
    if d.ndim > 1:
        d = d.mean(axis=1)
    if sr != SAMPLE_RATE:
        d = signal.resample(d, int(round(len(d) * SAMPLE_RATE / sr)))
    return d.astype(np.float32)


def batch_score(predict, specs, bs=512):
    out = []
    for i in range(0, len(specs), bs):
        chunk = np.asarray(specs[i:i + bs], dtype=np.float32)
        if chunk.ndim == 3:
            chunk = np.expand_dims(chunk, -1)
        out.extend(np.array(predict(jnp.asarray(chunk))))
    return np.array(out)


def load_index(index_csv="noise_bank_v2/index.csv"):
    rows = []
    if os.path.exists(index_csv):
        with open(index_csv, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                rows.append((r["path"].replace("/", os.sep), r["class"]))
    if not rows:
        for p in sorted(glob.glob(os.path.join("noise_bank", "*.wav"))):
            rows.append((p, os.path.splitext(os.path.basename(p))[0]))
    return rows


# ==============================================================================
# Sections
# ==============================================================================

def sec1_roc(predict, data_path):
    print("\n" + "=" * 86)
    print(" 1. VALIDATION ROC AND OPERATING POINTS")
    print("=" * 86)
    d = np.load(data_path, allow_pickle=True)
    X_val, y_val = d["X_val"], d["y_val"]
    p = batch_score(predict, X_val[:, :, :, 0])
    pos, neg = p[y_val == 1], p[y_val == 0]
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(y_val, p)
    print(f" val: {len(pos)} pos / {len(neg)} neg   AUC = {auc*100:.3f}%")
    print(f" positives : 1%={np.percentile(pos,1):.3f}  5%={np.percentile(pos,5):.3f}  "
          f"10%={np.percentile(pos,10):.3f}  median={np.median(pos):.3f}")
    print(f" negatives : 99%={np.percentile(neg,99):.3f}  99.9%={np.percentile(neg,99.9):.3f}  "
          f"max={neg.max():.3f}")
    print("-" * 86)
    print(f"{'FPR budget':>11} | {'threshold':>9} | {'TPR':>7} | {'actual FPR':>10} | {'FA / 1000 neg':>14}")
    for b in (0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05):
        allowed = int(np.floor(b * len(neg)))
        t = float(np.sort(neg)[::-1][allowed - 1]) if allowed > 0 else 1.0
        tpr = float(np.mean(pos >= t))
        print(f"{b*100:10.2f}% | {t:9.4f} | {tpr*100:6.2f}% | {float(np.mean(neg>=t))*100:9.3f}% | "
              f"{int(np.sum(neg>=t)):14d}")
    return pos, neg, auc


def sec1b_buckets(predict, data_path, threshold):
    print("\n" + "=" * 96)
    print(" 1b. PER-BUCKET BREAKDOWN AT THE OPERATING POINT")
    print("=" * 96)
    d = np.load(data_path, allow_pickle=True)
    if "b_val" not in d.files:
        print(" dataset has no bucket tags; skipping")
        return
    X_val, y_val = d["X_val"], d["y_val"]
    b_val = d["b_val"].astype(str)
    p = batch_score(predict, X_val[:, :, :, 0])
    order = sorted(np.unique(b_val), key=lambda b: (b[0] != "P", b))
    print(f" threshold {threshold:.4f}    val {len(p):,} samples")
    print(" Two tables on purpose: in a single column a 3% figure means failure for a")
    print(" positive bucket and success for a negative one, so the number is unreadable")
    print(" without knowing its polarity.")

    posb = [b for b in order if b.startswith("P")]
    negb = [b for b in order if b.startswith("N")]

    print(f"\n {'bucket':<28} | {'n':>6} | {'recall':>7} | {'MISSED':>7} | "
          f"{'mean conf':>9} | {'p05 conf':>8}   (higher recall is better)")
    print("-" * 96)
    for b in posb:
        m = b_val == b
        r = float(np.mean(p[m] >= threshold))
        print(f" {b:<28} | {int(m.sum()):6d} | {r*100:6.2f}% | {(1-r)*100:6.2f}% | "
              f"{p[m].mean():9.4f} | {np.percentile(p[m], 5):8.4f}")

    print(f"\n {'bucket':<28} | {'n':>6} | {'FA rate':>8} | {'FA count':>8} | "
          f"{'mean conf':>9} | {'p95 conf':>8}   (lower is better)")
    print("-" * 96)
    for b in negb:
        m = b_val == b
        r = float(np.mean(p[m] >= threshold))
        print(f" {b:<28} | {int(m.sum()):6d} | {r*100:7.2f}% | {int(np.sum(p[m] >= threshold)):8d} | "
              f"{p[m].mean():9.4f} | {np.percentile(p[m], 95):8.4f}")
    print("-" * 96)
    for prefix, name in (("P0", "direct"), ("P1", "+1 noise"),
                         ("P2", "+2 noise"), ("P3", "+2 noise hard")):
        sel = [b for b in posb if b.startswith(prefix)]
        if not sel:
            continue
        m = np.isin(b_val, sel)
        print(f" positive recipe {name:<14} pooled recall: {float(np.mean(p[m] >= threshold))*100:6.2f}%"
              f"  (n={int(m.sum()):,})")
    for grp, gname in (("N0", "filler direct"), ("N1", "filler +1 noise"),
                       ("N2", "filler +2 noise"), ("N3", "filler +2 noise hard")):
        sel = [b for b in negb if b.startswith(grp)]
        if not sel:
            continue
        m = np.isin(b_val, sel)
        print(f" filler recipe   {gname:<14} pooled FA  : {float(np.mean(p[m] >= threshold))*100:6.3f}%"
              f"  (n={int(m.sum()):,})")
    for grp, gname in (("N12", "noise alone"), ("N13", "noise +1 layer"),
                       ("N14", "noise +2 layers"), ("N15", "noise +2 hard")):
        sel = [b for b in negb if b.startswith(grp)]
        if not sel:
            continue
        m = np.isin(b_val, sel)
        print(f" noise  recipe   {gname:<14} pooled FA  : {float(np.mean(p[m] >= threshold))*100:6.3f}%"
              f"  (n={int(m.sum()):,})")


def sec1c_level(predict, data_path, threshold):
    print("\n" + "=" * 96)
    print(" 1c. LEVEL ROBUSTNESS (input gain / amplitude normalisation)")
    print("=" * 96)
    d = np.load(data_path, allow_pickle=True)
    X_val, y_val = d["X_val"], d["y_val"]
    pos = X_val[y_val == 1][:, :, :, 0]
    neg = X_val[y_val == 0][:, :, :, 0]
    if pos.shape[0] > 4000:
        pos = pos[:4000]
    if neg.shape[0] > 4000:
        neg = neg[:4000]
    k = 12.0 / 20.0 * np.log(10.0)  # dB amplitude -> log-mel offset
    print(f"{'gain':>8} | {'dB':>7} | {'pos recall':>10} | {'pos mean':>9} | {'neg FA':>8}")
    print("-" * 96)
    for g_db in (-24, -18, -12, -6, 0, 6, 12, 18):
        off = g_db / 20.0 * np.log(10.0)
        pp = batch_score(predict, np.clip(pos + off, -11.5129, 5.0))
        pn = batch_score(predict, np.clip(neg + off, -11.5129, 5.0))
        print(f"{10**(g_db/20.0):8.3f} | {g_db:+7.1f} | {float(np.mean(pp>=threshold))*100:9.2f}% | "
              f"{pp.mean():9.4f} | {float(np.mean(pn>=threshold))*100:7.3f}%")


def sec2_offset(predict, data_path, extractor):
    print("\n" + "=" * 86)
    print(" 2. OFFSET STRESS (keyword position inside the 1 s window)")
    print("=" * 86)
    d = np.load(data_path, allow_pickle=True)
    X_val, y_val = d["X_val"], d["y_val"]
    pos_specs = X_val[y_val == 1][:, :, :, 0]
    neg_probs = batch_score(predict, X_val[y_val == 0][:, :, :, 0])
    allowed = int(np.floor(0.005 * len(neg_probs)))
    thr = float(np.sort(neg_probs)[::-1][allowed - 1]) if allowed > 0 else 1.0
    print(f" threshold from 0.5% FPR budget: {thr:.4f}")
    print(f"{'roll (frames / ms)':>18} | {'TPR':>7} | {'mean conf':>10}")
    tprs = []
    for off in range(0, 25, 2):
        rolled = np.stack([np.roll(s, off, axis=0) for s in pos_specs])
        p = batch_score(predict, rolled)
        tpr = float(np.mean(p >= thr))
        tprs.append(tpr)
        print(f"{off:8d} / {off*20:4d} ms | {tpr*100:6.2f}% | {p.mean():10.4f}")
    print("-" * 86)
    print(f" TPR spread across offsets: min {min(tprs)*100:.2f}%  max {max(tprs)*100:.2f}%  "
          f"drop {100*(max(tprs)-min(tprs)):.2f} pts")


def sec3_soundscapes(predict, extractor, threshold, index_csv):
    print("\n" + "=" * 86)
    print(" 3. SOUNDSCAPE STRESS (noise_bank_v2)")
    print("=" * 86)
    rows = load_index(index_csv)
    by_class = defaultdict(list)
    for p, c in rows:
        by_class[c].append(p)
    total_sec = 0.0
    per_class = {}
    raw_fa = 0
    for cls, paths in sorted(by_class.items()):
        peaks = []
        secs = 0.0
        for p in paths:
            try:
                d = to_16k(*sf.read(p))
            except Exception:
                continue
            secs += len(d) / SAMPLE_RATE
            n = max(0, (len(d) - WIN) // 1600 + 1)
            for s in range(0, n, 10):  # every 1 s
                chunk = d[s * 1600:s * 1600 + WIN]
                if len(chunk) < WIN:
                    chunk = np.pad(chunk, (0, WIN - len(chunk)))
                if np.sqrt(np.mean(chunk ** 2)) < 0.002:
                    continue
                spec = extractor.compute_spectrogram(chunk)
                peaks.append(float(predict(jnp.expand_dims(jnp.expand_dims(jnp.array(spec), 0), -1))[0]))
        if not peaks:
            continue
        peaks = np.array(peaks)
        fa = int(np.sum(peaks >= threshold))
        raw_fa += fa
        total_sec += secs
        per_class[cls] = (peaks.max(), fa, secs, len(peaks))
    worst = sorted(per_class.items(), key=lambda kv: -kv[1][0])[:14]
    print(f"{'class':<28} | {'max conf':>8} | {'frames>=thr':>11} | {'hours':>6}")
    for cls, (mx, fa, secs, n) in worst:
        print(f"{cls:<28} | {mx:8.4f} | {fa:6d}/{n:<5d} | {secs/3600:6.2f}")
    hours = total_sec / 3600.0
    print("-" * 86)
    print(f" classes tested: {len(per_class)}   audio: {hours:.2f} h   "
          f"raw false alarms: {raw_fa}  ({raw_fa/max(hours,1e-9):.2f} FA/hour, 1 s stride)")


def sec4_speech_fa(predict, extractor, threshold, num_files=120, agg="maxhold"):
    print("\n" + "=" * 86)
    print(" 4. CONTINUOUS HUMAN SPEECH FALSE ALARMS (industrial debouncer)")
    print("=" * 86)
    flacs = []
    for root, _, files in os.walk("speech_corpus/LibriSpeech"):
        for f in files:
            if f.endswith(".flac"):
                flacs.append(os.path.join(root, f))
    if not flacs:
        print(" [warn] no LibriSpeech found")
        return
    rng = np.random.default_rng(0)
    sel = rng.choice(len(flacs), size=min(num_files, len(flacs)), replace=False)
    det = StreamingDetector(predict, extractor, threshold, agg=agg)
    total = 0.0
    hits = 0
    raw = 0
    t0 = time.time()
    for i in sel:
        try:
            d = to_16k(*sf.read(flacs[i]))
        except Exception:
            continue
        total += len(d) / SAMPLE_RATE
        dets, _, r = det.run(d)
        hits += len(dets)
        raw += r
    hrs = total / 3600.0
    el = time.time() - t0
    print(f" audio            : {total:.1f}s ({hrs:.3f} h) from {len(sel)} recordings")
    print(f" raw frame FAs    : {raw}  ({raw/max(hrs,1e-9):.2f} / hour)")
    print(f" debounced triggers: {hits}  ({hits/max(hrs,1e-9):.2f} / hour)")
    print(f" speed            : {el:.1f}s  ({total/max(el,1e-9):.1f}x real time)")


def sec5_streaming_recall(predict, extractor, threshold, keyword_bases, n_trials=300, agg="maxhold"):
    print("\n" + "=" * 86)
    print(f" 5. STREAMING RECALL (keyword spliced into real noise + speech beds, agg={agg})")
    print("=" * 86)
    if not keyword_bases:
        print(" [warn] no keyword bases available; skipping")
        return
    noise_rows = load_index()
    noise_paths = [p for p, _ in noise_rows]
    rng = np.random.default_rng(11)
    det = StreamingDetector(predict, extractor, threshold, agg=agg)

    snr_points = [20.0, 12.0, 6.0, 0.0, -3.0]
    results = {s: [0, 0, []] for s in snr_points}
    for _ in range(n_trials):
        try:
            nz_path = noise_paths[int(rng.integers(0, len(noise_paths)))]
            nz = to_16k(*sf.read(nz_path))
        except Exception:
            continue
        kw = keyword_bases[int(rng.integers(0, len(keyword_bases)))]
        kw = np.asarray(kw, dtype=np.float32).ravel()
        # Keep the whole utterance: truncating the keyword destroys the phonetic
        # pattern the detector is supposed to key on.
        if len(kw) > 11000:
            kw = kw[:11000]
        if len(kw) < 1500:
            continue
        bed_n = 16000 + 7000 + int(rng.integers(0, 3000))
        reps = int(np.ceil(bed_n / max(1, len(nz))))
        bed = np.tile(nz, reps)[:bed_n]
        start = int(rng.integers(1200, 3200))
        snr = float(snr_points[int(rng.integers(0, len(snr_points)))])
        # Scale the NOISE to the requested SNR against the keyword. Scaling the whole
        # mixture would be a no-op: SNR is scale invariant.
        sp = np.mean(kw ** 2) + 1e-9
        npow = np.mean(bed ** 2) + 1e-9
        bed = (bed * np.sqrt(sp / (10 ** (snr / 10.0) * npow))).astype(np.float32)
        bed[start:start + len(kw)] += kw
        peak = float(np.max(np.abs(bed)))
        if peak > 0.95:
            bed = (bed / peak * 0.95).astype(np.float32)
        dets, first, _ = det.run(bed)
        key = snr_points[int(np.argmin([abs(s - snr) for s in snr_points]))]
        res = results[key]
        res[1] += 1
        if first is not None and first <= start + len(kw) + 3200:
            res[0] += 1
            # Latency measured from the end of the spoken keyword to the trigger.
            res[2].append((first - (start + len(kw))) / SAMPLE_RATE)
    print(f"{'SNR (dB)':>9} | {'detected':>9} | {'recall':>7} | {'median lat':>11} | {'p90 lat':>8}")
    tot_hit = 0
    tot_n = 0
    for s in snr_points:
        hit, n, lats = results[s]
        if n == 0:
            continue
        tot_hit += hit
        tot_n += n
        med = np.median(lats) * 1000 if lats else float("nan")
        p90 = np.percentile(lats, 90) * 1000 if lats else float("nan")
        print(f"{s:9.1f} | {hit:4d}/{n:<4d} | {hit/n*100:6.2f}% | {med:9.0f}ms | {p90:6.0f}ms")
    print("-" * 86)
    if tot_n:
        print(f" overall streaming recall: {tot_hit}/{tot_n} = {tot_hit/tot_n*100:.2f}%")


def load_keyword_bases(path="keyword_bases_eval.npy", limit=400):
    """Held-out clean keyword waveforms for the streaming-recall test."""
    for cand in (path, "keyword_bases_eval.npy", "keyword_bases.npy"):
        if os.path.exists(cand):
            arr = np.load(cand, allow_pickle=True)
            bases = [np.asarray(x, dtype=np.float32).ravel() for x in list(arr)]
            bases = [b for b in bases if 1500 <= len(b) <= 13000]
            print(f"[eval] keyword bases: {cand} ({len(bases)} usable)")
            return bases[:limit]
    print("[eval] no keyword base cache found; streaming recall will be skipped")
    return []


# ==============================================================================
# main
# ==============================================================================

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", type=str, default="best_v2_50k.flax")
    ap.add_argument("--arch", type=str, default="bcconformer_50k")
    ap.add_argument("--data", type=str, default="dataset_v2.npz")
    ap.add_argument("--index", type=str, default="noise_bank_v2/index.csv")
    ap.add_argument("--threshold", type=float, default=-1.0, help="-1 = use value stored in checkpoint")
    ap.add_argument("--agg", type=str, default="maxhold", choices=["ema", "maxhold"])
    ap.add_argument("--speech_files", type=int, default=120)
    ap.add_argument("--recall_trials", type=int, default=300)
    args = ap.parse_args()

    predict, ck = load_predict(args.model, args.arch)
    extractor = AudioFeatureExtractor()
    thr = args.threshold if args.threshold >= 0 else float(ck.get("threshold", 0.85))

    print("#" * 86)
    print(f" INDUSTRIAL BENCHMARK v2  |  keyword = 'amaze'")
    print(f" model      : {args.model}  [{args.arch}]  {ck.get('param_count', 0):,} params")
    print(f" checkpoint : epoch {ck.get('epoch','?')}  stored threshold {thr:.4f}  "
          f"confirmation: {args.agg} 2-of-2 + 1.5 s refractory")
    if "tpr_at_budget" in ck:
        print(f" stored     : TPR@budget {ck.get('tpr_at_budget',0)*100:.2f}%  "
              f"offset-stress {ck.get('tpr_offset_stress',0)*100:.2f}%  "
              f"TPR@FPR0.12% {ck.get('tpr_at_0012',0)*100:.2f}%")
    print("#" * 86)

    sec1_roc(predict, args.data)
    sec1b_buckets(predict, args.data, thr)
    sec1c_level(predict, args.data, thr)
    sec2_offset(predict, args.data, extractor)
    sec3_soundscapes(predict, extractor, thr, args.index)
    sec4_speech_fa(predict, extractor, thr, num_files=args.speech_files, agg=args.agg)
    sec5_streaming_recall(predict, extractor, thr, load_keyword_bases(),
                          n_trials=args.recall_trials, agg=args.agg)


## Threshold sweep primitives

Pre-computes every soundscape frame confidence once so thresholds are applied to measured values rather than re-derived.

*Inlined from `sweep_threshold.py`: 213 lines, 3 CLI lines and 2 cross-module imports removed — everything shares one namespace here.*


In [14]:
"""
Operating-Point Sweep for the Streaming Detector
------------------------------------------------
Picks the deployment threshold from measured behaviour rather than a frame-level FPR
proxy. For each candidate threshold it reports

  * soundscape false alarms per hour over noise_bank_v2
  * debounced continuous-speech false alarms per hour over LibriSpeech
  * streaming recall with a held-out keyword spliced into real noise beds

The right threshold is the lowest one that keeps total streaming false alarms under
the budget, because every point of recall is worth more than a threshold that is
tighter than the hardware ever required.

    python sweep_threshold.py --model best_v2_50k.flax --arch bcconformer_50k \
        --thresholds 0.60 0.70 0.75 0.80 0.8246 0.85
"""

import argparse
import os
import time
from collections import defaultdict

import numpy as np
import soundfile as sf


SAMPLE_RATE = 16000
WIN = 16000
HOP = 800  # 0.5 s stride for the FA scan (denser than deployment, so FA/hr is an upper bound)


def soundscape_scan(predict, extractor, rows, stride_s=1.0):
    """Pre-computes every soundscape frame confidence once; thresholds are applied later."""
    per_class = {}
    step = int(stride_s * SAMPLE_RATE)
    for path, cls in rows:
        try:
            d = to_16k(*sf.read(path))
        except Exception:
            continue
        n = max(0, (len(d) - WIN) // step + 1)
        if n == 0:
            continue
        arr = np.empty((n, 49, 40), dtype=np.float32)
        keep = np.zeros(n, dtype=bool)
        for s in range(n):
            chunk = d[s * step:s * step + WIN]
            rms = float(np.sqrt(np.mean(chunk ** 2)))
            if rms < 0.002:
                continue
            arr[s] = extractor.compute_spectrogram(chunk)
            keep[s] = True
        if not keep.any():
            continue
        probs = batch_score(predict, arr[keep][:, :, :])
        per_class.setdefault(cls, []).append(probs)
    merged = {c: np.concatenate(v) for c, v in per_class.items()}
    return merged


def speech_stream_scan(predict, extractor, threshold, num_files, **det_kw):
    det = StreamingDetector(predict, extractor, threshold, **det_kw)
    flacs = []
    for root, _, files in os.walk("speech_corpus/LibriSpeech"):
        for f in files:
            if f.endswith(".flac"):
                flacs.append(os.path.join(root, f))
    rng = np.random.default_rng(0)
    sel = rng.choice(len(flacs), size=min(num_files, len(flacs)), replace=False)
    total = 0.0
    hits = 0
    for i in sel:
        try:
            d = to_16k(*sf.read(flacs[i]))
        except Exception:
            continue
        total += len(d) / SAMPLE_RATE
        dets, _, _ = det.run(d)
        hits += len(dets)
    return total / 3600.0, hits


def recall_scan(predict, extractor, threshold, bases, rows, trials, snr_points, seed=5, **det_kw):
    rng = np.random.default_rng(seed)
    paths = [p for p, _ in rows]
    det = StreamingDetector(predict, extractor, threshold, **det_kw)
    hits = defaultdict(int)
    counts = defaultdict(int)
    lats = defaultdict(list)
    for _ in range(trials):
        try:
            nz = to_16k(*sf.read(paths[int(rng.integers(0, len(paths)))]))
        except Exception:
            continue
        kw = np.asarray(bases[int(rng.integers(0, len(bases)))], dtype=np.float32).ravel()
        if len(kw) > 11000:
            kw = kw[:11000]
        if len(kw) < 1500:
            continue
        bed_n = WIN + 7000 + int(rng.integers(0, 3000))
        bed = np.tile(nz, int(np.ceil(bed_n / max(1, len(nz)))))[:bed_n]
        start = int(rng.integers(1200, 3200))
        snr = float(snr_points[int(rng.integers(0, len(snr_points)))])
        sp = np.mean(kw ** 2) + 1e-9
        npow = np.mean(bed ** 2) + 1e-9
        bed = (bed * np.sqrt(sp / (10 ** (snr / 10.0) * npow))).astype(np.float32)
        bed[start:start + len(kw)] += kw
        pk = float(np.max(np.abs(bed)))
        if pk > 0.95:
            bed = (bed / pk * 0.95).astype(np.float32)
        dets, first, _ = det.run(bed)
        counts[snr] += 1
        if first is not None and first <= start + len(kw) + 3200:
            hits[snr] += 1
            lats[snr].append((first - start) / SAMPLE_RATE)
    return hits, counts, lats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", type=str, default="best_v3_50k.flax")
    ap.add_argument("--arch", type=str, default="bcconformer_50k")
    ap.add_argument("--thresholds", type=float, nargs="+", default=[0.60, 0.70, 0.78, 0.84, 0.90])
    ap.add_argument("--policies", type=str, nargs="+", default=["2/2"],
                    help="need/span confirmation policies, e.g. 2/2 2/3 3/4 1/2")
    ap.add_argument("--aggs", type=str, nargs="+", default=["ema", "maxhold"],
                    help="frame-score aggregation: ema or maxhold")
    ap.add_argument("--refractory", type=int, default=15)
    ap.add_argument("--index", type=str, default="noise_bank_v2/index.csv")
    ap.add_argument("--speech_files", type=int, default=80)
    ap.add_argument("--recall_trials", type=int, default=250)
    ap.add_argument("--snr", type=float, nargs="+", default=[20.0, 12.0, 6.0, 0.0])
    ap.add_argument("--fa_budget", type=float, default=1.0, help="max total FA per hour")
    args = ap.parse_args()

    predict, ck = load_predict(args.model, args.arch)
    extractor = AudioFeatureExtractor()
    rows = load_index(args.index)
    bases = load_keyword_bases()

    print("=" * 108)
    print(f" THRESHOLD x POLICY SWEEP  |  {args.model}  [{args.arch}]  "
          f"| refractory {args.refractory*0.1:.1f}s  | FA budget {args.fa_budget}/hour")
    print("=" * 108)
    print(" scanning soundscapes once ...", flush=True)
    t0 = time.time()
    noise = soundscape_scan(predict, extractor, rows)
    noise_secs = 0.0
    for path, _ in rows:
        try:
            noise_secs += len(to_16k(*sf.read(path))) / SAMPLE_RATE
        except Exception:
            pass
    noise_hours = noise_secs / 3600.0
    print(f" {len(noise)} classes, {noise_hours:.2f} h scored in {time.time()-t0:.0f}s", flush=True)

    print(" scanning continuous speech ...", flush=True)
    speech_cache = []
    flacs = []
    for root, _, files in os.walk("speech_corpus/LibriSpeech"):
        for f in files:
            if f.endswith(".flac"):
                flacs.append(os.path.join(root, f))
    rng = np.random.default_rng(0)
    sel = rng.choice(len(flacs), size=min(args.speech_files, len(flacs)), replace=False)
    for i in sel:
        try:
            speech_cache.append(to_16k(*sf.read(flacs[i])))
        except Exception:
            pass
    speech_hours = sum(len(d) for d in speech_cache) / SAMPLE_RATE / 3600.0

    print("=" * 108)
    print(f"{'agg policy':>13} | {'thresh':>7} | {'soundscFA/h':>11} | {'speechFA/h':>9} | "
          f"{'totFA/h':>8} | {'rec@20':>7} | {'rec@12':>7} | {'rec@6':>7} | {'rec@0':>7} | {'mean':>7}")
    print("-" * 108)
    best = None
    results = []
    for agg in args.aggs:
      for pol in args.policies:
        need, span = (int(x) for x in pol.split("/"))
        for t in args.thresholds:
            det_kw = dict(need=need, span=span, refractory_steps=args.refractory, agg=agg)
            det = StreamingDetector(predict, extractor, t, **det_kw)
            hits = 0
            for d in speech_cache:
                dets, _, _ = det.run(d)
                hits += len(dets)
            speech_fa = hits / max(1e-9, speech_hours)
            noise_fa = sum(int(np.sum(p >= t)) for p in noise.values()) / max(1e-9, noise_hours)
            total_fa = noise_fa + speech_fa
            rh, rc, _ = recall_scan(predict, extractor, t, bases, rows,
                                    args.recall_trials, args.snr, **det_kw)
            def r_of(s):
                return rh.get(s, 0) / max(1, rc.get(s, 0)) * 100
            mean_r = sum(rh.get(s, 0) for s in args.snr) / max(1, sum(rc.values())) * 100
            flag = "" if total_fa > args.fa_budget else "  *"
            print(f"{agg+' '+pol:>13} | {t:7.4f} | {noise_fa:11.2f} | {speech_fa:9.2f} | {total_fa:8.2f} | "
                  f"{r_of(20.0):6.1f}% | {r_of(12.0):6.1f}% | {r_of(6.0):6.1f}% | "
                  f"{r_of(0.0):6.1f}% | {mean_r:6.1f}%{flag}", flush=True)
            results.append((agg, pol, t, total_fa, mean_r))
            if total_fa <= args.fa_budget:
                if best is None or mean_r > best[4]:
                    best = (agg, pol, t, total_fa, mean_r)
    print("-" * 108)
    if best:
        print(f" RECOMMENDED: threshold {best[2]:.4f}, agg={best[0]}, policy {best[1]} "
              f"(streaming recall {best[4]:.1f}%, total FA {best[3]:.2f}/hour)")
    else:
        print(" No (threshold, policy) met the FA budget.")
    print("=" * 108)


## Operating-point freeze

Writes the chosen threshold, aggregation, confirmation policy and refractory into the checkpoint.

*Inlined from `finalize_checkpoint.py`: 118 lines, 3 CLI lines and 3 cross-module imports removed — everything shares one namespace here.*


In [15]:
"""
Freeze a deployment operating point into a checkpoint
-----------------------------------------------------
The trainer stores the threshold that meets the frame-level FPR budget on the
validation split. That is a useful default, but the number you actually ship is the
one that meets the *streaming* false-alarm budget, which sweep_threshold.py measures
against real soundscape hours and continuous human speech.

This script re-reads a checkpoint, runs the streaming sweep, and writes the chosen
(threshold, aggregation, confirmation policy, refractory) into the checkpoint so
StreamingKWSEngine picks it up with no arguments.

    python finalize_checkpoint.py --model best_v3_50k.flax --arch bcconformer_50k \
        --data dataset_v3.npz --fa_budget 1.0
"""

import argparse
import os
import time

import numpy as np
import soundfile as sf
from flax import serialization


SAMPLE_RATE = 16000


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", type=str, default="best_v3_50k.flax")
    ap.add_argument("--arch", type=str, default="bcconformer_50k")
    ap.add_argument("--index", type=str, default="noise_bank_v2/index.csv")
    ap.add_argument("--thresholds", type=float, nargs="+",
                    default=[0.72, 0.78, 0.82, 0.84, 0.86, 0.90, 0.94])
    ap.add_argument("--aggs", type=str, nargs="+", default=["maxhold"])
    ap.add_argument("--policy", type=str, default="2/2")
    ap.add_argument("--refractory", type=int, default=15)
    ap.add_argument("--speech_files", type=int, default=150)
    ap.add_argument("--recall_trials", type=int, default=300)
    ap.add_argument("--snr", type=float, nargs="+", default=[20.0, 12.0, 6.0, 0.0])
    ap.add_argument("--fa_budget", type=float, default=1.0)
    ap.add_argument("--dry_run", action="store_true")
    args = ap.parse_args()

    predict, ck = load_predict(args.model, args.arch)
    extractor = AudioFeatureExtractor()
    rows = load_index(args.index)
    bases = load_keyword_bases()
    need, span = (int(x) for x in args.policy.split("/"))

    print("scanning soundscapes ...", flush=True)
    noise = soundscape_scan(predict, extractor, rows)
    noise_hours = sum(len(to_16k(*sf.read(p))) for p, _ in rows) / SAMPLE_RATE / 3600.0

    flacs = []
    for root, _, files in os.walk("speech_corpus/LibriSpeech"):
        for f in files:
            if f.endswith(".flac"):
                flacs.append(os.path.join(root, f))
    rng = np.random.default_rng(3)
    sel = rng.choice(len(flacs), size=min(args.speech_files, len(flacs)), replace=False)
    cache = []
    for i in sel:
        try:
            cache.append(to_16k(*sf.read(flacs[i])))
        except Exception:
            pass
    speech_hours = sum(len(d) for d in cache) / SAMPLE_RATE / 3600.0
    print(f"soundscapes {noise_hours:.2f} h | speech {speech_hours:.3f} h", flush=True)

    print(f"\n{'agg':>8} | {'thresh':>7} | {'soundscFA/h':>11} | {'speechFA/h':>10} | "
          f"{'totalFA/h':>9} | {'recall':>7}")
    print("-" * 68)
    best = None
    for agg in args.aggs:
        for t in args.thresholds:
            kw = dict(need=need, span=span, refractory_steps=args.refractory, agg=agg)
            det = StreamingDetector(predict, extractor, t, **kw)
            hits = 0
            for a in cache:
                dets, _, _ = det.run(a)
                hits += len(dets)
            speech_fa = hits / max(1e-9, speech_hours)
            noise_fa = sum(int(np.sum(p >= t)) for p in noise.values()) / max(1e-9, noise_hours)
            total = noise_fa + speech_fa
            rh, rc, _ = recall_scan(predict, extractor, t, bases, rows,
                                    args.recall_trials, args.snr, **kw)
            recall = sum(rh.values()) / max(1, sum(rc.values())) * 100
            ok = " *" if total <= args.fa_budget else ""
            print(f"{agg:>8} | {t:7.4f} | {noise_fa:11.2f} | {speech_fa:10.2f} | {total:9.2f} | "
                  f"{recall:6.1f}%{ok}", flush=True)
            if total <= args.fa_budget and (best is None or recall > best[2]):
                best = (agg, t, recall, total)
    print("-" * 68)
    if not best:
        print("No operating point met the FA budget; nothing written.")
        return
    agg, t, recall, total = best
    print(f" CHOSEN: threshold {t:.4f}, agg={agg}, policy {args.policy}, "
          f"refractory {args.refractory*0.1:.1f}s")
    print(f"         measured streaming recall {recall:.1f}%, total FA {total:.2f}/hour")
    if args.dry_run:
        return

    out = dict(ck)
    out["threshold"] = float(t)
    out["deploy_threshold"] = float(t)
    out["deploy_agg"] = agg
    out["deploy_need"] = need
    out["deploy_span"] = span
    out["deploy_refractory_steps"] = args.refractory
    out["deploy_recall"] = float(recall)
    out["deploy_fa_per_hour"] = float(total)
    with open(args.model, "wb") as f:
        f.write(serialization.to_bytes(out))
    print(f" Written into {args.model} ({os.path.getsize(args.model)/1024:.1f} KB)")


## Deployment engine

Sliding 1 s window, 100 ms stride, peak-hold confirmation, reads the frozen operating point out of the checkpoint.

*Inlined from `kws_engine.py`: 184 lines, 8 CLI lines and 2 cross-module imports removed — everything shares one namespace here.*


In [16]:
"""
Streaming Keyword Spotting (KWS) Engine in Python
-------------------------------------------------
Simulates real-time on-device keyword spotting on audio streams:
- Sliding 1.0s audio buffer (16,000 samples @ 16kHz) with 100ms stride
- On-the-fly 40-band Log-Mel feature extraction
- Fast JAX inference with pre-loaded weights
- 2-step moving average confidence debouncer (suppresses spurious false alarms)
- Real-time latency tracking
"""

import os
import time
import numpy as np
import jax
import jax.numpy as jnp
from flax import serialization


class StreamingKWSEngine:
    def __init__(self, checkpoint_path: str = "best_kws_model.flax", arch: str = "bcresnet",
                 confidence_threshold: float = None, ema_alpha: float = 0.60,
                 consecutive_required: int = 2, refractory_steps: int = 15,
                 aggregation: str = "maxhold", hold_decay: float = 0.85):
        """confidence_threshold=None -> use the operating point stored in the
        checkpoint (v2+ trainers store the threshold that meets the FPR budget).

        aggregation='maxhold' is the default on purpose. An EMA with alpha=0.6 needs
        roughly five consecutive high frames to cross the threshold, but a 0.6 s
        wake word only produces two or three good frames in a 1 s sliding window, so
        an EMA-confirmed detector discards most real detections. A decaying peak hold
        still requires `consecutive_required` frames above threshold, so isolated
        spikes are rejected just as well, but it confirms on a realistic burst.
        """
        self.extractor = AudioFeatureExtractor()
        self.arch = arch
        self.ema_alpha = ema_alpha
        self.consecutive_required = consecutive_required
        self.refractory_steps = refractory_steps
        self.aggregation = aggregation
        self.hold_decay = hold_decay

        # Reconstruct architecture template
        self.model = get_model(self.arch, num_classes=2)
        rng = jax.random.PRNGKey(0)
        dummy_input = jnp.ones((1, 49, 40, 1), dtype=jnp.float32)
        variables = self.model.init(rng, dummy_input, train=False)
        dummy_params = variables['params']

        if os.path.exists(checkpoint_path):
            with open(checkpoint_path, "rb") as f:
                encoded = f.read()

            template = {
                "params": dummy_params,
                "arch": arch,
                "param_count": 0,
                "val_acc": 0.0,
                "tpr": 0.0,
                "fpr": 0.0,
                "threshold": 0.85
            }
            # v1 checkpoints carry a fixed key set; v2+ add their own fields, so try
            # the fixed template first and fall back to a free-form restore.
            try:
                state_dict = serialization.from_bytes(template, encoded)
            except ValueError:
                state_dict = serialization.msgpack_restore(encoded)
            self.params = state_dict["params"]
            self.val_acc = state_dict.get("val_acc", 1.0)
            stored_thr = state_dict.get("threshold", 0.85)
            self.confidence_threshold = float(
                stored_thr if confidence_threshold is None else confidence_threshold)
            if aggregation == "maxhold" and "deploy_agg" in state_dict:
                aggregation = state_dict["deploy_agg"]
            if "deploy_refractory_steps" in state_dict:
                self.refractory_steps = int(state_dict["deploy_refractory_steps"])
            if "deploy_need" in state_dict:
                self.consecutive_required = int(state_dict["deploy_need"])
            self.aggregation = aggregation
            self.param_count = sum(x.size for x in jax.tree_util.tree_leaves(self.params))

            print("=====================================================================")
            print(f" Loaded KWS Engine: {self.arch.upper()} from {checkpoint_path}")
            print(f" Model Parameters : {self.param_count:,} (INT8 Footprint: ~{self.param_count / 1024:.2f} KB)")
            print(f" Detection Thresh : {self.confidence_threshold:.4f}"
                  f"{'  (from checkpoint)' if confidence_threshold is None else '  (override)'}")
            print(f" Confirmation     : {self.aggregation} {self.consecutive_required} hits, "
                  f"{self.refractory_steps * 0.1:.1f}s refractory")
            if "tpr_offset_stress" in state_dict:
                print(f" Stored TPR       : {state_dict['tpr_offset_stress'] * 100:.2f}% "
                      f"@ FPR <= {state_dict.get('fpr_budget', 0) * 100:.2f}%")
            if "deploy_recall" in state_dict:
                print(f" Stored streaming : recall {state_dict['deploy_recall']:.1f}%  "
                      f"FA {state_dict['deploy_fa_per_hour']:.2f}/hour")
            print("=====================================================================")
        else:
            print(f"[Warning] Checkpoint '{checkpoint_path}' not found. Using initialized weights.")
            self.params = dummy_params
            self.param_count = count_params(variables)
            self.confidence_threshold = 0.85 if confidence_threshold is None else confidence_threshold

        # JIT-compile the inference function
        @jax.jit
        def predict_fn(params, x):
            logits = self.model.apply({'params': params}, x, train=False)
            probs = jax.nn.softmax(logits, axis=-1)
            return probs[0, 1]  # Return target keyword probability

        self.predict_fn = predict_fn

        # Warmup JIT compilation
        _ = self.predict_fn(self.params, dummy_input)

        # Audio sliding window (1.0 second = 16,000 samples)
        self.buffer_len = 16000
        self.audio_buffer = np.zeros(self.buffer_len, dtype=np.float32)

        # Debouncing filter state
        self.ema = 0.0
        self.hold = 0.0
        self.consecutive_hits = 0
        self.lockout = 0

    def _aggregate(self, raw_prob: float) -> float:
        if self.aggregation == "maxhold":
            self.hold = max(raw_prob, self.hold * self.hold_decay)
            return self.hold
        self.ema = self.ema_alpha * raw_prob + (1.0 - self.ema_alpha) * self.ema
        return self.ema

    def push_audio_chunk(self, chunk: np.ndarray) -> tuple[bool, float, float]:
        """
        Feeds an incoming audio chunk (e.g. 50ms = 800 samples or 100ms = 1600 samples).
        Returns:
            (is_keyword_spotted, aggregated_confidence, inference_latency_ms)
        """
        chunk = np.array(chunk, dtype=np.float32)
        if np.max(np.abs(chunk)) > 1.0:
            chunk = chunk / 32768.0

        n = len(chunk)
        # Shift buffer left by n, append new chunk
        self.audio_buffer = np.roll(self.audio_buffer, -n)
        self.audio_buffer[-n:] = chunk

        # Feature extraction
        t0 = time.perf_counter()
        spec = self.extractor.compute_spectrogram(self.audio_buffer)
        tensor = jnp.expand_dims(jnp.expand_dims(jnp.array(spec), axis=0), axis=-1)

        # Model forward inference
        raw_prob = float(self.predict_fn(self.params, tensor))
        t1 = time.perf_counter()
        latency_ms = (t1 - t0) * 1000.0

        score = self._aggregate(raw_prob)

        if self.lockout > 0:
            self.lockout -= 1

        # Confirmation: N consecutive windows above threshold, then refractory lockout
        keyword_detected = False
        if score >= self.confidence_threshold:
            self.consecutive_hits += 1
            if self.consecutive_hits >= self.consecutive_required and self.lockout == 0:
                keyword_detected = True
                self.consecutive_hits = 0
                self.lockout = self.refractory_steps
                self.ema = 0.0
                self.hold = 0.0
        else:
            self.consecutive_hits = 0

        return keyword_detected, score, latency_ms

    def reset(self):
        """Clears audio buffer and detection state."""
        self.audio_buffer.fill(0)
        self.ema = 0.0
        self.hold = 0.0
        self.consecutive_hits = 0
        self.lockout = 0



## Stage 1a — raw sources

Nothing is uploaded by hand. ESC-50 comes from GitHub, UrbanSound8K from HuggingFace,
LibriSpeech from OpenSLR, and the TTS waveforms are synthesised with `edge-tts`. The
noise bank is only re-downloaded when the index is genuinely absent, because re-fetching
ESC-50 costs ~12 minutes and produces identical bytes.


In [17]:

# ==============================================================================
# Stage 1a -- raw sources, all from public URLs. Nothing is uploaded by hand.
# ==============================================================================
os.chdir(WORK)

def read_noise_index(path):
    if not os.path.exists(path):
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

print("resolving LibriSpeech dev-clean from OpenSLR ...")
librispeech = build_librispeech(os.path.join(WORK, "speech_corpus"))

index = os.path.join(WORK, "noise_bank_v2", "index.csv")
rows = read_noise_index(index)
have_us8k = any("us8k" in r.get("class", "") for r in rows)
print(f"noise bank: {len(rows)} clips indexed, urbansound={have_us8k}")

if not rows or not have_us8k:
    # ESC-50 is only re-downloaded when the index is genuinely absent: re-fetching it
    # costs ~12 minutes and produces identical bytes.
    build_esc50(max_files=2000)
    build_urbansound(shard_limit=2, per_class_cap=220)
    build_babble(per_config=10)
    build_procedural(per_name=6)
    write_index()
else:
    print("noise bank reused")

rows = read_noise_index(index)
classes = len({r["class"] for r in rows})
print(f"\nnoise bank: {len(rows)} clips / {classes} classes")
print("librispeech:", librispeech)
assert classes >= 50, f"noise bank only has {classes} classes"


resolving LibriSpeech dev-clean from OpenSLR ...
[librispeech] already present: 2,800 flac files under /kaggle/working/speech_corpus/LibriSpeech
noise bank: 3269 clips indexed, urbansound=True
noise bank reused

noise bank: 3269 clips / 84 classes
librispeech: 2800



## Stage 1b — build the corpus, or reuse a published one

The build fans out over processes. A fork-based pool needs `__main__` to be picklable by
reference, which a notebook kernel does not always guarantee, so it falls back to serial
rather than failing.

Two axes, explicit and identical for positives and negatives, so mixture layer count can
never leak the label: **recipe** (0 / 1 / 2 / 2-hard noise layers) × **content** (word,
unimportant word, real speech, live babble, soundscape, near-miss word).


In [20]:

# ==============================================================================
# Stage 1b -- build the corpus, or reuse a published one whose build key matches.
# ==============================================================================
def build_corpus_bundled(out_dir, cfg, workers=4, verbose=True):
    """Runs the bucketed builder over multiple processes and writes the npz."""
    # Same rule as build_corpus_v2.main(): only the bucketed profiles tag samples
    # with mixture buckets, and only those need the filler pool synthesised.
    use_buckets = cfg["profile"] in ("v4", "v5")
    _PROFILE.clear()
    _PROFILE.update(PROFILES[cfg["profile"]])

    pools = load_pool_cache(os.path.join(WORK, "cache"), want_filler=use_buckets)
    if pools is None:
        pools = run_coroutine_sync(build_tts_pools(per_voice=cfg["per_voice"], want_filler=use_buckets))
        save_pool_cache(pools, os.path.join(WORK, "cache"))
    kw = [(t, trim_silence(w)) for t, w in pools["kw"]]
    conf = [(t, trim_silence(w)) for t, w in pools.get("conf", [])]
    filler = [(t, trim_silence(w)) for t, w in pools.get("filler", [])]
    kw = [x for x in kw if x[1] is not None and len(x[1]) > 600]
    conf = [x for x in conf if x[1] is not None and len(x[1]) > 600]
    filler = [x for x in filler if x[1] is not None and len(x[1]) > 400]
    if verbose:
        print(f"[build] bases: keyword {len(kw)}  confusable {len(conf)}  filler {len(filler)}")

    nb = NoiseBank(index_csv=os.path.join(WORK, "noise_bank_v2", "index.csv"))
    sb = SpeechBank()

    total = cfg["pos"] + cfg["neg"]
    n_chunks = max(1, workers * 4)
    per_chunk = int(np.ceil(total / n_chunks))
    jobs, assigned = [], 0
    for c in range(n_chunks):
        take = min(per_chunk, total - assigned)
        if take <= 0:
            break
        pos_share = int(round(take * cfg["pos"] / total))
        jobs.append((assigned, pos_share, take - pos_share, cfg["seed"] + c * 7919,
                     use_buckets))
        assigned += take

    init_args = (nb.paths, sb.flacs, sb.cmds, kw, conf, cfg["seed"], cfg["profile"], filler)
    results = []
    try:
        from concurrent.futures import ProcessPoolExecutor
        with ProcessPoolExecutor(max_workers=workers, initializer=_init_worker,
                                 initargs=init_args) as ex:
            futs = [ex.submit(_run_chunk, j) for j in jobs]
            for k, f in enumerate(futs):
                results.append(f.result())
                if verbose and (k + 1) % 8 == 0:
                    print(f"  chunk {k+1}/{len(jobs)}", flush=True)
    except Exception as e:
        # Fork-based pools need __main__ picklable by reference, which a notebook
        # kernel does not always guarantee. Serial is slower but always correct.
        print(f"[build] parallel build failed ({str(e)[:120]}), falling back to serial")
        results = []
        _init_worker(*init_args)
        for j in jobs:
            results.append(_run_chunk(j))

    X = np.concatenate([r[0] for r in results], axis=0)
    y = np.concatenate([r[1] for r in results], axis=0)
    g = np.concatenate([r[2] for r in results], axis=0)
    s = np.concatenate([r[3] for r in results], axis=0)
    b = np.concatenate([r[4] for r in results], axis=0)
    l = np.concatenate([r[5] for r in results], axis=0)
    if X.ndim == 3:
        X = np.expand_dims(X, -1)

    tr, va = stratified_split(g, y, 0.2, seed=cfg["seed"], buckets=b)
    raw = os.path.join(out_dir, "corpus_fp32.npz")
    np.savez_compressed(raw, X_train=X[tr], y_train=y[tr], g_train=g[tr], snr_train=s[tr],
                        b_train=b[tr], lvl_train=l[tr],
                        X_val=X[va], y_val=y[va], g_val=g[va], snr_val=s[va],
                        b_val=b[va], lvl_val=l[va])
    del X
    return raw, len(tr), len(va)


def compact_to_fp16(src_npz, dst_npz):
    """The log-mel front end floors at log(1e-5) = -11.51 and rarely exceeds +5, so
    float16 resolution (~1e-3) is far below anything the model can resolve, and it
    halves the download."""
    d = np.load(src_npz, allow_pickle=True)
    out = {k: (d[k].astype(np.float16) if k.startswith("X_") and d[k].dtype == np.float32
               else d[k]) for k in d.files}
    d.close()
    np.savez_compressed(dst_npz, **out)
    before, after = os.path.getsize(src_npz), os.path.getsize(dst_npz)
    print(f"  [fp16] {before/1e6:.1f} MB -> {after/1e6:.1f} MB ({after/max(before,1)*100:.0f}%)")
    return dst_npz


BUILD_KEY = compute_build_key(CFG)
print("build key:", BUILD_KEY)
local_bundle = os.path.join(WORK, f"bundle-{BUILD_KEY}")
corpus_fp16 = os.path.join(local_bundle, "corpus.npz")

if not CFG["force_rebuild"] and manifest_of(local_bundle).get("build_key") == BUILD_KEY \
        and os.path.exists(corpus_fp16):
    CORPUS_SOURCE = "local-cache"
    print(f"[ensure] HIT local cache: {local_bundle}")
elif not CFG["force_rebuild"] and kaggle_remote_exists(CFG["corpus_slug"]):
    staged = os.path.join(WORK, "corpus-pull")
    shutil.rmtree(staged, ignore_errors=True)
    if kaggle_pull(CFG["corpus_slug"], staged) and manifest_of(staged).get("build_key") == BUILD_KEY:
        shutil.rmtree(local_bundle, ignore_errors=True)
        shutil.move(staged, local_bundle)
        CORPUS_SOURCE = "kaggle-pull"
        print(f"[ensure] HIT published dataset, cached at {local_bundle}")
    else:
        shutil.rmtree(staged, ignore_errors=True)
        print("[ensure] published bundle does not match the build key -> rebuilding")
        CORPUS_SOURCE = "build"
else:
    CORPUS_SOURCE = "build"

if CORPUS_SOURCE == "build":
    print("[ensure] MISS -> building from scratch")
    shutil.rmtree(local_bundle, ignore_errors=True)
    os.makedirs(local_bundle, exist_ok=True)
    t0 = time.time()
    raw, n_tr, n_va = build_corpus_bundled(local_bundle, CFG, workers=CFG["workers"])
    corpus_fp16 = compact_to_fp16(raw, corpus_fp16)
    os.remove(raw)
    for aux, dest in (("noise_bank_v2/index.csv", "index.csv"),
                      ("keyword_bases_eval.npy", "keyword_bases_eval.npy"),
                      ("keyword_bases.npy", "keyword_bases.npy")):
        src = os.path.join(WORK, aux)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(local_bundle, dest))
    pool = os.path.join(WORK, "cache", "tts_pools.pkl")
    if os.path.exists(pool):
        shutil.copy2(pool, os.path.join(local_bundle, "tts_pools.pkl"))
    man = write_manifest(local_bundle, BUILD_KEY,
                         extra={"build_seconds": round(time.time() - t0, 1),
                                "train": n_tr, "val": n_va})
    print(f"[ensure] built in {time.time()-t0:.0f}s, {man['total_bytes']/1e6:.1f} MB")

CORPUS = corpus_fp16
print("source :", CORPUS_SOURCE)
print("corpus :", CORPUS)
print("bundle :", local_bundle)


build key: 3171611c0ed043c2
[ensure] HIT local cache: /kaggle/working/bundle-3171611c0ed043c2
source : local-cache
corpus : /kaggle/working/bundle-3171611c0ed043c2/corpus.npz
bundle : /kaggle/working/bundle-3171611c0ed043c2



## Stage 1c — verify before training on it

Three failure modes are caught here rather than discovered two hours into a run: a
truncated download, an incomplete bucket taxonomy, and samples that fell through the
INMP441 level floor. The last one matters most — a sample sitting at the log-mel epsilon
carries no information, so it is label noise rather than a hard example.


In [21]:

# ==============================================================================
# Stage 1c -- verify before training on it.
# ==============================================================================
assert verify_bundle(local_bundle), "bundle failed hash verification"

d = np.load(CORPUS, allow_pickle=True)
D = {k: (d[k].astype(np.float32) if d[k].dtype == np.float16 else d[k]) for k in d.files}
for k in sorted(D):
    print(f"  {k:<12} {str(D[k].shape):<18} {D[k].dtype}")
print("train pos/neg:", int((D['y_train']==1).sum()), "/", int((D['y_train']==0).sum()))
print("val   pos/neg:", int((D['y_val']==1).sum()), "/", int((D['y_val']==0).sum()))

b = D['b_train'].astype(str)
lvl = D['lvl_train']
print(f"\n{'bucket':<28} {'n':>8} {'share':>7}   level dBFS")
for name in sorted(np.unique(b), key=lambda s: (s[0] != 'P', s)):
    m = b == name
    print(f"{name:<28} {int(m.sum()):8,} {m.mean()*100:6.2f}%   {lvl[m].min():6.1f}..{lvl[m].max():5.1f}")
print(f"\nbuckets     : {len(np.unique(b))} (expect 24)")
assert len(np.unique(b)) == 24, "bucket taxonomy incomplete"

# Level floor: reported, not gated. The corpus guard lifts a sample that falls through the
# floor back to `level_dbfs` -- but that value comes from the talker geometry, and a 3 m
# whisper is -84.5 dBFS by design, which is itself below the -70 floor. The lift cannot
# help, and the sample stays quiet on purpose. These are legitimate hard cases (a
# whispered keyword at the far end of the room), not dead features: the log-mel floor is
# log(1e-5) = -11.51, and -84.5 dBFS of rms still clears it comfortably.
lvl_bad = lvl < -70.0
print(f"\nlevel dBFS      : {lvl.min():.1f} .. {lvl.max():.1f}  (policy -50..-14)")
print(f"below -70 floor : {int(lvl_bad.sum()):,} of {len(lvl):,} "
      f"({lvl_bad.mean()*100:.2f}%)  worst {lvl.min():.1f} dBFS")
if lvl_bad.any():
    for name in sorted(set(b[lvl_bad])):
        m = (b == name) & lvl_bad
        print(f"    {name:<28} {int(m.sum()):7,}  min {lvl[m].min():6.1f} dBFS")
assert lvl_bad.mean() < 0.05, (
    f"{lvl_bad.mean()*100:.1f}% of samples below the level floor -- that is a corpus "
    "problem, not a policy one")

# A sample pinned at the log-mel epsilon carries no information: that is label noise,
# not a hard example. This is the assertion that must hold -- a quiet sample is fine,
# a dead one is not.
X = D['X_train']
eps = float(np.log(1e-5))
dead = sum(1 for i in range(0, min(len(X), 6000), 97) if np.all(X[i] <= eps + 1e-3))
print(f"dead samples : {dead} (checked {len(range(0, min(len(X), 6000), 97))})")
print(f"log-mel range: {X[:2000].min():.2f} .. {X[:2000].max():.2f}  (floor {eps:.2f})")
assert dead == 0, f"{dead} samples pinned at the log-mel epsilon"
del X


  [verify] ok: 3 files, 823.6 MB, build_key=3171611c0ed043c2
  X_train      (107999, 49, 40, 1) float32
  X_val        (27001, 49, 40, 1) float32
  b_train      (107999,)          <U25
  b_val        (27001,)           <U25
  g_train      (107999,)          <U51
  g_val        (27001,)           <U51
  lvl_train    (107999,)          float32
  lvl_val      (27001,)           float32
  snr_train    (107999,)          float32
  snr_val      (27001,)           float32
  y_train      (107999,)          int32
  y_val        (27001,)           int32
train pos/neg: 39997 / 68002
val   pos/neg: 10000 / 17001

bucket                              n   share   level dBFS
P0_word_direct                  4,038   3.74%    -50.1..-12.7
P1_word_one_noise              16,011  14.83%    -50.2..-12.6
P2_word_two_noise              11,087  10.27%    -50.6..-12.5
P3_word_two_noise_hard          8,861   8.20%    -51.2..-12.8
N0_filler_direct                3,482   3.22%    -50.2..-12.5
N10_babble_two_noise  

In [22]:

# ==============================================================================
# Stage 1d -- materialise float32 for the trainer, and stage the eval assets under
# WORK so the benchmark scripts find them by their documented relative paths.
# ==============================================================================
TRAIN_NPZ = os.path.join(WORK, "corpus_fp32.npz")
if not os.path.exists(TRAIN_NPZ) or os.path.getsize(TRAIN_NPZ) < 10_000_000:
    t0 = time.time()
    np.savez(TRAIN_NPZ, **D)
    print(f"materialised {TRAIN_NPZ} ({os.path.getsize(TRAIN_NPZ)/1e6:.0f} MB, {time.time()-t0:.0f}s)")
else:
    print(f"reusing {TRAIN_NPZ}")

for name, dest in (("index.csv", "noise_bank_v2/index.csv"),
                   ("keyword_bases_eval.npy", "keyword_bases_eval.npy"),
                   ("keyword_bases.npy", "keyword_bases.npy")):
    src = os.path.join(local_bundle, name)
    if not os.path.exists(src):
        continue
    dst = os.path.join(WORK, dest)
    os.makedirs(os.path.dirname(dst) or WORK, exist_ok=True)
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
        shutil.copy2(src, dst)
        print(f"  staged {dest}")
print("done")


materialised /kaggle/working/corpus_fp32.npz (1101 MB, 2s)
done



## Stage 2 — training

The objective is **TPR subject to FPR <= 0.5%**, not accuracy:

* checkpoint selection maximises *offset-stressed* TPR on the full validation ROC, and
  writes the satisfying threshold into the checkpoint
* validation positives are re-scored after a random temporal roll, so the selected model
  is one that works in a 100 ms-stride sliding window rather than on a fixed crop
* class weights follow a sqrt law with a positive bias; the old rule actively penalised
  negatives and was the main reason recall was capped
* on-batch hard-negative mining uses a fixed **count**, so the behaviour does not change
  with the batch size
* focal γ 1.2 rather than 2.0, which down-weights exactly the easy-but-quiet positives
  that dominate the recall loss
* ±12 dB of mic-gain jitter, so the model keys on spectral shape and not absolute level

Per-bucket loss and metrics print every epoch, so a regression in one mixture regime
cannot hide inside an average.


In [23]:

# ==============================================================================
# Stage 2 -- train. Selection maximises offset-stressed TPR subject to FPR <= budget,
# and the satisfying threshold is written into the checkpoint.
# ==============================================================================
TRAIN_KEY = hashlib.sha256(json.dumps({
    "corpus": BUILD_KEY, "arch": CFG["arch"], "epochs": CFG["epochs"],
    "batch": CFG["batch_size"], "lr": CFG["lr"], "fpr": CFG["fpr_budget"],
    "k": CFG["hard_neg_k"], "gamma": CFG["gamma"], "pos_bias": CFG["pos_bias"],
    "spec": CFG["spec_aug_prob"],
}, sort_keys=True).encode()).hexdigest()[:16]

CHECKPOINT = os.path.join(WORK, f"{CFG['arch']}_{TRAIN_KEY}.flax")
print("train key  :", TRAIN_KEY)
print("checkpoint :", CHECKPOINT)
print("model      :", CFG["arch"], f"{count_params(get_model(CFG['arch'], 2)):,} params")
print("data       :", TRAIN_NPZ, f"({len(D['y_train']):,} train / {len(D['y_val']):,} val)")

# Measured ETA rather than a guess: time a few real steps of the real architecture.
def estimate_epoch_seconds(arch, batch_size, steps=5):
    import jax, jax.numpy as jnp
    m = get_model(arch, 2)
    key = jax.random.PRNGKey(0)
    state = create_train_state(key, m, 1e-3, 10)
    bx = jnp.asarray(D["X_train"][:batch_size], jnp.float32)
    by = jnp.asarray(np.arange(batch_size) % 2, jnp.int32)
    cw = jnp.array([1.0, 1.0], jnp.float32)
    kw = dict(gamma=1.2, class_weights=cw, hard_neg_frac=0.75, hard_neg_k=0, bucket_w=None)
    st, loss, _ = train_step(state, bx, by, key, **kw)
    jax.block_until_ready(loss)
    t0 = time.time()
    for _ in range(steps):
        st, loss, _ = train_step(st, bx, by, key, **kw)
    jax.block_until_ready(loss)
    per_step = (time.time() - t0) / steps
    return per_step * (len(D["y_train"]) // batch_size), per_step

try:
    epoch_s, step_s = estimate_epoch_seconds(CFG["arch"], CFG["batch_size"])
    total_h = epoch_s * CFG["epochs"] / 3600.0
    print(f"\nmeasured   : {1000/step_s:.0f} ms/step, batch {CFG['batch_size']}")
    print(f"estimated  : {epoch_s/60:.1f} min/epoch -> {total_h:.1f} h for {CFG['epochs']} epochs")
    if total_h > 2 and not any(d.platform == "gpu" for d in jax.devices()):
        print("             that is a long CPU run. Consider lowering CFG['epochs'].")
except Exception as e:
    print(f"(could not estimate: {str(e)[:120]})")


train key  : 0a1e1b108f683ccd
checkpoint : /kaggle/working/bcconformer_v2_0a1e1b108f683ccd.flax
model      : bcconformer_v2 86,450 params
data       : /kaggle/working/corpus_fp32.npz (107,999 train / 27,001 val)

measured   : 16822 ms/step, batch 512
estimated  : 0.2 min/epoch -> 0.2 h for 60 epochs


In [24]:

NEED_TRAIN = True
if not CFG["force_retrain"] and kaggle_remote_exists(CFG["model_slug"]):
    staged = os.path.join(WORK, "models-pull")
    shutil.rmtree(staged, ignore_errors=True)
    if kaggle_pull(CFG["model_slug"], staged):
        for f in sorted(os.listdir(staged)):
            if not f.endswith(".flax"):
                continue
            with open(os.path.join(staged, f), "rb") as fh:
                cand = serialization.msgpack_restore(fh.read())
            if cand.get("train_key") == TRAIN_KEY:
                shutil.copy2(os.path.join(staged, f), CHECKPOINT)
                NEED_TRAIN = False
                print(f"REUSING {f}: train_key match, epoch {cand.get('epoch')}, "
                      f"stored TPR@budget {cand.get('tpr_at_budget', 0)*100:.2f}%")
                break
if NEED_TRAIN:
    print("no reusable checkpoint for this train key -> training")
    run_training(data_path=TRAIN_NPZ, arch=CFG["arch"], epochs=CFG["epochs"],
                 batch_size=CFG["batch_size"], lr=CFG["lr"], save_path=CHECKPOINT,
                 fpr_budget=CFG["fpr_budget"], hard_neg_k=CFG["hard_neg_k"],
                 gamma=CFG["gamma"], pos_bias=CFG["pos_bias"],
                 spec_aug_prob=CFG["spec_aug_prob"], bucket_weight=True,
                 val_metric_cap=CFG["val_metric_cap"])
assert os.path.exists(CHECKPOINT), "no checkpoint was produced"

with open(CHECKPOINT, "rb") as f:
    ck = serialization.msgpack_restore(f.read())
ck["train_key"] = TRAIN_KEY
ck["corpus_build_key"] = BUILD_KEY
ck["hparams"] = {k: CFG[k] for k in ("arch", "epochs", "batch_size", "lr", "fpr_budget",
                                    "hard_neg_k", "gamma", "pos_bias", "spec_aug_prob")}
with open(CHECKPOINT, "wb") as f:
    f.write(serialization.to_bytes(ck))
print("stamped train_key into the checkpoint")


no reusable checkpoint for this train key -> training
 INDUSTRIAL TRAINING v2  |  arch=bcconformer_v2  |  objective = max TPR @ FPR <= 0.50%
 dataset            : /kaggle/working/corpus_fp32.npz
 train              : 107,999  (39,997 pos / 68,002 neg)
 val                : 27,001  (10,000 pos / 17,001 neg)
 mixture buckets    : 24  (per-bucket loss + metrics enabled)
 model              : bcconformer_v2  86,450 params  (84.4 KB INT8)
 class weights      : neg=1.304 pos=1.956  (pos:neg count ratio 1:1.70)
 focal gamma        : 1.2   hard-negative mining: k=192
 spec-augment prob  : 0.3   mic-gain aug: p=0.6, +/-12.0 dB
 bucket loss weight : on
 offset-stress set  : 10000 positives rolled by U[0,20) frames
 per-bucket report  : 20,000 val samples across 24 buckets
--------------------------------------------------------------------------------------------
 Ep |    Loss |  TrAcc |     AUC |  TPR@bud |     FPR |    thr |  TPR@off |    TPR |     FPR
-----------------------------------------


## Held-out keyword waveforms — required, not optional

The streaming-recall test splices a clean utterance of the wake word into a live 1 s
sliding window. It needs clean keyword waveforms to splice, and they are **not** the
training bases: the corpus builder never dumps them, so without this cell
`load_keyword_bases()` returns nothing and the recall column comes out empty.

That is not a cosmetic gap. The sweep selects its operating point by
`recall > best_recall`, so with every recall reading 0 it would silently freeze the
*first* threshold that meets the false-alarm budget — the tightest one available, and the
worst possible choice. A missing recall test would quietly become a missing recall test in
the shipped checkpoint.

The set is bare "amaze" plus two realistic carriers, at rate and pitch offsets chosen
outside the training pool's own distribution.


In [27]:

# ==============================================================================
# Held-out clean keyword waveforms for the streaming-recall test.
# ==============================================================================
EVAL_RATES = ["-30%", "-22%", "-16%", "-8%", "+4%", "+10%", "+18%", "+28%", "+36%"]
EVAL_PITCHES = ["-30Hz", "-18Hz", "-6Hz", "+6Hz", "+18Hz", "+30Hz"]

async def _gather_eval_bases(texts, voices, rates, pitches, limit_per_voice=2):
    sem = asyncio.Semaphore(12)
    out = []

    async def one(text, v, r, p):
        async with sem:
            for _ in range(2):
                try:
                    w = await synth_edge(text, v, rate=r, pitch=p)
                except Exception:
                    w = None
                if w is not None and len(w) > 800:
                    out.append((f"{text}|{v}|{r}|{p}", trim_silence(w)))
                    return
                await asyncio.sleep(0.3)

    tasks = []
    for v in voices:
        combos = [(r, p) for r in rates for p in pitches]
        np.random.shuffle(combos)
        for k in range(limit_per_voice):
            r, p = combos[k]
            for t in texts:
                tasks.append(one(t, v, r, p))
    await asyncio.gather(*tasks)
    return out

EVAL_BASES = os.path.join(WORK, "keyword_bases_eval.npy")
if not os.path.exists(EVAL_BASES):
    print("synthesising held-out keyword waveforms via edge-tts (1-2 min) ...")
    bare = run_coroutine_sync(
        _gather_eval_bases(["amaze"], EDGE_VOICES, EVAL_RATES, EVAL_PITCHES, 2))
    carrier = run_coroutine_sync(
        _gather_eval_bases(["hey amaze", "ok amaze"], EDGE_VOICES[::3],
                           EVAL_RATES[:5], EVAL_PITCHES[:4], 1))
    pool = bare + carrier
    np.save(EVAL_BASES, np.array([w for _, w in pool], dtype=object), allow_pickle=True)
    print(f"[done] {len(pool)} held-out waveforms "
          f"(bare {len(bare)}, carrier {len(carrier)}) -> {EVAL_BASES}")
else:
    print(f"reusing {EVAL_BASES}")

os.chdir(WORK)
KEYWORD_BASES = load_keyword_bases("keyword_bases_eval.npy")
assert KEYWORD_BASES, (
    "no usable held-out keyword waveforms. The streaming-recall test cannot run without "
    "them, and the operating point must not be chosen with recall unavailable.")
print(f"usable keyword bases: {len(KEYWORD_BASES)} "
      f"({np.mean([len(b) for b in KEYWORD_BASES])/16000:.2f} s mean)")


synthesising held-out keyword waveforms via edge-tts (1-2 min) ...
[done] 98 held-out waveforms (bare 74, carrier 24) -> /kaggle/working/keyword_bases_eval.npy
[eval] keyword bases: keyword_bases_eval.npy (67 usable)
usable keyword bases: 67 (0.60 s mean)



## Stage 3a — benchmark the thing that actually ships

Validation FPR is **not** a deployment predictor: a checkpoint can look clearly better on
the validation split and be far worse on real audio, because the validation negatives are
level-normalised 1 s slices while deployment is un-normalised long-form speech. So every
number below comes from the sliding-window detector over 4.97 h of 84-class soundscapes
and continuous LibriSpeech.


In [28]:

# ==============================================================================
# Stage 3a -- full benchmark. Validation FPR is not a deployment predictor, so this
# scores the thing that actually ships: a 1 s sliding-window detector.
# ==============================================================================
os.chdir(WORK)
predict, ck = load_predict(CHECKPOINT, CFG["arch"])
extractor = AudioFeatureExtractor()
THR = float(ck.get("threshold", 0.85))
print(f"model {CFG['arch']}  {ck.get('param_count', 0):,} params  epoch {ck.get('epoch')}  "
      f"stored threshold {THR:.4f}")

sec1_roc(predict, TRAIN_NPZ)
sec1b_buckets(predict, TRAIN_NPZ, THR)
sec1c_level(predict, TRAIN_NPZ, THR)
sec2_offset(predict, TRAIN_NPZ, extractor)
sec3_soundscapes(predict, extractor, THR, os.path.join(WORK, "noise_bank_v2", "index.csv"))
sec4_speech_fa(predict, extractor, THR, num_files=CFG["speech_files"], agg="maxhold")
sec5_streaming_recall(predict, extractor, THR, KEYWORD_BASES,
                      n_trials=CFG["recall_trials"], agg="maxhold")


model bcconformer_v2  86,450 params  epoch 49  stored threshold 0.9391

 1. VALIDATION ROC AND OPERATING POINTS
 val: 10000 pos / 17001 neg   AUC = 99.056%
 positives : 1%=0.028  5%=0.422  10%=0.742  median=0.996
 negatives : 99%=0.874  99.9%=0.993  max=0.999
--------------------------------------------------------------------------------------
 FPR budget | threshold |     TPR | actual FPR |  FA / 1000 neg
      0.05% |    0.9966 |  47.16% |     0.047% |              8
      0.10% |    0.9936 |  54.64% |     0.100% |             17
      0.20% |    0.9842 |  63.86% |     0.200% |             34
      0.50% |    0.9391 |  77.74% |     0.500% |             85
      1.00% |    0.8745 |  84.53% |     1.000% |            170
      2.00% |    0.7382 |  90.07% |     2.000% |            340
      5.00% |    0.3358 |  95.83% |     5.000% |            850

 1b. PER-BUCKET BREAKDOWN AT THE OPERATING POINT
 threshold 0.9391    val 27,001 samples
bucket                       |      n | recall/FA |


## Stage 3b — pick and freeze the operating point

Both aggregations are shown because this is the single largest deployment decision in the
system. An EMA at α=0.6 needs roughly five consecutive high frames to cross the
threshold, but a 0.6 s wake word only produces two or three good frames in a 1 s window —
so an EMA-confirmed detector is mathematically unable to confirm a realistic burst. A
decaying peak hold confirms the same burst while still rejecting isolated spikes.

The winner is written into the checkpoint, so the engine picks it up with no arguments.


In [29]:

# ==============================================================================
# Stage 3b -- threshold x policy sweep on measured false alarms per hour, then freeze
# the winner into the checkpoint.
# ==============================================================================
need, span = (int(x) for x in CFG["policy"].split("/"))
rows = load_index(os.path.join(WORK, "noise_bank_v2", "index.csv"))
bases = KEYWORD_BASES
assert bases, ("no held-out keyword waveforms, so recall is unmeasured. Selecting a "
                "threshold on false alarms alone always picks the tightest one, which "
                "is the worst operating point. Re-run the held-out bases cell first.")

print("scanning soundscapes once ...", flush=True)
t0 = time.time()
noise = soundscape_scan(predict, extractor, rows)
noise_hours = sum(len(to_16k(*sf.read(p))) for p, _ in rows) / 16000 / 3600.0
print(f"  {len(noise)} classes, {noise_hours:.2f} h scored in {time.time()-t0:.0f}s", flush=True)

flacs = []
for root, _dirs, files in os.walk(os.path.join(WORK, "speech_corpus", "LibriSpeech")):
    for f in files:
        if f.endswith(".flac"):
            flacs.append(os.path.join(root, f))
rng = np.random.default_rng(0)
sel = rng.choice(len(flacs), size=min(CFG["speech_files"], len(flacs)), replace=False)
speech_cache = []
for i in sel:
    try:
        speech_cache.append(to_16k(*sf.read(flacs[i])))
    except Exception:
        pass
speech_hours = sum(len(d) for d in speech_cache) / 16000 / 3600.0
print(f"speech {speech_hours:.3f} h from {len(speech_cache)} recordings", flush=True)

print("\n" + "=" * 96)
print(f"{'agg':>8} | {'thresh':>7} | {'soundscFA/h':>11} | {'speechFA/h':>10} | "
      f"{'totalFA/h':>9} | {'recall':>7}")
print("-" * 96)
best = None
for agg in ("maxhold", "ema"):
    for t in CFG["thresholds"]:
        kw = dict(need=need, span=span, refractory_steps=CFG["refractory"], agg=agg)
        det = StreamingDetector(predict, extractor, t, **kw)
        hits = sum(len(det.run(a)[0]) for a in speech_cache)
        speech_fa = hits / max(1e-9, speech_hours)
        noise_fa = sum(int(np.sum(p >= t)) for p in noise.values()) / max(1e-9, noise_hours)
        total = noise_fa + speech_fa
        rh, rc, _ = recall_scan(predict, extractor, t, bases, rows,
                               CFG["recall_trials"], [20.0, 12.0, 6.0, 0.0], **kw)
        recall = sum(rh.values()) / max(1, sum(rc.values())) * 100
        ok = " *" if total <= CFG["fa_budget"] else ""
        print(f"{agg:>8} | {t:7.4f} | {noise_fa:11.2f} | {speech_fa:10.2f} | "
              f"{total:9.2f} | {recall:6.1f}%{ok}", flush=True)
        if total <= CFG["fa_budget"] and (best is None or recall > best["recall"]):
            best = {"agg": agg, "t": t, "recall": recall, "fa": total}
print("-" * 96)

if best is None:
    print("No operating point met the FA budget; the trainer's stored threshold is kept.")
else:
    print(f"CHOSEN: threshold {best['t']:.4f}, agg={best['agg']}, policy {CFG['policy']}, "
          f"refractory {CFG['refractory']*0.1:.1f}s")
    print(f"        measured streaming recall {best['recall']:.1f}%, total FA {best['fa']:.2f}/hour")
    ck["threshold"] = float(best["t"])
    ck["deploy_threshold"] = float(best["t"])
    ck["deploy_agg"] = best["agg"]
    ck["deploy_need"] = need
    ck["deploy_span"] = span
    ck["deploy_refractory_steps"] = CFG["refractory"]
    ck["deploy_recall"] = float(best["recall"])
    ck["deploy_fa_per_hour"] = float(best["fa"])
    with open(CHECKPOINT, "wb") as f:
        f.write(serialization.to_bytes(ck))
    THR = float(best["t"])
    print(f"written into {os.path.basename(CHECKPOINT)}")


scanning soundscapes once ...
  84 classes, 4.71 h scored in 46s
speech 0.291 h from 150 recordings

     agg |  thresh | soundscFA/h | speechFA/h | totalFA/h |  recall
------------------------------------------------------------------------------------------------
 maxhold |  0.7800 |       10.83 |      37.80 |     48.63 |   76.7%
 maxhold |  0.8200 |        9.56 |      17.18 |     26.74 |   70.0%
 maxhold |  0.8400 |        8.28 |      17.18 |     25.46 |   64.0%
 maxhold |  0.8600 |        7.44 |      13.74 |     21.18 |   58.3%
 maxhold |  0.9000 |        6.59 |      10.31 |     16.89 |   51.7%
 maxhold |  0.9400 |        4.25 |       0.00 |      4.25 |   39.7%
     ema |  0.7800 |       10.83 |       3.44 |     14.27 |   21.0%
     ema |  0.8200 |        9.56 |       3.44 |     13.00 |   14.7%
     ema |  0.8400 |        8.28 |       3.44 |     11.72 |    0.3%
     ema |  0.8600 |        7.44 |       3.44 |     10.87 |    0.3%
     ema |  0.9000 |        6.59 |       0.00 |      6

In [ ]:

# ==============================================================================
# Stage 3c -- the deployment entry point, with no arguments.
# ==============================================================================
engine = StreamingKWSEngine(checkpoint_path=CHECKPOINT, arch=CFG["arch"])
spotted, conf, lat = engine.push_audio_chunk(
    np.random.normal(0, 0.01, 1600).astype(np.float32))   # 100 ms @ 16 kHz
print(f"\nsmoke test: spotted={spotted} score={conf:.4f} latency={lat:.2f} ms")



smoke test: spotted=False score=0.0076 latency=6.39 ms



## Stage 4 — publish

Two datasets, both private, both owned by this notebook. The next run pulls them and
skips straight to stage 3.

**Do not use `/kaggle/working` as the notebook's output directory** — that would
re-upload the corpus you already published.


In [35]:

# ==============================================================================
# Stage 4 -- publish the corpus and the frozen model as Kaggle Datasets, so the next
# run of this notebook pulls instead of rebuilding and retraining.
# ==============================================================================
if CFG["publish_corpus"] and CORPUS_SOURCE == "build":
    ok = kaggle_publish(local_bundle, CFG["corpus_slug"],
                        title="Amaze KWS corpus",
                        subtitle=f"bucketed 1s log-mel windows, build_key {BUILD_KEY}",
                        description=("Wake-word training corpus for 'amaze' on ESP32-S3 / INMP441. "
                                     "24 mixture buckets, INMP441 level policy, float16 log-mel. "
                                     "Match on build_key; never regenerate by hand."),
                        tags=["keyword-spotting", "wake-word", "speech", "audio", "esp32"])
    print("corpus published:", ok)

if CFG["publish_model"]:
    mb = os.path.join(WORK, "model-bundle")
    shutil.rmtree(mb, ignore_errors=True)
    os.makedirs(mb, exist_ok=True)
    shutil.copy2(CHECKPOINT, os.path.join(mb, os.path.basename(CHECKPOINT)))
    summary = {k: ck.get(k) for k in
               ("arch", "param_count", "epoch", "val_auc", "fpr_budget", "threshold",
                "tpr_at_budget", "tpr_offset_stress", "tpr_at_0012",
                "deploy_agg", "deploy_need", "deploy_span", "deploy_refractory_steps",
                "deploy_recall", "deploy_fa_per_hour", "train_key", "corpus_build_key",
                "hparams")}
    with open(os.path.join(mb, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, sort_keys=True, default=str)
    write_manifest(mb, TRAIN_KEY)
    ok = kaggle_publish(mb, CFG["model_slug"],
                        title="Amaze KWS trained models",
                        subtitle=f"{CFG['arch']} train_key {TRAIN_KEY} threshold {ck.get('threshold',0):.4f}",
                        description=("Frozen checkpoints with the deployment operating point already "
                                     "written in. Match on train_key."),
                        tags=["keyword-spotting", "wake-word", "esp32", "inmp441", "flax"])
    print("model published:", ok)


[kaggle] publish failed: KaggleApi.dataset_create_new() got an unexpected keyword argument 'ignore_patterns'
model published: False


In [36]:

# ==============================================================================
# Summary
# ==============================================================================
print(f"build key        : {BUILD_KEY}")
print(f"train key        : {ck.get('train_key')}")
print(f"checkpoint       : {os.path.basename(CHECKPOINT)}")
print(f"architecture     : {ck.get('arch')}  {ck.get('param_count', 0):,} params  "
      f"({ck.get('param_count', 0)/1024:.1f} KB INT8)")
print(f"epoch            : {ck.get('epoch')}")
print()
print(f"val AUC          : {ck.get('val_auc', 0)*100:.2f}%")
print(f"TPR @ FPR<={ck.get('fpr_budget',0)*100:.2f}%    : {ck.get('tpr_at_budget', 0)*100:.2f}%")
print(f"offset-stressed  : {ck.get('tpr_offset_stress', 0)*100:.2f}%")
print(f"TPR @ FPR<=0.12% : {ck.get('tpr_at_0012', 0)*100:.2f}%")
print()
print(f"threshold        : {ck.get('threshold', 0):.4f}")
print(f"confirmation     : {ck.get('deploy_agg', 'maxhold')} "
      f"{ck.get('deploy_need', 2)}/{ck.get('deploy_span', 2)}, "
      f"{ck.get('deploy_refractory_steps', 15)*0.1:.1f} s refractory")
print(f"measured         : recall {ck.get('deploy_recall', float('nan')):.1f}%  "
      f"FA {ck.get('deploy_fa_per_hour', float('nan')):.2f}/hour")
print("\nHardware note: the frozen operating point assumes an INMP441 at default sensitivity "
      "with no AGC. A different board preamp gain needs a re-freeze against measured FA/hour.")


build key        : 3171611c0ed043c2
train key        : 0a1e1b108f683ccd
checkpoint       : bcconformer_v2_0a1e1b108f683ccd.flax
architecture     : bcconformer_v2  86,450 params  (84.4 KB INT8)
epoch            : 49

val AUC          : 99.06%
TPR @ FPR<=0.50%    : 77.74%
offset-stressed  : 65.52%
TPR @ FPR<=0.12% : 57.27%

threshold        : 0.9391
confirmation     : maxhold 2/2, 1.5 s refractory
measured         : recall nan%  FA nan/hour

Hardware note: the frozen operating point assumes an INMP441 at default sensitivity with no AGC. A different board preamp gain needs a re-freeze against measured FA/hour.
